# 最終演習K 2台の録音機（非同期録音）の時計のずれを直して分離する（ノートブック版）

教科書の最終演習K（id = fink）のうち、**シミュレーションと合成の録音で動く部分**を、K-5 の手順の段階の順に1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約 23 分（試しの設定のとき）。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 時計のずれの大きさを計算する | 段階1 の 3 | （K-3-2 の式） |
| 2 | シミュレーションの部品を 1 試行で試す | 段階2 の 2 | `clockfix.py`・`sim_async.py` |
| 3 | 予備実験（シミュレーション）で K・S・n を決める（**試しの設定**） | 段階2 の 3・4 | `sim_async.py`・`fink_power.py` |
| 4 | 再生ファイルを作る（音源は録音の代わりに librosa の例） | 段階3 の 1 | `make_play.py` |
| 5 | 合成の録音で、解析の手順を先に確かめる（**種 1 つの試しの設定**） | 段階3 の 2 | `synth_session.py`・`real_mix.py`・`fink_real_stats.py` |
| 6 | 実録音のセッションの数 M を出す（ずれの値は見本） | 段階3 の 5 | `fink_power.py --real-eps` |
| 7 | 本実験（シミュレーション）と主解析（**試しの設定**） | 段階4 の 1・段階5 の 1 | `sim_async.py`・`fink_stats.py` |
| 8 | 副次の確かめ（作りと直しが同じ問題。**試しの設定**） | 段階5 の 3 | `sim_async.py --gen fft / --drift 0.5`・`fink_check.py` |
| 9 | 合成の録音の結果を、シミュレーションの予想と比べる（実録音の解析の練習） | 段階5 の 2 | `fink_real_stats.py --sim` |

- 次のセルたちが、共通ライブラリ（IV-2 の `common/__init__.py`、IV-5 の `common/array.py`）と、第7章の `bss.py`、この演習の `clockfix.py` を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の各章のコードと同じなので、ほかのファイルは要りません（`common/` のほかのファイルは、この演習のプログラムが使わないので書き出しません）。
- 各節のコードは K-6 のプログラムそのものです。変えたのは、ファイルの場所（`sys.path.insert` で `../../ch7`・`../../partIV` の代わりにノートブックの隣を読む）、図をセルの下に表示すること（`matplotlib.use("Agg")` を外し、`plt.show()` を足す）、日本語のフォントの候補を足すこと、コマンドの引数を `sys.argv` で渡すこと、やり直せるように前に作った CSV・フォルダを先に消すこと（`fresh()`。教科書のスクリプトは上書きしないように止まります）、出力をいったん `cap` に受け取って次のセルで表示し、`ok()` で K-5 の確認ポイントを確かめることだけです。ノートブックで足したセルは、1 節の計算と、4 節の音源を作るセルと、音を聞くセルです。
- **重い計算は試しの設定を既定にしています**（3・5・7・8 節）。ILRMA 1回（8秒の区間、100回の反復）が執筆時の Mac で約 5 秒かかり、教科書の本番（予備実験 約45分、本実験 150 試行で約 5.4 時間）はノートブックには向きません。本番の設定は、各セルの `sys.argv` の上にコメントで書いてあります。試しの設定では試行が少ないので、教科書の K-7-1 の平均・区間・p 値とは合いません。ノートブックの `ok()` は、試行の数に関係なく成り立つこと（行の数、ずれの推定の精度、除外の判定、差の向きなど）だけを確かめます。
- **入れていないもの**（実際の機器が要る、またはノートブックでは意味のない手順）：`setup_fink.sh`（ここでは `%%writefile` が代わり）と `run_fink.sh`（このノートブックが代わり）、段階1 の論文を読むこと、段階2 の事前登録（`prereg.md` を書いて Git にコミットする。K-4-5 のひな形）、段階3・4 の**実際の2台の録音**（第7章の `record.py` で自分の声と楽器を録る、スマホの録音と `afinfo`・`afconvert`、予備のセッション・本実験のセッションを録って鳴らす、`sessions.csv` を書く）、`datahash.py` でデータの版を固めること、任意の聴取実験 `listen_test.py`（ターミナルで聞いて点をつける。家族に聞いてもらうときの同意の注意は K-2）、段階6 の報告と発表。実録音のかわりに、5 節・9 節では `synth_session.py` の**合成の録音**（答えの分かったデータ）を使います。自分の録音で行うときは、`data/` に wav を置き、`sessions.csv` を書いて、5 節の `real_mix.py` のセルの引数を教科書の段階5 のとおりにします。
- **音を聞くときの注意**：再生ボタンは `listen_file()` が作ります。最大値が 0.3（約 −10 dBFS）を超える wav は 0.3 まで**小さくしてから**鳴らし、大きくはしません（`IPython.display.Audio(..., normalize=False)`。**`normalize=False` を消さないでください**。消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を一番小さくしてください。合図のチャープ（300 Hz → 6 kHz）は音源より約 11 dB 大きく鳴ります。
- 音源は librosa の例の音源 `libri1`・`libri2`・`trumpet` を使います（初回はダウンロードします。ネットワークが要ります）。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、pyroomacoustics・librosa・scipy の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile"), ("statsmodels", "statsmodels"),
                 ("pyroomacoustics", "pyroomacoustics"), ("fast_bss_eval", "fast_bss_eval")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, statsmodels, pyroomacoustics, librosa
print("numpy", numpy.__version__, "scipy", scipy.__version__, "statsmodels", statsmodels.__version__,
      "pyroomacoustics", pyroomacoustics.__version__, "librosa", librosa.__version__)
print("（教科書は pyroomacoustics 0.10.1・librosa 0.11 系で確かめた。版はノートと計画書に書く：段階2 の 1）")
import os
for d in ["common", "data"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の 4 つのセルで、ほかの章のファイル（IV-2 の `common/__init__.py`、IV-5 の `common/array.py`、第7章の `bss.py`）と、この演習の共通部品 `clockfix.py`（K-6）を書き出します。`common/array.py` は 8 節の `--gen fft` だけが使います。

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile common/array.py
# IV-5 の ~/home-lab/partIV/common/array.py と同じもの（8 節の --gen fft で sfo_resample を使う）
"""第IV部の共通ライブラリ：マイクロホンアレイ（IV-5 で定義。IV-6・最終演習K でも使う）

約束（IV-5 の本文の記号と同じ）
  mics      : マイクの位置。形 (D, M)（D = 2 か 3、単位 m）。pyroomacoustics の R と同じ並び
  azimuth   : 方位角 θ [rad]。x 軸から反時計回り（pyroomacoustics と同じ）。
              直線アレイを x 軸に沿って置くと、θ = 90° が正面（ブロードサイド）、0°・180° が軸の方向（エンドファイア）
  u(θ)      : 原点から音源の方向を向く単位ベクトル [cosθ, sinθ(, 0)]
  遅延      : 平面波（遠方場）では、マイク m に届く時刻は原点より τ_m = −u·r_m / c だけ遅い
  ステアリングベクトル a_f(θ) の成分 a_m = exp(−j2πf τ_m) = exp(+j2πf u·r_m / c)
              （STFT の e^{−j2πkm/L} の約束で「τ 遅れると e^{−j2πfτ} が掛かる」。pyroomacoustics.doa と同じ符号）
  X         : 観測の STFT。形 (F, T, M)（common.stft.stft の多チャネルの出力）
  重み w     : 出力 y_ft = w_fᴴ x_ft = Σ_m conj(w_fm) x_ftm。W の形は (F, M)

関数
  linear_array, circular_array, unit_vector, far_field_delays, steering_vector,
  scm, ds_weights, mvdr_weights, apply_weights, beam_pattern, white_noise_gain,
  gcc_phat, tdoa_to_azimuth, music_spectrum, srp_phat_spectrum,
  sfo_resample, sfo_phase
"""
import numpy as np

C_SOUND = 343.0          # 音速 [m/s]（約 20 ℃。pyroomacoustics の既定と同じ）

__all__ = ["C_SOUND", "linear_array", "circular_array", "unit_vector", "far_field_delays",
           "steering_vector", "scm", "ds_weights", "mvdr_weights", "apply_weights", "beam_pattern",
           "white_noise_gain", "gcc_phat", "tdoa_to_azimuth", "music_spectrum", "srp_phat_spectrum",
           "sfo_resample", "sfo_phase"]


# ---------------------------------------------------------------- 配置
def linear_array(M, d, center=(0.0, 0.0), angle=0.0):
    """間隔 d [m] の M 本の直線アレイ。中心が center、向きが angle [rad]（x 軸から）。形 (2, M)
    （pyroomacoustics.linear_2D_array と同じ）"""
    u = np.array([np.cos(angle), np.sin(angle)])
    k = np.arange(M) - (M - 1) / 2
    return np.asarray(center, float)[:, None] + d * k[None, :] * u[:, None]


def circular_array(M, radius, center=(0.0, 0.0), phi0=0.0):
    """半径 radius [m] の M 本の円形アレイ。形 (2, M)（pyroomacoustics.circular_2D_array と同じ）"""
    phi = phi0 + 2 * np.pi * np.arange(M) / M
    return np.asarray(center, float)[:, None] + radius * np.vstack([np.cos(phi), np.sin(phi)])


def unit_vector(azimuth, dim=2):
    """方位角（スカラーか (K,)）の単位ベクトル。形 (dim,) か (dim, K)"""
    az = np.asarray(azimuth, float)
    u = [np.cos(az), np.sin(az)] + ([np.zeros_like(az)] if dim == 3 else [])
    return np.array(u)


# ---------------------------------------------------------------- ステアリングベクトル
def far_field_delays(mics, azimuth, c=C_SOUND):
    """平面波の到来時刻の遅れ τ_m = −u·r_m / c [s]（原点を基準）。形 (M,) か (K, M)"""
    u = unit_vector(azimuth, dim=mics.shape[0])
    return -(u.T @ mics) / c


def steering_vector(mics, freqs, azimuth=None, source=None, c=C_SOUND, amplitude=False):
    """ステアリングベクトル。
    遠方場（azimuth を与える）：a_m = exp(−j2πf τ_m)、τ_m = −u·r_m / c。
        azimuth がスカラーなら形 (F, M)、(K,) なら (F, K, M)
    近距離場（source に音源の位置 (D,) を与える）：アレイの中心 c̄ からの距離 ρ_0 を基準に
        τ_m = (|p − r_m| − |p − c̄|) / c。amplitude=True なら距離の減衰 ρ_0/|p − r_m| も掛ける。形 (F, M)
    """
    f = np.atleast_1d(np.asarray(freqs, float))
    if (azimuth is None) == (source is None):
        raise ValueError("azimuth か source のどちらか一方を与える")
    if azimuth is not None:
        tau = far_field_delays(mics, azimuth, c)                      # (M,) か (K, M)
        return np.exp(-2j * np.pi * f.reshape((-1,) + (1,) * tau.ndim) * tau)
    p = np.asarray(source, float).reshape(-1, 1)[: mics.shape[0]]
    dist = np.linalg.norm(p - mics, axis=0)                          # (M,)
    ref = np.linalg.norm(p[:, 0] - mics.mean(axis=1))
    a = np.exp(-2j * np.pi * f[:, None] * (dist - ref)[None, :] / c)
    return a * (ref / dist)[None, :] if amplitude else a


# ---------------------------------------------------------------- 空間相関行列とビームフォーマ
def scm(X):
    """空間相関行列 R_f = (1/T) Σ_t x_ft x_ftᴴ（複素正規分布の共分散の最尤推定。IV-4-3-4）。
    X (F, T, M) → (F, M, M)"""
    return np.einsum("ftm,ftn->fmn", X, X.conj()) / X.shape[1]


def ds_weights(a):
    """遅延和ビームフォーマ w = a / M（目的の方向の利得が 1）。a (F, M) → (F, M)"""
    return a / a.shape[-1]


def mvdr_weights(a, R, loading=0.0):
    """MVDR（最小分散無歪み応答）w = R⁻¹a / (aᴴR⁻¹a)。
    loading > 0 なら対角装荷：R + ε I、ε = loading × tr(R)/M（周波数ごと）。a (F, M)、R (F, M, M)"""
    R = np.asarray(R, complex)
    M = R.shape[-1]
    if loading > 0:
        eps = loading * np.real(np.trace(R, axis1=-2, axis2=-1)) / M
        R = R + eps[:, None, None] * np.eye(M)[None]
    Ria = np.linalg.solve(R, a[..., None])[..., 0]                    # R⁻¹a
    den = np.einsum("fm,fm->f", a.conj(), Ria)                       # aᴴR⁻¹a（実数で正）
    return Ria / den[:, None]


def apply_weights(W, X):
    """y_ft = w_fᴴ x_ft。W (F, M)、X (F, T, M) → (F, T)"""
    return np.einsum("fm,ftm->ft", W.conj(), X)


def beam_pattern(W, mics, freqs, azimuths, c=C_SOUND):
    """ビームパターン B_f(θ) = w_fᴴ a_f(θ)（複素数）。W (F, M) → (F, K)"""
    A = steering_vector(mics, freqs, azimuth=np.asarray(azimuths, float), c=c)   # (F, K, M)
    return np.einsum("fm,fkm->fk", W.conj(), A)


def white_noise_gain(W, a):
    """白色雑音利得 WNG = |wᴴa|² / (wᴴw)（マイクごとに独立な雑音を何倍に抑えるか。DS で M）"""
    num = np.abs(np.einsum("fm,fm->f", W.conj(), a)) ** 2
    return num / np.real(np.einsum("fm,fm->f", W.conj(), W))


# ---------------------------------------------------------------- 到来方向推定
def gcc_phat(x1, x2, fs, max_tau=None, interp=1):
    """GCC-PHAT で、x2 が x1 より何秒遅れているか（τ > 0 なら x2 が遅い）を推定する。
    G(ω) = X2(ω) X1*(ω) / |X2(ω) X1*(ω)|、cc(τ) = IDFT[G]（長さを interp 倍にして細かくする）
    戻り値: tau [s], lags [s], cc（lags の順に並べた相互相関）"""
    x1 = np.asarray(x1, float)
    x2 = np.asarray(x2, float)
    n = len(x1) + len(x2)                                            # 円状のたたみ込みにならない長さ
    n += n % 2
    G = np.fft.rfft(x2, n) * np.conj(np.fft.rfft(x1, n))
    G /= np.abs(G) + 1e-20
    cc = np.fft.irfft(G, interp * n)
    half = interp * n // 2
    cc = np.concatenate([cc[-half:], cc[:half]])                     # ラグ −half 〜 half−1
    lags = (np.arange(-half, half)) / (interp * fs)
    if max_tau is not None:
        keep = np.abs(lags) <= max_tau
        cc, lags = cc[keep], lags[keep]
    return float(lags[np.argmax(cc)]), lags, cc


def tdoa_to_azimuth(tau, d, c=C_SOUND):
    """2本のマイク（x 軸上、マイク2 が +x 側に d）の時間差 τ（マイク2 の遅れ）から方位角 θ [rad]。
    τ = −d cosθ / c なので θ = arccos(−cτ/d)（0〜π。表と裏は区別できない）"""
    return float(np.arccos(np.clip(-c * tau / d, -1.0, 1.0)))


def music_spectrum(X, mics, freqs, azimuths, n_src=1, normalize=False, c=C_SOUND):
    """広帯域 MUSIC の空間スペクトル（周波数ごとの 1/|aᴴ E_n E_nᴴ a| の平均）。
    X (F', T, M) は使う周波数のビンだけを取り出した STFT、freqs (F',) はその周波数 [Hz]。
    normalize=True なら周波数ごとに最大を 1 にしてから平均（pyroomacoustics の NormMUSIC）"""
    R = scm(X)
    _, V = np.linalg.eigh(R)                                         # 固有値の小さい順
    En = V[:, :, : X.shape[2] - n_src]                               # 雑音部分空間 (F', M, M−N)
    A = steering_vector(mics, freqs, azimuth=azimuths, c=c)          # (F', K, M)
    proj = np.einsum("fkm,fmj->fkj", A.conj(), En)                   # aᴴ E_n
    P = 1.0 / np.sum(np.abs(proj) ** 2, axis=2)                      # (F', K)
    if normalize:
        P = P / P.max(axis=1, keepdims=True)
    return P.mean(axis=0)


def srp_phat_spectrum(X, mics, freqs, azimuths, c=C_SOUND):
    """SRP-PHAT：振幅を 1 にそろえた観測 x̃ = x/|x|（成分ごと）に遅延和をかけた出力のパワー
    |aᴴ x̃|² を、フレームと周波数で平均したもの。X (F', T, M)"""
    Xn = X / np.maximum(np.abs(X), 1e-14)
    A = steering_vector(mics, freqs, azimuth=azimuths, c=c)          # (F', K, M)
    out = np.einsum("fkm,ftm->fkt", A.conj(), Xn)
    return np.mean(np.abs(out) ** 2, axis=(0, 2))


# ---------------------------------------------------------------- 非同期録音（標本化周波数のずれ）
def sfo_resample(x, eps, pad=4096):
    """標本化周波数が fs(1+eps) の機器で録った信号を作る：y[n] = x_c(n / (fs(1+eps)))。
    FFT による帯域制限の補間（scipy.signal.resample）。eps はサンプル数の都合で少し丸める。
    戻り値: y（長さは x と同じ）, 実際に使った eps"""
    from scipy.signal import resample
    x = np.asarray(x, float)
    N = len(x) + pad                                                 # 端の回り込みを防ぐゼロ
    if eps != 0:                                                     # N·eps が整数に近くなる N を選ぶ（丸めの誤差を小さく）
        cand = np.arange(N, N + int(2 / abs(eps)) + 2)
        N = int(cand[np.argmin(np.abs(cand * eps - np.round(cand * eps)))])
        pad = N - len(x)
    num = int(round(N * (1 + eps)))
    y = resample(np.concatenate([x, np.zeros(pad)]), num)
    return y[: len(x)], num / N - 1


def sfo_phase(F, T, L, S, eps):
    """フレーム t（中心の時刻 tS/fs）での、ずれ eps による位相の回転 exp(−j2π k eps t S / L)。形 (F, T)。
    標本化周波数が fs(1+eps) のチャネルは、時刻 τ で eps·τ だけ遅れて見える（IV-5-3-10）。
    補正は、そのチャネルの STFT にこの共役を掛ける"""
    k = np.arange(F)[:, None]
    t = np.arange(T)[None, :]
    return np.exp(-2j * np.pi * k * eps * t * S / L)

In [ ]:
%%writefile bss.py
# 第7章の ~/home-lab/ch7/bss.py（AuxIVA・ILRMA の実装）と同じもの
#!/usr/bin/env python3
"""第7章：AuxIVA と ILRMA を NumPy でゼロから実装する（決定系：音源の数 N ＝ マイクの数 M）

記号（論文の周波数・時間の番号 i, j を、この章では f, t と書く。T はフレームの数で、
      論文の基底行列 T_n（ここでは Tb）とは別のもの）
  X : 観測の STFT。形は (F, T, M)。x_ft = [x_ft1, …, x_ftM]ᵀ
  W : 分離行列。形は (F, N, M)。y_ft = W_f x_ft。W_f の n 行目が w_nfᴴ
  Y : 分離信号 (F, T, N)、P = |Y|²
  r : 音源 n の分散（音源モデル）。AuxIVA は r_nt = √(Σ_f |y_nft|²)（周波数をまとめた大きさ）、
      ILRMA は r_nft = (T_n V_n)_ft（音源ごとの NMF）
更新（反復射影法 IP）
  U_nf = (1/T) Σ_t x_ft x_ftᴴ / r_n(ft)     （AuxIVA では φ(r)=1/r を掛ける）
  w_nf ← (W_f U_nf)⁻¹ e_n、 w_nf ← w_nf / √(w_nfᴴ U_nf w_nf)
コスト（負の対数尤度。反復で増えないことを確かめる）
  ILRMA : Σ_n Σ_ft (|y_nft|²/r_nft + log r_nft) − 2T Σ_f log|det W_f|
  AuxIVA: Σ_n Σ_t r_nt − T Σ_f log|det W_f|      （ラプラス分布の音源モデル）
          log|det| の係数が ILRMA の半分なのは、尺度を変えたラプラス分布の負の対数尤度を2で割ったものだから。
          定数倍なので、反復で増えないことの確認には影響しない
"""
import numpy as np

EPS = 1e-10


def demix(W, X):
    """Y = W X（周波数ごと）"""
    return np.einsum("fnm,ftm->ftn", W, X)


def ip_update(W, X, phi, n):
    """反復射影法で W の n 行目を更新する。phi は (F, T) の重み（1/r）"""
    F, T, M = X.shape
    U = np.einsum("ftm,ftk,ft->fmk", X, X.conj(), phi) / T          # (F, M, M)
    WU = W @ U
    e = np.zeros((F, M, 1), dtype=complex)
    e[:, n, 0] = 1.0
    w = np.linalg.solve(WU, e)[:, :, 0]                               # (F, M)
    denom = np.sqrt(np.real(np.einsum("fm,fmk,fk->f", w.conj(), U, w)))
    w = w / (denom[:, None] + EPS)
    W[:, n, :] = w.conj()
    return W


def logdet(W):
    return np.sum(np.log(np.abs(np.linalg.det(W)) + EPS))


def projection_back(Y, W, ref=0):
    """スケールのあいまいさを、基準マイク ref で観測した大きさに合わせて直す"""
    A = np.linalg.inv(W)                                              # (F, M, N) 混合行列の推定
    return Y * A[:, ref, :][:, None, :]


def auxiva(X, n_iter=100, return_cost=False):
    F, T, M = X.shape
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = []
    for _ in range(n_iter):
        for n in range(M):
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y[:, :, n]) ** 2, axis=0)) + EPS   # (T,)
            W = ip_update(W, X, np.broadcast_to(1.0 / r, (F, T)), n)
        if return_cost:
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y) ** 2, axis=0)) + EPS           # (T, N)
            hist.append(float(np.sum(r) - T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)


def ilrma(X, n_basis=4, n_iter=100, seed=0, return_cost=False):
    F, T, M = X.shape
    rng = np.random.default_rng(seed)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    Tb = rng.uniform(0.1, 1.0, (M, F, n_basis))                       # 基底 T_n
    Vb = rng.uniform(0.1, 1.0, (M, n_basis, T))                       # アクティベーション V_n
    R = Tb @ Vb                                                       # (N, F, T)
    hist = []
    for _ in range(n_iter):
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)               # (N, F, T)
        # 音源モデルの更新（IS ダイバージェンスの NMF。補助関数法の乗法更新）
        Tb *= np.sqrt(((P / R ** 2) @ np.swapaxes(Vb, 1, 2)) / ((1 / R) @ np.swapaxes(Vb, 1, 2) + EPS))
        R = Tb @ Vb + EPS
        Vb *= np.sqrt((np.swapaxes(Tb, 1, 2) @ (P / R ** 2)) / (np.swapaxes(Tb, 1, 2) @ (1 / R) + EPS))
        R = Tb @ Vb + EPS
        # 分離行列の更新（IP）
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
        # スケールの正規化（W の各行と、音源モデルの大きさをそろえる。結果は変わらない）
        Y = demix(W, X)
        lam = np.sqrt(np.mean(np.abs(Y) ** 2, axis=(0, 1))) + EPS     # (N,)
        W /= lam[None, :, None]
        R /= lam[:, None, None] ** 2
        Tb /= lam[:, None, None] ** 2
        if return_cost:
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            hist.append(float(np.sum(P / R + np.log(R)) - 2 * T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)

In [ ]:
%%writefile clockfix.py
#!/usr/bin/env python3
"""最終演習K：2台の録音機の「時計のずれ」を推定して直すための道具

記号（II-4・II-5 の時計合わせと同じ考え方）
  機器1（基準。Mac）のサンプル番号 n1、機器2（スマホ）のサンプル番号 n2 の対応を
      n2 = alpha + (1 + eps) * n1
  で表す。eps が「サンプリング周波数のずれ」（機器2 の fs が機器1 の (1+eps) 倍。ppm = eps×1e6）、
  alpha が「録音開始のずれ」（機器2 のサンプル数）。どちらも未知なので、合図音（チャープ）から推定する。

関数
  frac_resample(x, start, step, n_out) : x を位置 start, start+step, start+2*step, … で読み直す（帯域制限の補間）
  frac_read(x, pos)                    : x を任意の位置の列 pos で読む（時計の進みが時間とともに変わる録音を作るとき）
  make_chirp(fs)                       : 合図音（対数チャープ）
  find_chirps(x, chirp, spacing)       : 整合フィルタで、間隔の分かった合図の組をさがす（小数点以下まで）
  fit_clock(c1, c2, pairs)             : 合図の位置の組から eps と alpha を推定する
  drift_search(x1, x2, fs)             : 合図を使わず、2チャネルの信号そのものから eps をさがす（相関の最大化）
"""
import numpy as np
from scipy import ndimage, signal

HALF = 32                 # 補間の窓の半分の長さ [サンプル]（64 タップ）
BETA = 8.6                # カイザー窓の形
CUT = 0.94                # 通す帯域（ナイキスト周波数に対する割合）


def frac_resample(x, start, step, n_out, half=HALF):
    """y[n] = x(start + n*step) を、窓つき sinc 補間で求める（範囲の外は 0）

    step = 1 + eps なら「機器2 の録音を機器1 の時刻の格子で読み直す」ことになる。
    1回に 2*half 個の点の重みつき和なので、長い信号は区切って計算する。
    """
    return frac_read(x, start + np.arange(n_out) * step, half)


def frac_read(x, pos, half=HALF):
    """x を任意の位置 pos（小数の配列）で読む（frac_resample の中身。位置の間隔が一定でなくてもよい）"""
    x = np.asarray(x, dtype=float)
    pos_all = np.asarray(pos, dtype=float)
    n_out = len(pos_all)
    y = np.zeros(n_out)
    k = np.arange(-half + 1, half + 1)
    for a in range(0, n_out, 8192):
        n = np.arange(a, min(a + 8192, n_out))
        pos = pos_all[n]
        base = np.floor(pos).astype(int)
        frac = pos - base
        idx = base[:, None] + k[None, :]                   # 使う元の点
        d = frac[:, None] - k[None, :]                     # 元の点からの距離
        w = np.i0(BETA * np.sqrt(np.clip(1 - (d / half) ** 2, 0, 1))) / np.i0(BETA)
        h = CUT * np.sinc(CUT * d) * w
        ok = (idx >= 0) & (idx < len(x))
        y[n] = np.sum(np.where(ok, x[np.clip(idx, 0, len(x) - 1)], 0.0) * h, axis=1)
    return y


def make_chirp(fs, dur=0.3, f0=300.0, f1=6000.0):
    """合図音：対数チャープ（300Hz → 6kHz）。両端を 5ms でなめらかにする。
    周波数は fs によらず同じにする（48kHz で鳴らし、16kHz に直した録音の中から 16kHz の合図でさがすため）"""
    t = np.arange(int(dur * fs)) / fs
    c = signal.chirp(t, f0, dur, f1, method="logarithmic")
    r = int(0.005 * fs)
    win = np.ones_like(c)
    win[:r] = np.sin(np.linspace(0, np.pi / 2, r)) ** 2
    win[-r:] = win[:r][::-1]
    return c * win


def _peak(c, j):
    """相関 c の整数の山 j の位置を、前後 ±32 点を 16 倍に補間（FFT による帯域制限の補間）して放物線で小数点以下まで求める"""
    base = max(j - 32, 0)
    seg = c[base: j + 33]
    up = signal.resample(seg * np.hanning(len(seg) + 2)[1:-1], 16 * len(seg))
    mid = 16 * (j - base)
    k = max(mid - 64, 1) + int(np.argmax(up[max(mid - 64, 1): mid + 64]))
    k = min(k, len(up) - 2)
    a, b, cc = up[k - 1], up[k], up[k + 1]
    den = a - 2 * b + cc
    return base + (k + (0.5 * (a - cc) / den if den != 0 else 0.0)) / 16


def find_chirps(x, chirp, spacing, n_pairs=2, tol=0.002):
    """x の中から「間隔 spacing で並んだ合図の組」を n_pairs 組さがし、始まりの位置（小数点以下まで）を返す

    1. 整合フィルタ（x と合図音の相互相関）の包絡線 env を作り、「i に合図があり、i+spacing（±tol の割合）にも
       合図がある」ことの得点 env[i] + max(env[i+spacing±…]) が高い i を、重ならないように n_pairs 個選ぶ。
       2つの合図の間隔が分かっているので、声や音楽の中の偶然の山を合図と間違えにくい。
    2. 始めの合図の位置：整合フィルタの山を、16倍に補間して小数点以下まで求める。
    3. 終わりの合図の位置：始めの合図を「録った音」（合図 0.3 秒と、そのあとの部屋の響きを含めて 0.6 秒）と、終わりの合図のあたりの録音の
       相互相関の山から求める。同じ場所から鳴った2つの合図は、部屋の響きまで同じ形で録られているので、
       直接音と強い反射が同じくらいの大きさでも、どちらを山とみなすかで迷わない（整合フィルタだけだと、
       雑音で山の選び方が変わり、数十サンプル間違えることがある）。
    戻り値：(位置 [組1の始め, 組1の終わり, 組2の始め, …], 各合図の高さ ÷ 包絡線の中央値)
    """
    corr = signal.correlate(x, chirp, mode="valid", method="fft")
    env = np.abs(signal.hilbert(corr))
    floor = np.median(env) + 1e-12
    w = int(np.ceil(tol * spacing))
    lim = len(env) - spacing - w - 1
    if lim <= 0:
        raise ValueError("録音が短すぎて、合図の組が入りきりません")
    env_far = ndimage.maximum_filter1d(env, 2 * w + 1)        # i+spacing の前後 ±w の最大
    score = env[:lim] + env_far[spacing: spacing + lim]
    picks = []
    for i in np.argsort(score)[::-1]:
        if all(abs(i - p) > spacing + 2 * w for p in picks):
            picks.append(i)
            if len(picks) == n_pairs:
                break
    if len(picks) < n_pairs:
        raise ValueError("合図の組が見つかりません")
    n = len(chirp)
    out, q = [], []
    for i in sorted(picks):
        lo = max(i - 40, 1)
        i0 = lo + int(np.argmax(env[lo: i + 41]))
        j = max(i0 - 40, 1) + int(np.argmax(corr[max(i0 - 40, 1): i0 + 41]))
        start = _peak(corr, j)
        s0 = int(start) - n // 10
        tmpl = x[s0: s0 + 2 * n]                                  # 録った合図（響きを含む）
        r0 = s0 + spacing - w - 40
        region = x[r0: r0 + 2 * w + 80 + 2 * n]
        if s0 < 0 or len(tmpl) < 2 * n or len(region) < 2 * w + 80 + 2 * n:
            raise ValueError("合図の前後の録音が足りません（最後の合図のあと 1 秒以上録る）")
        c2 = signal.correlate(region, tmpl, mode="valid", method="fft")
        end = r0 + _peak(c2, int(np.argmax(c2))) + (start - s0)
        out += [start, end]
        q += [env[i0] / floor, env[int(round(end))] / floor]
    return np.array(out), np.array(q)


def fit_clock(c1, c2, pairs):
    """合図の位置から eps と alpha を推定する

    c1, c2 : 機器1・機器2 で見つけた合図の位置（同じ順番）
    pairs  : 同じ場所で鳴らした合図の組 [(始め, 終わり), …]（例：[(0, 1), (2, 3)]）
             同じ場所の組の間隔の比を使うので、音が届くまでの時間（場所で変わる）は打ち消し合う
    戻り値 : dict(eps, alpha, eps_each)。alpha は最初の組の始めの合図で決める
    """
    c1, c2 = np.asarray(c1, float), np.asarray(c2, float)
    each = np.array([(c2[e] - c2[s]) / (c1[e] - c1[s]) - 1.0 for s, e in pairs])
    w = np.array([c1[e] - c1[s] for s, e in pairs])
    eps = float(np.sum(each * w) / np.sum(w))              # 間隔の長い組ほど精度がよいので重みをつける
    s0 = pairs[0][0]
    alpha = float(c2[s0] - (1 + eps) * c1[s0])
    return dict(eps=eps, alpha=alpha, eps_each=each)


def drift_search(x1, x2, fs, eps_max=200e-6, nfft=4096, hop=2048, coarse=2e-6, fine=0.05e-6):
    """合図を使わずに eps をさがす（相関の最大化）

    短い区間ごとの白色化した相互スペクトル G_w(f) = X1 X2* / |X1 X2*| は、機器2 が eps だけ速いと、
    区間の時刻 t_w に比例して位相が回る（機器2 の遅れ eps*t_w）。候補の eps で位相の回りを戻してから全区間を足し、
    遅れの関数（逆 FFT）のピークがいちばん高くなる eps を選ぶ。音源が2つでも、2つのピークが同じように動くので使える。
    戻り値：(eps の推定値, 候補の eps の配列, ピークの高さの配列)
    """
    f, t, X1 = signal.stft(x1, fs, nperseg=nfft, noverlap=nfft - hop, boundary=None, padded=False)
    _, _, X2 = signal.stft(x2, fs, nperseg=nfft, noverlap=nfft - hop, boundary=None, padded=False)
    G = X1 * np.conj(X2)
    G = G / (np.abs(G) + 1e-12)                            # (周波数, 区間)
    t = t - t.mean()

    def score(eps_list):
        out = []
        for e in eps_list:
            # 機器2 の信号は時刻 t で e*t 秒だけ遅れる → G = X1 X2* の位相が +2πf e t だけ回るので戻す
            S = np.sum(G * np.exp(-2j * np.pi * f[:, None] * e * t[None, :]), axis=1)
            r = np.fft.irfft(S, n=4 * nfft)
            out.append(np.max(np.abs(r)))
        return np.array(out)

    grid = np.arange(-eps_max, eps_max + coarse / 2, coarse)
    sc = score(grid)
    e0 = grid[np.argmax(sc)]
    g2 = np.arange(e0 - 2 * coarse, e0 + 2 * coarse + fine / 2, fine)
    s2 = score(g2)
    return float(g2[np.argmax(s2)]), np.r_[grid, g2], np.r_[sc, s2]


# ---------------------------------------------------------------------------
# 再生ファイルの並び（make_play.py と sim_async.py で共通）
#   [無音 0.2s][合図 0.3s][無音 0.7s][音源 seg_sec × nseg][無音 0.7s][合図 0.3s][無音 0.2s]
# ---------------------------------------------------------------------------
LEAD, CHIRP_SEC, GAP = 0.2, 0.3, 0.7
BODY_RMS = 0.05            # 音源の実効値（−26 dBFS）。声や楽器のピークは実効値の 10 倍前後なので割れない
CHIRP_AMP = 0.25           # 合図の振幅（実効値 −15 dBFS。音源より約 11dB 大きい。音量は鳴らす機器の側で控えめに）


def build_play(src, fs, nseg, seg_sec):
    """音源 src（fs）から再生ファイルを作り、(波形, 並びの情報) を返す。位置はファイルのサンプル番号"""
    ch = CHIRP_AMP * make_chirp(fs, CHIRP_SEC)
    body = np.asarray(src[: int(round(nseg * seg_sec * fs))], dtype=float)
    if len(body) < int(round(nseg * seg_sec * fs)):
        raise ValueError("音源が短すぎます")
    body = body / (np.sqrt(np.mean(body ** 2)) + 1e-12) * BODY_RMS      # 音源の大きさ（実効値）をそろえる
    if np.max(np.abs(body)) > 0.95:
        raise ValueError("音源のピークが大きすぎて割れます（BODY_RMS を下げる）")
    z = lambda s: np.zeros(int(round(s * fs)))
    parts = [z(LEAD), ch, z(GAP), body, z(GAP), ch, z(LEAD)]
    wave = np.concatenate(parts)
    c0 = int(round(LEAD * fs))
    s0 = c0 + len(ch) + int(round(GAP * fs))
    c1 = s0 + len(body) + int(round(GAP * fs))
    lay = dict(fs=fs, chirp_start=c0, chirp_end=c1,
               seg_starts=[s0 + int(round(j * seg_sec * fs)) for j in range(nseg)], seg_sec=seg_sec)
    return wave, lay


def seg_pos(c_start, c_end, lay, j):
    """録音の中で見つけた合図の位置 c_start・c_end（録音のサンプル番号）から、
    区間 j の始まりの位置を比例で求める（再生機・録音機どちらの時計のずれも自動で吸収される）"""
    u = (lay["seg_starts"][j] - lay["chirp_start"]) / (lay["chirp_end"] - lay["chirp_start"])
    return c_start + u * (c_end - c_start)


def build_mixture(rec1, rec2, c1, lays, j, L, eps, alpha, mode):
    """「2つの音源を同時に鳴らした」ときの2チャネルの観測を、1つずつ鳴らした録音の足し算で作る（重ね合わせ）

    rec1, rec2 : 機器1・機器2 の録音（同じ fs に直したもの）
    c1         : 機器1 で見つけた合図の位置 [ファイル1の始め, 終わり, ファイル2の始め, 終わり]
    lays       : 2つの再生ファイルの並びの情報
    j          : 使う区間の番号
    L          : 区間の長さ（機器1 のサンプル数）
    eps, alpha : 機器1 → 機器2 の対応 n2 = alpha + (1+eps) n1（推定値、または真の値）
    mode       : "none"     … 開始だけ合わせる（機器2 は自分の時計のサンプルの間隔のまま L 個読む）
                 "resample" … 機器2 を機器1 の時刻の格子で読み直す（ずれの補正）
                 どちらも同じ補間（frac_resample）を通すので、違いは読む間隔（1 か 1+eps か）だけ
    戻り値     : (観測 (2, L), 機器1 での音源ごとの観測 (2, L) ＝ 評価の正解)
    """
    parts1, ch2 = [], np.zeros(L)
    for k in range(2):
        n1 = seg_pos(c1[2 * k], c1[2 * k + 1], lays[k], j)
        a = int(round(n1))
        parts1.append(rec1[a: a + L])
        m = alpha + (1 + eps) * a
        if mode == "none":                                     # 小数の位置から、自分のサンプルの間隔のまま読む（丸めない）
            ch2 += frac_resample(rec2, m, 1.0, L)
        elif mode == "resample":
            ch2 += frac_resample(rec2, m, 1 + eps, L)
        else:
            raise ValueError(mode)
    parts1 = np.stack(parts1)
    return np.stack([parts1.sum(axis=0), ch2]), parts1

In [ ]:
import sys, re, shutil, pathlib, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
warnings.filterwarnings("ignore", message="Glyph")
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display

import bss, clockfix
from common import array as _arr                  # 8 節の --gen fft で使う（読み込めるかを先に確かめる）
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 本文の候補（0-1 のセルで用意したフォントが使われる）
matplotlib.rcParams["axes.unicode_minus"] = False
print("第7章の bss.py・clockfix.py・common/array.py を読み込めました：", bss.__file__, clockfix.__file__, _arr.__file__)


def fresh(*paths):
    """やり直せるように、前に作ったファイル・フォルダを消す（ノートブック版だけ。自分の実験のデータには使わない）"""
    for p in map(pathlib.Path, paths):
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists():
            p.unlink()


def listen_file(path, peak=0.3):
    """wav を再生ボタンにする。最大値が peak（約 −10 dBFS）を超えるときは小さくしてから鳴らす（大きくはしない）。normalize=False を消さない"""
    import numpy as np, soundfile   # 各節のプログラムが np などの名前を使うので、ここで読み直す
    x, fs = soundfile.read(path)
    m = np.max(np.abs(x))
    if m > peak:
        x = x * (peak / m)
    assert np.max(np.abs(x)) <= peak + 1e-9, "大きすぎる音は鳴らさない"
    display(Audio(x.T if x.ndim > 1 else x, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, i=1, text=None):
    """直前のセルの出力（cap.stdout）から、pattern の (…) の数を取り出す"""
    import re
    m = re.search(pattern, cap.stdout if text is None else text)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))   # 出力の「−」（全角のマイナス）も数にする


_ok, _num = ok, num   # 各節のプログラムが ok などの名前を使うことがあるので、確かめのセルで戻す

## 1. 自分の機器のずれの大きさを予想する（K-3-2、段階1 の 3）

K-3-2 の式で、区間の終わり（8 秒）でのずれ（サンプル数、16 kHz）と、2 kHz・4 kHz での位相の回転（度）を計算します。**先に紙で計算してから**実行してください（このセルはノートブックで足したものです）。

In [ ]:
%%capture cap
# ノートブック版で足したセル：K-3-2 の式  ずれ [サンプル] = ε × T × fs、位相の回転 [度] = 360 × f × ε × T
fs, T = 16000, 8.0
for eps_ppm in (10, 30, 100):
    eps = eps_ppm * 1e-6
    print(f"ε = {eps_ppm:3d} ppm：区間 {T:.0f} 秒の終わりで {eps * T * fs:.2f} サンプル（{eps * T * 1e6:.0f} µs）、"
          f"位相の回転 2kHz で {360 * 2000 * eps * T:.0f}°、4kHz で {360 * 4000 * eps * T:.0f}°")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
ok("ε =  30 ppm：区間 8 秒の終わりで 3.84 サンプル" in cap.stdout and "4kHz で 346°" in cap.stdout, "30 ppm で 3.84 サンプル、4kHz で 346°（図K-1 と同じ）")
ok("ε = 100 ppm：区間 8 秒の終わりで 12.80 サンプル" in cap.stdout and "4kHz で 1152°" in cap.stdout, "100 ppm で 12.8 サンプル、4kHz で 1152°")

## 2. シミュレーションの部品を試す（K-3-4、段階2 の 2、`sim_async.py`）

1 試行だけ動かします（約 20 秒〜1 分）。部屋の中で 2 つの再生ファイルを順番に鳴らし、機器2 の録音を「時計のずれ 30 ppm」で作り、sync（同期の理想）・none（開始だけ合わせる）・chirp（合図で推定して読み直す）の 3 条件で ILRMA の SDR 改善量を比べます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
fresh('t_try.csv')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ['sim_async.py', '--trials', '1', '--rt60', '0.256', '--eps', '30', '--conds', 'sync', 'none', 'chirp', '--out', 't_try.csv']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python sim_async.py --trials 1 --rt60 0.256 --eps 30 --conds sync none chirp --out t_try.csv
#!/usr/bin/env python3
"""最終演習K：2台の録音機による「非同期の2チャネル録音」をシミュレーションで作り、ずれの補正の効果を ILRMA で調べる

使い方: cd ~/home-lab/final/k
        uv run python sim_async.py --trials 6 --seed 1000 --k 2 4 --out sim_pilot.csv                 （予備実験）
        uv run python sim_async.py --trials 150 --seed 2000 --k 4 --out sim_main.csv                  （本実験。値は計画書のもの）
  --sources : 音源の wav、または話者ごとのフォルダ（中の wav・flac をつなぐ。JVS・LibriTTS の1人分など）。3つ以上。
              組み合わせを試行ごとに順に使う。省略すると librosa の例 libri1・libri2・trumpet（短い音源はくり返してつなぐ）
  --eps     : 機器2 のサンプリング周波数のずれ [ppm]（いくつでも）
  --rt60    : 残響時間 [s]（いくつでも。同じ部屋の形・置き方で吸音だけ変える）
  --k       : ILRMA の基底の数（いくつでも。予備実験で選び、本実験では1つに固定する）
  --seed    : 乱数の種のはじめ（試行 i は seed + i）。予備実験と本実験で重ならないようにする
  --gen fft / --drift 0.5 : 副次の確かめ。機器2 の録音を「直すのとは別の方法」で作る（作りと直しが同じだと、同等になるのが
              当たり前になる。inverse crime）、または時計のずれが時間とともに変わる録音にする
  --first   : 最初の試行の番号（例：--first 12 --trials 12 で後半だけ。分けて実行し、あとで CSV をつなぐ）

1回の試行（部屋の形・機器と音源の置き方・音源の組と区間）の中で、次の条件をすべて同じ録音から作る（対応のある比較）
  sync   : 同期した理想の録音機（機器2 の時計が機器1 と同じ。ε = 0）  … シミュレーションでしか作れない
  none   : 開始のずれだけ合図で合わせ、サンプリング周波数のずれは直さない
  chirp  : 合図（チャープ）で eps と開始のずれを推定し、機器2 を読み直す
  blind  : eps だけを、合図を使わずに none の観測（開始のずれは合図で合わせたもの）から推定し、
           その eps で並べ直して読み直す（合図は開始の位置だけに使う）
  oracle : eps だけ真の値で読み直す（chirp との差＝eps の推定の誤差の影響）
  機器2 の録音は、どの条件も同じ経路（大きさ ±6dB・帯域制限の読み直し・同じ雑音）で作り、開始は合図の始めの位置で合わせる。
  そのため sync と oracle の差は「ずれのある時計で録って、読み直したこと」だけによる。
結果は1行＝試行×T60×ずれ×条件×K×初期値の種。sdr などは2つの音源の平均。所要時間の目安は K-2 に書いた
"""
import argparse
import itertools
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import librosa
import pyroomacoustics as pra
import fast_bss_eval

sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
import bss                                                   # 第7章の自前の ILRMA（~/home-lab/ch7/bss.py）
import clockfix as cf

try:                                                         # IV-5 の共通ライブラリ（--gen fft のときだけ使う）
    sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
    from common import array as arr
except ImportError:
    arr = None

FS = 16000
NFFT, HOP = 2048, 512                                        # 第7章と同じ（16kHz で 128ms の窓）
SNR_DB = 35.0                                                # 録音機の雑音（静かな部屋の目安）


def load_sources(paths, fs):
    if not paths:
        paths = [librosa.ex(n) for n in ("libri1", "libri2", "trumpet")]
    out = []
    for p in paths:
        p = pathlib.Path(p)
        if p.is_dir():                                         # フォルダなら、中の wav・flac を名前の順につなぐ（1人の話者＝1つの音源）
            files = sorted([*p.rglob("*.wav"), *p.rglob("*.flac")])[:60]
            x = np.concatenate([librosa.effects.trim(librosa.load(f, sr=fs, mono=True)[0], top_db=40)[0] for f in files])
        else:
            x, _ = librosa.load(p, sr=fs, mono=True)
        x = x / (np.std(x) + 1e-9)
        out.append(x)
    return out


def excerpt(x, n, rng):
    """長さ n を切り出す（短い音源はくり返してつなぐ。つなぎ目は 20ms で重ねる）"""
    while len(x) < n + 1:
        r = int(0.02 * FS)
        fade = np.linspace(0, 1, r)
        x = np.r_[x[:-r], x[-r:] * (1 - fade) + x[:r] * fade, x[r:]]
    a = rng.integers(0, len(x) - n)
    return x[a: a + n]


def place(rng):
    """家の部屋（6〜8畳くらい）と、机の上の2台の録音機、2つの音源の位置"""
    dim = np.array([rng.uniform(3.2, 4.5), rng.uniform(2.6, 3.6), 2.4])
    while True:
        c = np.array([rng.uniform(1.2, dim[0] - 1.2), rng.uniform(1.2, dim[1] - 1.2), 0.8])
        d = rng.uniform(0.3, 1.0)                              # 2台の間隔 30〜100cm
        th = rng.uniform(0, np.pi)
        u = np.array([np.cos(th), np.sin(th), 0])
        mics = np.c_[c - d / 2 * u, c + d / 2 * u]
        ang = rng.uniform(0, 2 * np.pi, 2)
        diff = abs((ang[0] - ang[1] + np.pi) % (2 * np.pi) - np.pi)
        if diff < np.radians(40):
            continue
        srcs = []
        for a in ang:
            r = rng.uniform(0.8, 1.6)
            srcs.append(c + np.array([r * np.cos(a), r * np.sin(a), rng.uniform(0.3, 0.5)]))
        if not all(np.all(s[:2] > 0.4) and np.all(s[:2] < dim[:2] - 0.4) for s in srcs):
            continue
        # 2本のマイクで見分けられる置き方か：2つの音源の到達時間差（マイク2 − マイク1）が 3 サンプル以上違うこと。
        # マイクを結ぶ線をはさんで鏡に映した位置や、2台の真ん中の垂直な面の近くに2つとも置くと、到達時間差が同じになり分けられない
        tdoa = [(np.linalg.norm(s - mics[:, 1]) - np.linalg.norm(s - mics[:, 0])) / 343.0 * FS for s in srcs]
        if abs(tdoa[0] - tdoa[1]) >= 3.0:
            return dim, mics, srcs, d


def simulate(dim, mics, srcs, rt60, session):
    """部屋の中で、2つの再生ファイルを順番に鳴らしたときの、2本のマイクの連続した信号（同期・理想）"""
    if rt60 == 0:                                              # 無響（直接音だけ）
        room = pra.ShoeBox(dim, fs=FS, max_order=0)
    else:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    room.add_microphone_array(mics)
    for k in range(2):
        room.add_source(srcs[k], signal=session[k])
    room.simulate()
    return room.mic_array.signals                               # (2, 時間)


def evaluate(ref, est):
    sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return float(np.mean(sdr)), float(np.mean(sir)), float(np.mean(sar))


def stft(x):
    return np.stack([librosa.stft(c, n_fft=NFFT, hop_length=HOP) for c in x], axis=-1)     # (F, T, M)


def istft(Y, n):
    return np.stack([librosa.istft(Y[:, :, k], hop_length=HOP, length=n) for k in range(Y.shape[2])])


def separate(mix, k, n_iter, seed):
    return istft(bss.ilrma(stft(mix), k, n_iter, seed=seed)[0], mix.shape[1])


def oracle_w(mix, ref):
    """正解を使った「時間に依らない分離行列」の上限：周波数ごとに W_f = R_sx R_xx⁻¹（最小二乗）

    ILRMA と同じ「周波数ごとに1つの W_f」というモデルで、正解を知っていれば到達できる成績。
    これが下がるなら、ずれのせいで「モデルそのもの」が合わなくなっている。これが高いのに ILRMA だけ下がるなら「推定」のせい
    """
    X, S = stft(mix), stft(ref)
    Rxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Rsx = np.einsum("ftn,ftm->fnm", S, X.conj())
    W = Rsx @ np.linalg.inv(Rxx + 1e-9 * np.trace(Rxx, axis1=1, axis2=2)[:, None, None] * np.eye(2))
    return istft(np.einsum("fnm,ftm->ftn", W, X), mix.shape[1])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sources", nargs="*")
    ap.add_argument("--trials", type=int, default=2)
    ap.add_argument("--first", type=int, default=0, help="最初の試行の番号（何回かに分けて実行するとき）")
    ap.add_argument("--seed", type=int, default=1000)
    ap.add_argument("--eps", type=float, nargs="+", default=[10, 30, 100])
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.0, 0.256])
    ap.add_argument("--init-seeds", type=int, default=1, help="ILRMA の初期値の乱数の種をいくつ試すか")
    ap.add_argument("--k", type=int, nargs="+", default=[2])
    ap.add_argument("--dur", type=float, default=8.0)
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--conds", nargs="+", default=["sync", "none", "chirp", "blind", "oracle"])
    ap.add_argument("--gen", choices=["sinc", "fft"], default="sinc",
                    help="機器2 の録音の作り方（副次の確かめ）。fft は IV-5 の common/array.py の sfo_resample（直すほうとは別の方法）")
    ap.add_argument("--drift", type=float, default=0.0,
                    help="機器2 の時計のずれが時間とともに変わる速さ [ppm/分]（副次の確かめ。ε(t) = ε + drift·t）")
    ap.add_argument("--out", default="sim_pilot.csv")
    a = ap.parse_args()
    if pathlib.Path(a.out).exists():
        sys.exit(f"{a.out} はすでにあります（上書きしないように止めました）")
    srcs = load_sources(a.sources, FS)
    pairs = list(itertools.combinations(range(len(srcs)), 2))
    L = int(a.dur * FS)
    chirp = cf.make_chirp(FS, cf.CHIRP_SEC)
    rows, t_all = [], time.time()
    for i in range(a.first, a.first + a.trials):
        seed = a.seed + i
        rng = np.random.default_rng(seed)
        pair = pairs[i % len(pairs)]
        dim, mics, spos, d_mic = place(rng)
        # 再生ファイル2つ（音源ごとに区間を乱数で選ぶ）と、それを順番に鳴らすセッション
        plays, lays = [], []
        for k in range(2):
            w, lay = cf.build_play(excerpt(srcs[pair[k]], L, rng), FS, 1, a.dur)
            plays.append(w)
            lays.append(lay)
        spacing = lays[0]["chirp_end"] - lays[0]["chirp_start"]   # 1つのファイルの中の合図の間隔（サンプル）
        pad, gap = int(1.5 * FS), int(1.0 * FS)
        total = pad + len(plays[0]) + gap + len(plays[1]) + pad
        session = np.zeros((2, total))
        session[0, pad: pad + len(plays[0])] = plays[0]
        session[1, pad + len(plays[0]) + gap: pad + len(plays[0]) + gap + len(plays[1])] = plays[1]
        s0 = rng.uniform(0.2, 1.0) * FS                          # 機器2 が録音を始めた位置（機器1 のサンプル番号）
        gain2 = 10 ** (rng.uniform(-6, 6) / 20)                  # 機器2 の録音の大きさ（機器ごとに違う）
        for rt60 in a.rt60:
            mic = simulate(dim, mics, spos, rt60, session)[:, :total]
            sig_rms = np.sqrt(np.mean(mic[:, pad: -pad] ** 2))
            # 雑音は条件・ずれによらず同じものを使う（試行と T60 ごとに別の乱数の種から作る）
            nrng = np.random.default_rng([seed, int(rt60 * 1000), 7])
            n2max = int((total - s0 - FS) * (1 + max(list(a.eps) + [0.0]) * 1e-6)) + 1
            noise1 = nrng.standard_normal(total) * sig_rms * 10 ** (-SNR_DB / 20)
            noise2 = nrng.standard_normal(n2max) * gain2 * sig_rms * 10 ** (-SNR_DB / 20)
            rec1 = mic[0] + noise1
            c1, q1 = cf.find_chirps(rec1, chirp, spacing)
            for eps_ppm in [0.0] + list(a.eps):
                eps = eps_ppm * 1e-6
                # 機器2 の録音：sync（ε = 0 の理想の録音機）も、ずれのある録音機も、同じ経路（大きさ・読み直し・雑音）で作る
                n2 = int((total - s0 - FS) * (1 + eps))
                if a.gen == "fft":                                     # 作るほうを別の方法（FFT の読み直し）にする
                    s0i = int(round(s0))
                    y2, _ = arr.sfo_resample(mic[1][s0i:], eps)
                    rec2 = gain2 * y2[:n2] + noise2[:n2]
                elif a.drift and eps_ppm != 0:                        # ε(t) = ε + β t（t は機器1 の秒）。sync（ε=0）は同期のまま
                    beta = a.drift * 1e-6 / 60
                    nn = np.arange(n2)
                    rec2 = gain2 * cf.frac_read(mic[1], s0 + nn / (1 + eps) - beta * nn ** 2 / (2 * FS)) + noise2[:n2]
                else:
                    rec2 = gain2 * cf.frac_resample(mic[1], s0, 1 / (1 + eps), n2) + noise2[:n2]
                c2, q2 = cf.find_chirps(rec2, chirp, spacing)
                clk = cf.fit_clock(c1, c2, [(0, 1), (2, 3)])
                # 開始の合わせ方はどの条件も同じ（合図の始めの位置。P1 から2台までの到達時間の差が一定の遅れとして入る）
                start = lambda e: float(c2[0] - (1 + e) * c1[0])
                conds = {}
                if eps_ppm == 0.0:
                    if "sync" in a.conds:
                        conds["sync"] = (0.0, start(0.0), "resample", np.nan)
                else:
                    if "none" in a.conds or "blind" in a.conds:
                        conds["none"] = (clk["eps"], clk["alpha"], "none", clk["eps"])
                    if "chirp" in a.conds:
                        conds["chirp"] = (clk["eps"], clk["alpha"], "resample", clk["eps"])
                    if "oracle" in a.conds:
                        conds["oracle"] = (eps, start(eps), "resample", eps)    # ε だけ真の値（開始は合図）
                for name, (e, al, mode, e_hat) in conds.items():
                    mix, ref = cf.build_mixture(rec1, rec2, c1, lays, 0, L, e, al, mode)
                    todo = [(name, mix, e_hat)]
                    if name == "none" and "blind" in a.conds:
                        eb = cf.drift_search(mix[0], mix[1], FS)[0]           # none の観測から ε だけを推定
                        mb, _ = cf.build_mixture(rec1, rec2, c1, lays, 0, L, eb, start(eb), "resample")
                        todo.append(("blind", mb, eb))
                        if "none" not in a.conds:
                            todo = todo[1:]
                    with np.errstate(divide="ignore"):
                        sdr_in = evaluate(ref, np.stack([mix[0], mix[0]]))[0]
                    for cname, m, eh in todo:
                        orw = evaluate(ref, oracle_w(m, ref))[0] - sdr_in
                        for k in a.k:
                            for s_init in range(a.init_seeds):
                                t0 = time.time()
                                y = separate(m, k, a.iter, seed + 100000 * s_init)
                                sdr, sir, sar = evaluate(ref, y)
                                rows.append(dict(trial=i, seed=seed, init=s_init, gen=a.gen, drift=a.drift if eps_ppm else 0.0,
                                                 pair=f"{pair[0]}-{pair[1]}",
                                                 d_mic=round(d_mic, 3), rt60=rt60, eps_ppm=eps_ppm, cond=cname, k=k,
                                                 eps_hat_ppm=eh * 1e6, sdr_in=sdr_in, sdr=sdr, sir=sir, sar=sar,
                                                 sdr_imp=sdr - sdr_in, orw_imp=orw,
                                                 chirp_q=float(min(q1.min(), q2.min())),
                                                 sec=time.time() - t0))
                                print(f"試行 {i:2d} T60 {rt60*1000:3.0f}ms ずれ {eps_ppm:5.0f}ppm {cname:6s} K={k} 初期値{s_init}"
                                      f"  推定 {eh*1e6:8.2f}ppm  SDR改善 {sdr - sdr_in:6.2f} dB（正解の W なら {orw:6.2f}）"
                                      f"  {time.time()-t0:4.1f}秒", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)       # 途中でも保存する
    print(f"{a.out} に保存しました（{len(rows)} 行、{(time.time()-t_all)/60:.1f} 分）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
pat = r"ずれ\s+(\d+)ppm (\w+)\s+K=2 初期値0  推定\s+(\S+)ppm  SDR改善\s+([-\d.]+) dB（正解の W なら\s+([-\d.]+)）"
r = {m.group(2): (float(m.group(3)), float(m.group(4)), float(m.group(5))) for m in re.finditer(pat, cap.stdout)}
ok(set(r) == {"sync", "none", "chirp"}, "3 条件（sync・none・chirp）が 1 行ずつ表示された")
ok(abs(r["chirp"][0] - 30) < 0.05, f"合図による推定が 30.01 ppm 前後（{r['chirp'][0]} ppm。誤差 0.02 ppm 程度）")
ok(abs(r["sync"][1] - r["chirp"][1]) <= 0.2 and r["none"][1] < r["chirp"][1],
   f"SDR 改善量が sync（{r['sync'][1]}）と chirp（{r['chirp'][1]}）でほぼ同じで、none（{r['none'][1]}）はそれより低い（見本 6.57・6.56・4.99 dB）")
ok(abs(r["sync"][2] - r["chirp"][2]) <= 0.05 and r["none"][2] < r["sync"][2],
   f"「正解の W なら」が sync・chirp で同じ（{r['sync'][2]}・{r['chirp'][2]}。見本 10.99 dB）、none で少し低い（{r['none'][2]}。見本 10.02 dB）")

## 3. 予備実験で K・S・n を決める（K-4-2・K-4-3、段階2 の 3・4、`sim_async.py`・`fink_power.py`）

**ノートブックの既定は試しの設定**です。教科書の予備実験（`sim_pilot.csv` は 6 試行・ずれ 10・30・100 ppm・K の候補 2 と 4 で約 30 分、`sim_init.csv` は 6 試行・初期値の種 4 つで約 15 分）のかわりに、`sim_pilot.csv` を **2 試行・ずれ 10・30 ppm だけ**（約 6 分）、`sim_init.csv` を **2 試行・初期値の種 2 つ**（約 3 分）にしています。本番にするときは、セルの `sys.argv` の行をコメントの行に取りかえてください。乱数の種は 1000〜・1100〜（本実験の 2000〜 と重ならない）です。

試しの設定では試行が少ないので、選ばれる K、初期値の種だけでの標準偏差（教科書の見本 約 3.2 dB）、S・n（見本 S = 1、n = 150）は教科書と合わないことがあります。**本当の計画は、本番の設定の予備実験で決めます。** Δ（同等とみなす幅、見本 1.0 dB）は結果を見る前に決めておきます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
# 本番（教科書の段階2 の 3。約30分）：sys.argv = ["sim_async.py", "--trials", "6", "--seed", "1000", "--k", "2", "4", "--out", "sim_pilot.csv"]
fresh('sim_pilot.csv')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ['sim_async.py', '--trials', '2', '--seed', '1000', '--k', '2', '4', '--eps', '10', '30', '--out', 'sim_pilot.csv']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python sim_async.py --trials 2 --seed 1000 --k 2 4 --eps 10 30 --out sim_pilot.csv
#!/usr/bin/env python3
"""最終演習K：2台の録音機による「非同期の2チャネル録音」をシミュレーションで作り、ずれの補正の効果を ILRMA で調べる

使い方: cd ~/home-lab/final/k
        uv run python sim_async.py --trials 6 --seed 1000 --k 2 4 --out sim_pilot.csv                 （予備実験）
        uv run python sim_async.py --trials 150 --seed 2000 --k 4 --out sim_main.csv                  （本実験。値は計画書のもの）
  --sources : 音源の wav、または話者ごとのフォルダ（中の wav・flac をつなぐ。JVS・LibriTTS の1人分など）。3つ以上。
              組み合わせを試行ごとに順に使う。省略すると librosa の例 libri1・libri2・trumpet（短い音源はくり返してつなぐ）
  --eps     : 機器2 のサンプリング周波数のずれ [ppm]（いくつでも）
  --rt60    : 残響時間 [s]（いくつでも。同じ部屋の形・置き方で吸音だけ変える）
  --k       : ILRMA の基底の数（いくつでも。予備実験で選び、本実験では1つに固定する）
  --seed    : 乱数の種のはじめ（試行 i は seed + i）。予備実験と本実験で重ならないようにする
  --gen fft / --drift 0.5 : 副次の確かめ。機器2 の録音を「直すのとは別の方法」で作る（作りと直しが同じだと、同等になるのが
              当たり前になる。inverse crime）、または時計のずれが時間とともに変わる録音にする
  --first   : 最初の試行の番号（例：--first 12 --trials 12 で後半だけ。分けて実行し、あとで CSV をつなぐ）

1回の試行（部屋の形・機器と音源の置き方・音源の組と区間）の中で、次の条件をすべて同じ録音から作る（対応のある比較）
  sync   : 同期した理想の録音機（機器2 の時計が機器1 と同じ。ε = 0）  … シミュレーションでしか作れない
  none   : 開始のずれだけ合図で合わせ、サンプリング周波数のずれは直さない
  chirp  : 合図（チャープ）で eps と開始のずれを推定し、機器2 を読み直す
  blind  : eps だけを、合図を使わずに none の観測（開始のずれは合図で合わせたもの）から推定し、
           その eps で並べ直して読み直す（合図は開始の位置だけに使う）
  oracle : eps だけ真の値で読み直す（chirp との差＝eps の推定の誤差の影響）
  機器2 の録音は、どの条件も同じ経路（大きさ ±6dB・帯域制限の読み直し・同じ雑音）で作り、開始は合図の始めの位置で合わせる。
  そのため sync と oracle の差は「ずれのある時計で録って、読み直したこと」だけによる。
結果は1行＝試行×T60×ずれ×条件×K×初期値の種。sdr などは2つの音源の平均。所要時間の目安は K-2 に書いた
"""
import argparse
import itertools
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import librosa
import pyroomacoustics as pra
import fast_bss_eval

sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
import bss                                                   # 第7章の自前の ILRMA（~/home-lab/ch7/bss.py）
import clockfix as cf

try:                                                         # IV-5 の共通ライブラリ（--gen fft のときだけ使う）
    sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
    from common import array as arr
except ImportError:
    arr = None

FS = 16000
NFFT, HOP = 2048, 512                                        # 第7章と同じ（16kHz で 128ms の窓）
SNR_DB = 35.0                                                # 録音機の雑音（静かな部屋の目安）


def load_sources(paths, fs):
    if not paths:
        paths = [librosa.ex(n) for n in ("libri1", "libri2", "trumpet")]
    out = []
    for p in paths:
        p = pathlib.Path(p)
        if p.is_dir():                                         # フォルダなら、中の wav・flac を名前の順につなぐ（1人の話者＝1つの音源）
            files = sorted([*p.rglob("*.wav"), *p.rglob("*.flac")])[:60]
            x = np.concatenate([librosa.effects.trim(librosa.load(f, sr=fs, mono=True)[0], top_db=40)[0] for f in files])
        else:
            x, _ = librosa.load(p, sr=fs, mono=True)
        x = x / (np.std(x) + 1e-9)
        out.append(x)
    return out


def excerpt(x, n, rng):
    """長さ n を切り出す（短い音源はくり返してつなぐ。つなぎ目は 20ms で重ねる）"""
    while len(x) < n + 1:
        r = int(0.02 * FS)
        fade = np.linspace(0, 1, r)
        x = np.r_[x[:-r], x[-r:] * (1 - fade) + x[:r] * fade, x[r:]]
    a = rng.integers(0, len(x) - n)
    return x[a: a + n]


def place(rng):
    """家の部屋（6〜8畳くらい）と、机の上の2台の録音機、2つの音源の位置"""
    dim = np.array([rng.uniform(3.2, 4.5), rng.uniform(2.6, 3.6), 2.4])
    while True:
        c = np.array([rng.uniform(1.2, dim[0] - 1.2), rng.uniform(1.2, dim[1] - 1.2), 0.8])
        d = rng.uniform(0.3, 1.0)                              # 2台の間隔 30〜100cm
        th = rng.uniform(0, np.pi)
        u = np.array([np.cos(th), np.sin(th), 0])
        mics = np.c_[c - d / 2 * u, c + d / 2 * u]
        ang = rng.uniform(0, 2 * np.pi, 2)
        diff = abs((ang[0] - ang[1] + np.pi) % (2 * np.pi) - np.pi)
        if diff < np.radians(40):
            continue
        srcs = []
        for a in ang:
            r = rng.uniform(0.8, 1.6)
            srcs.append(c + np.array([r * np.cos(a), r * np.sin(a), rng.uniform(0.3, 0.5)]))
        if not all(np.all(s[:2] > 0.4) and np.all(s[:2] < dim[:2] - 0.4) for s in srcs):
            continue
        # 2本のマイクで見分けられる置き方か：2つの音源の到達時間差（マイク2 − マイク1）が 3 サンプル以上違うこと。
        # マイクを結ぶ線をはさんで鏡に映した位置や、2台の真ん中の垂直な面の近くに2つとも置くと、到達時間差が同じになり分けられない
        tdoa = [(np.linalg.norm(s - mics[:, 1]) - np.linalg.norm(s - mics[:, 0])) / 343.0 * FS for s in srcs]
        if abs(tdoa[0] - tdoa[1]) >= 3.0:
            return dim, mics, srcs, d


def simulate(dim, mics, srcs, rt60, session):
    """部屋の中で、2つの再生ファイルを順番に鳴らしたときの、2本のマイクの連続した信号（同期・理想）"""
    if rt60 == 0:                                              # 無響（直接音だけ）
        room = pra.ShoeBox(dim, fs=FS, max_order=0)
    else:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    room.add_microphone_array(mics)
    for k in range(2):
        room.add_source(srcs[k], signal=session[k])
    room.simulate()
    return room.mic_array.signals                               # (2, 時間)


def evaluate(ref, est):
    sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return float(np.mean(sdr)), float(np.mean(sir)), float(np.mean(sar))


def stft(x):
    return np.stack([librosa.stft(c, n_fft=NFFT, hop_length=HOP) for c in x], axis=-1)     # (F, T, M)


def istft(Y, n):
    return np.stack([librosa.istft(Y[:, :, k], hop_length=HOP, length=n) for k in range(Y.shape[2])])


def separate(mix, k, n_iter, seed):
    return istft(bss.ilrma(stft(mix), k, n_iter, seed=seed)[0], mix.shape[1])


def oracle_w(mix, ref):
    """正解を使った「時間に依らない分離行列」の上限：周波数ごとに W_f = R_sx R_xx⁻¹（最小二乗）

    ILRMA と同じ「周波数ごとに1つの W_f」というモデルで、正解を知っていれば到達できる成績。
    これが下がるなら、ずれのせいで「モデルそのもの」が合わなくなっている。これが高いのに ILRMA だけ下がるなら「推定」のせい
    """
    X, S = stft(mix), stft(ref)
    Rxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Rsx = np.einsum("ftn,ftm->fnm", S, X.conj())
    W = Rsx @ np.linalg.inv(Rxx + 1e-9 * np.trace(Rxx, axis1=1, axis2=2)[:, None, None] * np.eye(2))
    return istft(np.einsum("fnm,ftm->ftn", W, X), mix.shape[1])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sources", nargs="*")
    ap.add_argument("--trials", type=int, default=2)
    ap.add_argument("--first", type=int, default=0, help="最初の試行の番号（何回かに分けて実行するとき）")
    ap.add_argument("--seed", type=int, default=1000)
    ap.add_argument("--eps", type=float, nargs="+", default=[10, 30, 100])
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.0, 0.256])
    ap.add_argument("--init-seeds", type=int, default=1, help="ILRMA の初期値の乱数の種をいくつ試すか")
    ap.add_argument("--k", type=int, nargs="+", default=[2])
    ap.add_argument("--dur", type=float, default=8.0)
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--conds", nargs="+", default=["sync", "none", "chirp", "blind", "oracle"])
    ap.add_argument("--gen", choices=["sinc", "fft"], default="sinc",
                    help="機器2 の録音の作り方（副次の確かめ）。fft は IV-5 の common/array.py の sfo_resample（直すほうとは別の方法）")
    ap.add_argument("--drift", type=float, default=0.0,
                    help="機器2 の時計のずれが時間とともに変わる速さ [ppm/分]（副次の確かめ。ε(t) = ε + drift·t）")
    ap.add_argument("--out", default="sim_pilot.csv")
    a = ap.parse_args()
    if pathlib.Path(a.out).exists():
        sys.exit(f"{a.out} はすでにあります（上書きしないように止めました）")
    srcs = load_sources(a.sources, FS)
    pairs = list(itertools.combinations(range(len(srcs)), 2))
    L = int(a.dur * FS)
    chirp = cf.make_chirp(FS, cf.CHIRP_SEC)
    rows, t_all = [], time.time()
    for i in range(a.first, a.first + a.trials):
        seed = a.seed + i
        rng = np.random.default_rng(seed)
        pair = pairs[i % len(pairs)]
        dim, mics, spos, d_mic = place(rng)
        # 再生ファイル2つ（音源ごとに区間を乱数で選ぶ）と、それを順番に鳴らすセッション
        plays, lays = [], []
        for k in range(2):
            w, lay = cf.build_play(excerpt(srcs[pair[k]], L, rng), FS, 1, a.dur)
            plays.append(w)
            lays.append(lay)
        spacing = lays[0]["chirp_end"] - lays[0]["chirp_start"]   # 1つのファイルの中の合図の間隔（サンプル）
        pad, gap = int(1.5 * FS), int(1.0 * FS)
        total = pad + len(plays[0]) + gap + len(plays[1]) + pad
        session = np.zeros((2, total))
        session[0, pad: pad + len(plays[0])] = plays[0]
        session[1, pad + len(plays[0]) + gap: pad + len(plays[0]) + gap + len(plays[1])] = plays[1]
        s0 = rng.uniform(0.2, 1.0) * FS                          # 機器2 が録音を始めた位置（機器1 のサンプル番号）
        gain2 = 10 ** (rng.uniform(-6, 6) / 20)                  # 機器2 の録音の大きさ（機器ごとに違う）
        for rt60 in a.rt60:
            mic = simulate(dim, mics, spos, rt60, session)[:, :total]
            sig_rms = np.sqrt(np.mean(mic[:, pad: -pad] ** 2))
            # 雑音は条件・ずれによらず同じものを使う（試行と T60 ごとに別の乱数の種から作る）
            nrng = np.random.default_rng([seed, int(rt60 * 1000), 7])
            n2max = int((total - s0 - FS) * (1 + max(list(a.eps) + [0.0]) * 1e-6)) + 1
            noise1 = nrng.standard_normal(total) * sig_rms * 10 ** (-SNR_DB / 20)
            noise2 = nrng.standard_normal(n2max) * gain2 * sig_rms * 10 ** (-SNR_DB / 20)
            rec1 = mic[0] + noise1
            c1, q1 = cf.find_chirps(rec1, chirp, spacing)
            for eps_ppm in [0.0] + list(a.eps):
                eps = eps_ppm * 1e-6
                # 機器2 の録音：sync（ε = 0 の理想の録音機）も、ずれのある録音機も、同じ経路（大きさ・読み直し・雑音）で作る
                n2 = int((total - s0 - FS) * (1 + eps))
                if a.gen == "fft":                                     # 作るほうを別の方法（FFT の読み直し）にする
                    s0i = int(round(s0))
                    y2, _ = arr.sfo_resample(mic[1][s0i:], eps)
                    rec2 = gain2 * y2[:n2] + noise2[:n2]
                elif a.drift and eps_ppm != 0:                        # ε(t) = ε + β t（t は機器1 の秒）。sync（ε=0）は同期のまま
                    beta = a.drift * 1e-6 / 60
                    nn = np.arange(n2)
                    rec2 = gain2 * cf.frac_read(mic[1], s0 + nn / (1 + eps) - beta * nn ** 2 / (2 * FS)) + noise2[:n2]
                else:
                    rec2 = gain2 * cf.frac_resample(mic[1], s0, 1 / (1 + eps), n2) + noise2[:n2]
                c2, q2 = cf.find_chirps(rec2, chirp, spacing)
                clk = cf.fit_clock(c1, c2, [(0, 1), (2, 3)])
                # 開始の合わせ方はどの条件も同じ（合図の始めの位置。P1 から2台までの到達時間の差が一定の遅れとして入る）
                start = lambda e: float(c2[0] - (1 + e) * c1[0])
                conds = {}
                if eps_ppm == 0.0:
                    if "sync" in a.conds:
                        conds["sync"] = (0.0, start(0.0), "resample", np.nan)
                else:
                    if "none" in a.conds or "blind" in a.conds:
                        conds["none"] = (clk["eps"], clk["alpha"], "none", clk["eps"])
                    if "chirp" in a.conds:
                        conds["chirp"] = (clk["eps"], clk["alpha"], "resample", clk["eps"])
                    if "oracle" in a.conds:
                        conds["oracle"] = (eps, start(eps), "resample", eps)    # ε だけ真の値（開始は合図）
                for name, (e, al, mode, e_hat) in conds.items():
                    mix, ref = cf.build_mixture(rec1, rec2, c1, lays, 0, L, e, al, mode)
                    todo = [(name, mix, e_hat)]
                    if name == "none" and "blind" in a.conds:
                        eb = cf.drift_search(mix[0], mix[1], FS)[0]           # none の観測から ε だけを推定
                        mb, _ = cf.build_mixture(rec1, rec2, c1, lays, 0, L, eb, start(eb), "resample")
                        todo.append(("blind", mb, eb))
                        if "none" not in a.conds:
                            todo = todo[1:]
                    with np.errstate(divide="ignore"):
                        sdr_in = evaluate(ref, np.stack([mix[0], mix[0]]))[0]
                    for cname, m, eh in todo:
                        orw = evaluate(ref, oracle_w(m, ref))[0] - sdr_in
                        for k in a.k:
                            for s_init in range(a.init_seeds):
                                t0 = time.time()
                                y = separate(m, k, a.iter, seed + 100000 * s_init)
                                sdr, sir, sar = evaluate(ref, y)
                                rows.append(dict(trial=i, seed=seed, init=s_init, gen=a.gen, drift=a.drift if eps_ppm else 0.0,
                                                 pair=f"{pair[0]}-{pair[1]}",
                                                 d_mic=round(d_mic, 3), rt60=rt60, eps_ppm=eps_ppm, cond=cname, k=k,
                                                 eps_hat_ppm=eh * 1e6, sdr_in=sdr_in, sdr=sdr, sir=sir, sar=sar,
                                                 sdr_imp=sdr - sdr_in, orw_imp=orw,
                                                 chirp_q=float(min(q1.min(), q2.min())),
                                                 sec=time.time() - t0))
                                print(f"試行 {i:2d} T60 {rt60*1000:3.0f}ms ずれ {eps_ppm:5.0f}ppm {cname:6s} K={k} 初期値{s_init}"
                                      f"  推定 {eh*1e6:8.2f}ppm  SDR改善 {sdr - sdr_in:6.2f} dB（正解の W なら {orw:6.2f}）"
                                      f"  {time.time()-t0:4.1f}秒", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)       # 途中でも保存する
    print(f"{a.out} に保存しました（{len(rows)} 行、{(time.time()-t_all)/60:.1f} 分）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
import pandas as pd
d = pd.read_csv("sim_pilot.csv")
nt = d["trial"].nunique()
ne = d.loc[d["eps_ppm"] > 0, "eps_ppm"].nunique()
ok(len(d) == nt * 2 * (1 + 4 * ne) * 2, f"sim_pilot.csv は {nt} 試行 × 2 T60 × {1 + 4 * ne} 観測（sync と、ずれ {ne} 通りの none・chirp・blind・oracle）× 2 K ＝ {len(d)} 行"
   "（教科書の本番は 6 試行 × 2 T60 × 13 観測 × 2 K ＝ 312 行）")
ch = d[d["cond"] == "chirp"]
ok((ch["eps_hat_ppm"] - ch["eps_ppm"]).abs().max() < 0.05, f"合図による ε の推定の誤差が 0.05 ppm 未満（最大 {(ch['eps_hat_ppm'] - ch['eps_ppm']).abs().max():.3f} ppm）")
sy = d[d["cond"] == "sync"].groupby("rt60")["sdr_imp"].mean()
ok(sy[0.0] > 10 and 1 < sy[0.256] < sy[0.0], f"同期（sync）の SDR 改善量が T60 0ms で十数dB（{sy[0.0]:.2f}）、256ms で数dB（{sy[0.256]:.2f}）")
m = d[d["eps_ppm"] == 30].groupby("cond")["sdr_imp"].mean()
ok(m["none"] < m["chirp"], f"30 ppm の none（平均 {m['none']:.2f} dB）は chirp（{m['chirp']:.2f} dB）より低い（教科書：none は 100 ppm で 0dB 前後まで下がる）")

1 回目の `fink_power.py`：予備実験の sync（同期した理想の録音）の SDR 改善量の平均で K を選びます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['fink_power.py', 'sim_pilot.csv']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python fink_power.py sim_pilot.csv
#!/usr/bin/env python3
"""最終演習K：予備実験の結果から、K を選び、初期値の種の数 S と試行の数 n を決める（検出力の計算）

使い方: cd ~/home-lab/final/k
  uv run python fink_power.py sim_pilot.csv                       （1回目：K を選ぶだけ）
  uv run python fink_power.py sim_pilot.csv sim_init.csv --eps 30 --delta 1.0 --sesoi 1.0   （2回目：S と n を決める）
  sim_pilot.csv : 予備実験（K の候補を全部、初期値の種は1つ）
  sim_init.csv  : 初期値の種だけを変えた実験（1回目で選んだ K、30 ppm、sync・none・chirp・blind、種 4 つ）
  --delta 1.0   : 同等性の幅 Δ [dB]（事前登録の値。計画書に理由を書く）
  --sesoi 1.0   : 「補正の効果」として見逃したくない最小の大きさ [dB]
  --alpha 0.05 --power 0.8 --tests 3 : 有意水準・検出力・確証的な検定の数（ホルム法の最悪の場合 α/3 で計画する）

やること
  1. K の選び方：予備実験の sync（同期した理想の録音）の SDR 改善量の平均が一番大きい K（本実験の試行は使わない）
  2. 初期値の種だけを変えたときのばらつき
  3. 試行ごとの差 D（2つの T60 の平均）の分散を、2つの成分に分ける（変量効果のモデル）
       D_is = μ + b_i + e_is   （i：試行、s：初期値の種。b_i の分散 σ_b²、e_is の分散 σ_w²）
       試行ごとに S 個の種で平均した差の分散は σ_D²(S) = σ_b² + σ_w²/S
       σ_w² は「同じ試行の中での種の間の分散」の平均、σ_b² は「試行ごとの平均の分散 − σ_w²/S0」（S0＝実験の種の数）
  4. S ごとに、必要な試行の数 n を求める
       H1（片側の対応のある t 検定、効果 sesoi）：非心 t 分布で検出力を計算し、目標を超える最小の n。
       H2・H3（TOST、本当の差 0、幅 Δ）：近似式で n を出し、正規分布の乱数で TOST を実際に行うシミュレーションで確かめる
     ILRMA の回数（＝計算時間）が最も少ない (S, n) を選ぶ
"""
import argparse

import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(0)
CONTRASTS = [("H1 補正の効果 chirp − none > 0", "chirp", "none", "t"),
             ("H2 同等性 chirp − sync", "chirp", "sync", "tost"),
             ("H3 同等性 blind − sync", "blind", "sync", "tost")]


def per_init_diff(q, eps, a, b):
    """初期値の種ごとの、試行ごとの差 D_is（2つの T60 の平均）。表は 試行 × 種"""
    def get(c):
        e = 0.0 if c == "sync" else eps
        return q[(q["cond"] == c) & (q["eps_ppm"] == e)].set_index(["trial", "init", "rt60"])["sdr_imp"]
    x = (get(a) - get(b)).dropna()
    return x.groupby(level=["trial", "init"]).mean().unstack("init")


def power_t(n, delta, sd, alpha):
    df = n - 1
    return 1 - stats.nct.cdf(stats.t.ppf(1 - alpha, df), df, delta / (sd / np.sqrt(n)))


def power_tost_approx(n, margin, sd, alpha):
    p_one = power_t(n, margin, sd, alpha)
    return max(0.0, 1 - 2 * (1 - p_one))


def power_tost_sim(n, margin, sd, alpha, sims=20000):
    x = RNG.normal(0.0, sd, (sims, n))
    m, se = x.mean(1), x.std(1, ddof=1) / np.sqrt(n)
    h = stats.t.ppf(1 - alpha, n - 1) * se
    return np.mean((m - h > -margin) & (m + h < margin))


def need_n(kind, sd, a, alpha):
    for n in range(4, 501):
        pw = power_t(n, a.sesoi, sd, alpha) if kind == "t" else power_tost_approx(n, a.delta, sd, alpha)
        if pw >= a.power:
            return n, pw
    return np.nan, np.nan


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("pilot")
    ap.add_argument("init", nargs="?")
    ap.add_argument("--eps", type=float, default=30.0)
    ap.add_argument("--delta", type=float, default=1.0)
    ap.add_argument("--sesoi", type=float, default=1.0)
    ap.add_argument("--alpha", type=float, default=0.05)
    ap.add_argument("--power", type=float, default=0.8)
    ap.add_argument("--tests", type=int, default=3)
    ap.add_argument("--real-eps", type=float, help="予備のセッションで測った時計のずれ [ppm]（実録音のセッション数 M を出す）")
    a = ap.parse_args()
    d = pd.read_csv(a.pilot)
    print(f"予備実験：{d['trial'].nunique()} 試行、T60 {[f'{r*1000:.0f}ms' for r in sorted(d['rt60'].unique())]}、"
          f"ずれ {[f'{e:.0f}' for e in sorted(d['eps_ppm'].unique()) if e]} ppm、K の候補 {[int(x) for x in sorted(d['k'].unique())]}")

    print("\n[1] K の選び方（予備実験の sync の SDR 改善量の平均 [dB]。本実験の試行は使わない）")
    sync = d[d["cond"] == "sync"].groupby(["k", "rt60"])["sdr_imp"].mean().unstack()
    sync.columns = [f"T60 {c*1000:.0f}ms" for c in sync.columns]
    print(sync.round(2).to_string())
    k = int(d[d["cond"] == "sync"].groupby("k")["sdr_imp"].mean().idxmax())
    print(f"  → K = {k}（初期値の実験 sim_init.csv も、この K で行う）")
    if not a.init:
        print("  sim_init.csv を作ったら、もう一度 sim_init.csv をつけて実行する")
        return
    q = pd.read_csv(a.init)
    if int(q["k"].iloc[0]) != k:
        print(f"  注意：sim_init.csv の K（{int(q['k'].iloc[0])}）が選んだ K と違う。選んだ K でやり直す")

    S0 = q["init"].nunique()
    g = q.groupby(["trial", "rt60", "cond"])["sdr_imp"]
    print(f"\n[2] 初期値の種だけを {S0} 通り変えたときのばらつき（{q['trial'].nunique()} 試行、{len(g)} 組）")
    print(f"  同じ試行・同じ条件の中の標準偏差：{np.sqrt(g.var(ddof=1).mean()):.2f} dB、最大 − 最小の平均 {(g.max() - g.min()).mean():.2f} dB")
    by = q.groupby(["cond", "rt60"]).apply(lambda x: np.sqrt(x.groupby("trial")["sdr_imp"].var(ddof=1).mean()))
    print("  条件・T60 ごと：" + "、".join(f"{c} {r*1000:.0f}ms {v:.2f}" for (c, r), v in by.items()))

    alpha_plan = a.alpha / a.tests
    print(f"\n[3] 差 D の分散の成分（{a.eps:.0f} ppm、2つの T60 の平均）。σ_D(S)² = σ_b² + σ_w²/S")
    comp = {}
    for name, c1, c2, kind in CONTRASTS:
        t = per_init_diff(q, a.eps, c1, c2)
        sw2 = float(t.var(axis=1, ddof=1).mean())
        sb2 = max(float(t.mean(axis=1).var(ddof=1)) - sw2 / S0, 0.0)
        comp[name] = (sb2, sw2, kind, float(t.values.mean()))
        print(f"  {name}：平均 {t.values.mean():+.2f} dB、σ_b = {np.sqrt(sb2):.2f}、σ_w = {np.sqrt(sw2):.2f} dB"
              f"（種1つなら σ_D = {np.sqrt(sb2 + sw2):.2f}）")

    dk = d[d["k"] == k]
    runs_trial = dk.groupby("trial").size().mean()                     # 種1つのときの1試行の ILRMA の回数
    sec_run = dk["sec"].median()                                       # 中央値（スリープなどの外れ値に引っぱられないように）
    print(f"\n[4] 種の数 S と必要な試行の数 n（α = {a.alpha}/{a.tests} = {alpha_plan:.4f}、検出力 {a.power}、"
          f"H1 の効果 {a.sesoi} dB、同等性の幅 ±{a.delta} dB）")
    print(f"  1試行の ILRMA は種1つで {runs_trial:.0f} 回、1回 {sec_run:.1f} 秒（予備実験の中央値）")
    best = None
    for S in (1, 2, 3, 5, 10):
        ns = []
        txt = []
        for name, (sb2, sw2, kind, m) in comp.items():
            sd = np.sqrt(sb2 + sw2 / S)
            n, pw = need_n(kind, sd, a, alpha_plan)
            ns.append(n)
            txt.append(f"{name[:2]} σ_D {sd:.2f}→n {n}")
        if any(np.isnan(ns)):
            print(f"  S = {S:2d}：" + "、".join(txt) + " → 必要な n が 500 を超える（Δ を見直すか、ばらつきを減らす工夫が要る）")
            continue
        n_all = int(np.ceil(max(ns + [12]) / 3) * 3)                    # 音源の組（3組）を同じ数ずつ。最低12
        hours = n_all * runs_trial * S * sec_run / 3600
        print(f"  S = {S:2d}：" + "、".join(txt) + f" → n = {n_all}、ILRMA {n_all*runs_trial*S:.0f} 回、約 {hours:.1f} 時間")
        if best is None or hours < best[2]:
            best = (S, n_all, hours)
    if best is None:
        print("  どの S でも n が 500 を超えた。計画を見直す")
        return
    S, n_all, hours = best
    print(f"  → 計算時間が最も短いのは S = {S}、n = {n_all}（約 {hours:.1f} 時間。ターミナルを3つに分けて --first で並べると約1/3）")

    print(f"\n[5] 確かめ：S = {S}、n = {n_all} の検出力（式 と シミュレーション）")
    for name, (sb2, sw2, kind, m) in comp.items():
        sd = np.sqrt(sb2 + sw2 / S)
        if kind == "t":
            f = power_t(n_all, a.sesoi, sd, alpha_plan)
            sim = np.mean([stats.ttest_1samp(RNG.normal(a.sesoi, sd, n_all), 0, alternative="greater").pvalue < alpha_plan
                           for _ in range(4000)])
        else:
            f = power_tost_approx(n_all, a.delta, sd, alpha_plan)
            sim = power_tost_sim(n_all, a.delta, sd, alpha_plan)
        print(f"  {name}：式 {f:.3f}、シミュレーション {sim:.3f}")
        if kind == "tost" and abs(m) > a.delta / 2:
            print(f"    注意：予備の差の平均 {m:+.2f} dB が幅の半分を超えている。本当に差があれば、同等と言える確率はこれより低い")

    if a.real_eps is not None:
        # 実録音（H4）のセッションの数 M：シミュレーションの T60 256ms の「試行の間の chirp − none」の平均と標準偏差を、
        # 測ったずれに合わせて使う（ずれ 0 で効果 0、ずれの点の間を直線で補間。標準偏差は近いずれの値）
        g = dk[np.isclose(dk["rt60"], 0.256)].pivot_table(index=["trial", "eps_ppm"], columns="cond", values="sdr_imp")
        dd = (g["chirp"] - g["none"]).dropna().groupby(level="eps_ppm")   # sync だけの組（eps 0）は差が NaN なので除く
        mu, sd = dd.mean(), dd.std(ddof=1)
        e = abs(a.real_eps)
        eff = float(np.interp(e, np.r_[0.0, mu.index.values], np.r_[0.0, mu.values]))
        s_near = float(sd.iloc[int(np.argmin(np.abs(sd.index.values - max(e, sd.index.min()))))])
        print(f"\n[6] 実録音のセッションの数 M（測ったずれ {e:.1f} ppm、T60 256ms のシミュレーションから：効果 {eff:+.2f} dB、標準偏差 {s_near:.2f} dB）")
        if eff < 1.0:
            print("  予想される効果が 1dB 未満。H4 は検定せず、差と区間の記述にとどめる（事前登録どおり）")
        else:
            for M in range(3, 101):
                pw = power_t(M, eff, s_near, a.alpha)
                if pw >= a.power:
                    break
            print(f"  片側 α = {a.alpha}、検出力 {a.power}：M = {max(M, 6)}（計算では {M}、最低6。検出力 {pw:.3f}）")
            print("  注意：シミュレーションの部屋は家の部屋より分離しやすいことが多いので、予備のセッションの区間の差も見て、多めにとる")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
K_SEL = int(num(r"→ K = (\d+)（初期値の実験"))
print(f"K の候補 2・4 から K = {K_SEL} が選ばれた（教科書の見本は K = 4。試しの設定では違うことがある）")

選んだ K で、初期値の種だけを変えた実験 `sim_init.csv` を作ります。下のセルの `--k` は、上で選ばれた K を自動で入れます（教科書では、表示された K を自分で書きます）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
# 本番（教科書の段階2 の 3。約15分）：--trials 6 --init-seeds 4
# K_SEL は、上の 1 回目の fink_power.py で選ばれた K
fresh('sim_init.csv')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ["sim_async.py", "--trials", "2", "--seed", "1100", "--k", str(K_SEL), "--eps", "30", "--conds", "sync", "none", "chirp", "blind",
            "--init-seeds", "2", "--out", "sim_init.csv"]   # ノートブック版：ターミナルでの引数の代わり（試しの設定）
# 教科書：uv run python sim_async.py --trials 6 --seed 1100 --k 4 --eps 30 --conds sync none chirp blind --init-seeds 4 --out sim_init.csv
#!/usr/bin/env python3
"""最終演習K：2台の録音機による「非同期の2チャネル録音」をシミュレーションで作り、ずれの補正の効果を ILRMA で調べる

使い方: cd ~/home-lab/final/k
        uv run python sim_async.py --trials 6 --seed 1000 --k 2 4 --out sim_pilot.csv                 （予備実験）
        uv run python sim_async.py --trials 150 --seed 2000 --k 4 --out sim_main.csv                  （本実験。値は計画書のもの）
  --sources : 音源の wav、または話者ごとのフォルダ（中の wav・flac をつなぐ。JVS・LibriTTS の1人分など）。3つ以上。
              組み合わせを試行ごとに順に使う。省略すると librosa の例 libri1・libri2・trumpet（短い音源はくり返してつなぐ）
  --eps     : 機器2 のサンプリング周波数のずれ [ppm]（いくつでも）
  --rt60    : 残響時間 [s]（いくつでも。同じ部屋の形・置き方で吸音だけ変える）
  --k       : ILRMA の基底の数（いくつでも。予備実験で選び、本実験では1つに固定する）
  --seed    : 乱数の種のはじめ（試行 i は seed + i）。予備実験と本実験で重ならないようにする
  --gen fft / --drift 0.5 : 副次の確かめ。機器2 の録音を「直すのとは別の方法」で作る（作りと直しが同じだと、同等になるのが
              当たり前になる。inverse crime）、または時計のずれが時間とともに変わる録音にする
  --first   : 最初の試行の番号（例：--first 12 --trials 12 で後半だけ。分けて実行し、あとで CSV をつなぐ）

1回の試行（部屋の形・機器と音源の置き方・音源の組と区間）の中で、次の条件をすべて同じ録音から作る（対応のある比較）
  sync   : 同期した理想の録音機（機器2 の時計が機器1 と同じ。ε = 0）  … シミュレーションでしか作れない
  none   : 開始のずれだけ合図で合わせ、サンプリング周波数のずれは直さない
  chirp  : 合図（チャープ）で eps と開始のずれを推定し、機器2 を読み直す
  blind  : eps だけを、合図を使わずに none の観測（開始のずれは合図で合わせたもの）から推定し、
           その eps で並べ直して読み直す（合図は開始の位置だけに使う）
  oracle : eps だけ真の値で読み直す（chirp との差＝eps の推定の誤差の影響）
  機器2 の録音は、どの条件も同じ経路（大きさ ±6dB・帯域制限の読み直し・同じ雑音）で作り、開始は合図の始めの位置で合わせる。
  そのため sync と oracle の差は「ずれのある時計で録って、読み直したこと」だけによる。
結果は1行＝試行×T60×ずれ×条件×K×初期値の種。sdr などは2つの音源の平均。所要時間の目安は K-2 に書いた
"""
import argparse
import itertools
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import librosa
import pyroomacoustics as pra
import fast_bss_eval

sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
import bss                                                   # 第7章の自前の ILRMA（~/home-lab/ch7/bss.py）
import clockfix as cf

try:                                                         # IV-5 の共通ライブラリ（--gen fft のときだけ使う）
    sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
    from common import array as arr
except ImportError:
    arr = None

FS = 16000
NFFT, HOP = 2048, 512                                        # 第7章と同じ（16kHz で 128ms の窓）
SNR_DB = 35.0                                                # 録音機の雑音（静かな部屋の目安）


def load_sources(paths, fs):
    if not paths:
        paths = [librosa.ex(n) for n in ("libri1", "libri2", "trumpet")]
    out = []
    for p in paths:
        p = pathlib.Path(p)
        if p.is_dir():                                         # フォルダなら、中の wav・flac を名前の順につなぐ（1人の話者＝1つの音源）
            files = sorted([*p.rglob("*.wav"), *p.rglob("*.flac")])[:60]
            x = np.concatenate([librosa.effects.trim(librosa.load(f, sr=fs, mono=True)[0], top_db=40)[0] for f in files])
        else:
            x, _ = librosa.load(p, sr=fs, mono=True)
        x = x / (np.std(x) + 1e-9)
        out.append(x)
    return out


def excerpt(x, n, rng):
    """長さ n を切り出す（短い音源はくり返してつなぐ。つなぎ目は 20ms で重ねる）"""
    while len(x) < n + 1:
        r = int(0.02 * FS)
        fade = np.linspace(0, 1, r)
        x = np.r_[x[:-r], x[-r:] * (1 - fade) + x[:r] * fade, x[r:]]
    a = rng.integers(0, len(x) - n)
    return x[a: a + n]


def place(rng):
    """家の部屋（6〜8畳くらい）と、机の上の2台の録音機、2つの音源の位置"""
    dim = np.array([rng.uniform(3.2, 4.5), rng.uniform(2.6, 3.6), 2.4])
    while True:
        c = np.array([rng.uniform(1.2, dim[0] - 1.2), rng.uniform(1.2, dim[1] - 1.2), 0.8])
        d = rng.uniform(0.3, 1.0)                              # 2台の間隔 30〜100cm
        th = rng.uniform(0, np.pi)
        u = np.array([np.cos(th), np.sin(th), 0])
        mics = np.c_[c - d / 2 * u, c + d / 2 * u]
        ang = rng.uniform(0, 2 * np.pi, 2)
        diff = abs((ang[0] - ang[1] + np.pi) % (2 * np.pi) - np.pi)
        if diff < np.radians(40):
            continue
        srcs = []
        for a in ang:
            r = rng.uniform(0.8, 1.6)
            srcs.append(c + np.array([r * np.cos(a), r * np.sin(a), rng.uniform(0.3, 0.5)]))
        if not all(np.all(s[:2] > 0.4) and np.all(s[:2] < dim[:2] - 0.4) for s in srcs):
            continue
        # 2本のマイクで見分けられる置き方か：2つの音源の到達時間差（マイク2 − マイク1）が 3 サンプル以上違うこと。
        # マイクを結ぶ線をはさんで鏡に映した位置や、2台の真ん中の垂直な面の近くに2つとも置くと、到達時間差が同じになり分けられない
        tdoa = [(np.linalg.norm(s - mics[:, 1]) - np.linalg.norm(s - mics[:, 0])) / 343.0 * FS for s in srcs]
        if abs(tdoa[0] - tdoa[1]) >= 3.0:
            return dim, mics, srcs, d


def simulate(dim, mics, srcs, rt60, session):
    """部屋の中で、2つの再生ファイルを順番に鳴らしたときの、2本のマイクの連続した信号（同期・理想）"""
    if rt60 == 0:                                              # 無響（直接音だけ）
        room = pra.ShoeBox(dim, fs=FS, max_order=0)
    else:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    room.add_microphone_array(mics)
    for k in range(2):
        room.add_source(srcs[k], signal=session[k])
    room.simulate()
    return room.mic_array.signals                               # (2, 時間)


def evaluate(ref, est):
    sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return float(np.mean(sdr)), float(np.mean(sir)), float(np.mean(sar))


def stft(x):
    return np.stack([librosa.stft(c, n_fft=NFFT, hop_length=HOP) for c in x], axis=-1)     # (F, T, M)


def istft(Y, n):
    return np.stack([librosa.istft(Y[:, :, k], hop_length=HOP, length=n) for k in range(Y.shape[2])])


def separate(mix, k, n_iter, seed):
    return istft(bss.ilrma(stft(mix), k, n_iter, seed=seed)[0], mix.shape[1])


def oracle_w(mix, ref):
    """正解を使った「時間に依らない分離行列」の上限：周波数ごとに W_f = R_sx R_xx⁻¹（最小二乗）

    ILRMA と同じ「周波数ごとに1つの W_f」というモデルで、正解を知っていれば到達できる成績。
    これが下がるなら、ずれのせいで「モデルそのもの」が合わなくなっている。これが高いのに ILRMA だけ下がるなら「推定」のせい
    """
    X, S = stft(mix), stft(ref)
    Rxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Rsx = np.einsum("ftn,ftm->fnm", S, X.conj())
    W = Rsx @ np.linalg.inv(Rxx + 1e-9 * np.trace(Rxx, axis1=1, axis2=2)[:, None, None] * np.eye(2))
    return istft(np.einsum("fnm,ftm->ftn", W, X), mix.shape[1])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sources", nargs="*")
    ap.add_argument("--trials", type=int, default=2)
    ap.add_argument("--first", type=int, default=0, help="最初の試行の番号（何回かに分けて実行するとき）")
    ap.add_argument("--seed", type=int, default=1000)
    ap.add_argument("--eps", type=float, nargs="+", default=[10, 30, 100])
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.0, 0.256])
    ap.add_argument("--init-seeds", type=int, default=1, help="ILRMA の初期値の乱数の種をいくつ試すか")
    ap.add_argument("--k", type=int, nargs="+", default=[2])
    ap.add_argument("--dur", type=float, default=8.0)
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--conds", nargs="+", default=["sync", "none", "chirp", "blind", "oracle"])
    ap.add_argument("--gen", choices=["sinc", "fft"], default="sinc",
                    help="機器2 の録音の作り方（副次の確かめ）。fft は IV-5 の common/array.py の sfo_resample（直すほうとは別の方法）")
    ap.add_argument("--drift", type=float, default=0.0,
                    help="機器2 の時計のずれが時間とともに変わる速さ [ppm/分]（副次の確かめ。ε(t) = ε + drift·t）")
    ap.add_argument("--out", default="sim_pilot.csv")
    a = ap.parse_args()
    if pathlib.Path(a.out).exists():
        sys.exit(f"{a.out} はすでにあります（上書きしないように止めました）")
    srcs = load_sources(a.sources, FS)
    pairs = list(itertools.combinations(range(len(srcs)), 2))
    L = int(a.dur * FS)
    chirp = cf.make_chirp(FS, cf.CHIRP_SEC)
    rows, t_all = [], time.time()
    for i in range(a.first, a.first + a.trials):
        seed = a.seed + i
        rng = np.random.default_rng(seed)
        pair = pairs[i % len(pairs)]
        dim, mics, spos, d_mic = place(rng)
        # 再生ファイル2つ（音源ごとに区間を乱数で選ぶ）と、それを順番に鳴らすセッション
        plays, lays = [], []
        for k in range(2):
            w, lay = cf.build_play(excerpt(srcs[pair[k]], L, rng), FS, 1, a.dur)
            plays.append(w)
            lays.append(lay)
        spacing = lays[0]["chirp_end"] - lays[0]["chirp_start"]   # 1つのファイルの中の合図の間隔（サンプル）
        pad, gap = int(1.5 * FS), int(1.0 * FS)
        total = pad + len(plays[0]) + gap + len(plays[1]) + pad
        session = np.zeros((2, total))
        session[0, pad: pad + len(plays[0])] = plays[0]
        session[1, pad + len(plays[0]) + gap: pad + len(plays[0]) + gap + len(plays[1])] = plays[1]
        s0 = rng.uniform(0.2, 1.0) * FS                          # 機器2 が録音を始めた位置（機器1 のサンプル番号）
        gain2 = 10 ** (rng.uniform(-6, 6) / 20)                  # 機器2 の録音の大きさ（機器ごとに違う）
        for rt60 in a.rt60:
            mic = simulate(dim, mics, spos, rt60, session)[:, :total]
            sig_rms = np.sqrt(np.mean(mic[:, pad: -pad] ** 2))
            # 雑音は条件・ずれによらず同じものを使う（試行と T60 ごとに別の乱数の種から作る）
            nrng = np.random.default_rng([seed, int(rt60 * 1000), 7])
            n2max = int((total - s0 - FS) * (1 + max(list(a.eps) + [0.0]) * 1e-6)) + 1
            noise1 = nrng.standard_normal(total) * sig_rms * 10 ** (-SNR_DB / 20)
            noise2 = nrng.standard_normal(n2max) * gain2 * sig_rms * 10 ** (-SNR_DB / 20)
            rec1 = mic[0] + noise1
            c1, q1 = cf.find_chirps(rec1, chirp, spacing)
            for eps_ppm in [0.0] + list(a.eps):
                eps = eps_ppm * 1e-6
                # 機器2 の録音：sync（ε = 0 の理想の録音機）も、ずれのある録音機も、同じ経路（大きさ・読み直し・雑音）で作る
                n2 = int((total - s0 - FS) * (1 + eps))
                if a.gen == "fft":                                     # 作るほうを別の方法（FFT の読み直し）にする
                    s0i = int(round(s0))
                    y2, _ = arr.sfo_resample(mic[1][s0i:], eps)
                    rec2 = gain2 * y2[:n2] + noise2[:n2]
                elif a.drift and eps_ppm != 0:                        # ε(t) = ε + β t（t は機器1 の秒）。sync（ε=0）は同期のまま
                    beta = a.drift * 1e-6 / 60
                    nn = np.arange(n2)
                    rec2 = gain2 * cf.frac_read(mic[1], s0 + nn / (1 + eps) - beta * nn ** 2 / (2 * FS)) + noise2[:n2]
                else:
                    rec2 = gain2 * cf.frac_resample(mic[1], s0, 1 / (1 + eps), n2) + noise2[:n2]
                c2, q2 = cf.find_chirps(rec2, chirp, spacing)
                clk = cf.fit_clock(c1, c2, [(0, 1), (2, 3)])
                # 開始の合わせ方はどの条件も同じ（合図の始めの位置。P1 から2台までの到達時間の差が一定の遅れとして入る）
                start = lambda e: float(c2[0] - (1 + e) * c1[0])
                conds = {}
                if eps_ppm == 0.0:
                    if "sync" in a.conds:
                        conds["sync"] = (0.0, start(0.0), "resample", np.nan)
                else:
                    if "none" in a.conds or "blind" in a.conds:
                        conds["none"] = (clk["eps"], clk["alpha"], "none", clk["eps"])
                    if "chirp" in a.conds:
                        conds["chirp"] = (clk["eps"], clk["alpha"], "resample", clk["eps"])
                    if "oracle" in a.conds:
                        conds["oracle"] = (eps, start(eps), "resample", eps)    # ε だけ真の値（開始は合図）
                for name, (e, al, mode, e_hat) in conds.items():
                    mix, ref = cf.build_mixture(rec1, rec2, c1, lays, 0, L, e, al, mode)
                    todo = [(name, mix, e_hat)]
                    if name == "none" and "blind" in a.conds:
                        eb = cf.drift_search(mix[0], mix[1], FS)[0]           # none の観測から ε だけを推定
                        mb, _ = cf.build_mixture(rec1, rec2, c1, lays, 0, L, eb, start(eb), "resample")
                        todo.append(("blind", mb, eb))
                        if "none" not in a.conds:
                            todo = todo[1:]
                    with np.errstate(divide="ignore"):
                        sdr_in = evaluate(ref, np.stack([mix[0], mix[0]]))[0]
                    for cname, m, eh in todo:
                        orw = evaluate(ref, oracle_w(m, ref))[0] - sdr_in
                        for k in a.k:
                            for s_init in range(a.init_seeds):
                                t0 = time.time()
                                y = separate(m, k, a.iter, seed + 100000 * s_init)
                                sdr, sir, sar = evaluate(ref, y)
                                rows.append(dict(trial=i, seed=seed, init=s_init, gen=a.gen, drift=a.drift if eps_ppm else 0.0,
                                                 pair=f"{pair[0]}-{pair[1]}",
                                                 d_mic=round(d_mic, 3), rt60=rt60, eps_ppm=eps_ppm, cond=cname, k=k,
                                                 eps_hat_ppm=eh * 1e6, sdr_in=sdr_in, sdr=sdr, sir=sir, sar=sar,
                                                 sdr_imp=sdr - sdr_in, orw_imp=orw,
                                                 chirp_q=float(min(q1.min(), q2.min())),
                                                 sec=time.time() - t0))
                                print(f"試行 {i:2d} T60 {rt60*1000:3.0f}ms ずれ {eps_ppm:5.0f}ppm {cname:6s} K={k} 初期値{s_init}"
                                      f"  推定 {eh*1e6:8.2f}ppm  SDR改善 {sdr - sdr_in:6.2f} dB（正解の W なら {orw:6.2f}）"
                                      f"  {time.time()-t0:4.1f}秒", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)       # 途中でも保存する
    print(f"{a.out} に保存しました（{len(rows)} 行、{(time.time()-t_all)/60:.1f} 分）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
import pandas as pd
q = pd.read_csv("sim_init.csv")
nt, s0 = q["trial"].nunique(), q["init"].nunique()
ok(len(q) == nt * 2 * 4 * s0 and set(q["k"]) == {K_SEL}, f"sim_init.csv は {nt} 試行 × 2 T60 × 4 条件 × 種 {s0} つ ＝ {len(q)} 行（K = {K_SEL}）")
sd = q.groupby(["trial", "rt60", "cond"])["sdr_imp"].std(ddof=1)
ok(sd.mean() > 0.1, f"初期値の種だけを変えても SDR 改善量が変わる（同じ試行・条件の中の標準偏差の平均 {sd.mean():.2f} dB。教科書の見本は約 3.2 dB）")

2 回目の `fink_power.py`：初期値の種の数 S と試行の数 n を決めます（Δ = 1.0 dB、見逃したくない最小の効果 1.0 dB）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['fink_power.py', 'sim_pilot.csv', 'sim_init.csv', '--eps', '30', '--delta', '1.0', '--sesoi', '1.0']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python fink_power.py sim_pilot.csv sim_init.csv --eps 30 --delta 1.0 --sesoi 1.0
#!/usr/bin/env python3
"""最終演習K：予備実験の結果から、K を選び、初期値の種の数 S と試行の数 n を決める（検出力の計算）

使い方: cd ~/home-lab/final/k
  uv run python fink_power.py sim_pilot.csv                       （1回目：K を選ぶだけ）
  uv run python fink_power.py sim_pilot.csv sim_init.csv --eps 30 --delta 1.0 --sesoi 1.0   （2回目：S と n を決める）
  sim_pilot.csv : 予備実験（K の候補を全部、初期値の種は1つ）
  sim_init.csv  : 初期値の種だけを変えた実験（1回目で選んだ K、30 ppm、sync・none・chirp・blind、種 4 つ）
  --delta 1.0   : 同等性の幅 Δ [dB]（事前登録の値。計画書に理由を書く）
  --sesoi 1.0   : 「補正の効果」として見逃したくない最小の大きさ [dB]
  --alpha 0.05 --power 0.8 --tests 3 : 有意水準・検出力・確証的な検定の数（ホルム法の最悪の場合 α/3 で計画する）

やること
  1. K の選び方：予備実験の sync（同期した理想の録音）の SDR 改善量の平均が一番大きい K（本実験の試行は使わない）
  2. 初期値の種だけを変えたときのばらつき
  3. 試行ごとの差 D（2つの T60 の平均）の分散を、2つの成分に分ける（変量効果のモデル）
       D_is = μ + b_i + e_is   （i：試行、s：初期値の種。b_i の分散 σ_b²、e_is の分散 σ_w²）
       試行ごとに S 個の種で平均した差の分散は σ_D²(S) = σ_b² + σ_w²/S
       σ_w² は「同じ試行の中での種の間の分散」の平均、σ_b² は「試行ごとの平均の分散 − σ_w²/S0」（S0＝実験の種の数）
  4. S ごとに、必要な試行の数 n を求める
       H1（片側の対応のある t 検定、効果 sesoi）：非心 t 分布で検出力を計算し、目標を超える最小の n。
       H2・H3（TOST、本当の差 0、幅 Δ）：近似式で n を出し、正規分布の乱数で TOST を実際に行うシミュレーションで確かめる
     ILRMA の回数（＝計算時間）が最も少ない (S, n) を選ぶ
"""
import argparse

import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(0)
CONTRASTS = [("H1 補正の効果 chirp − none > 0", "chirp", "none", "t"),
             ("H2 同等性 chirp − sync", "chirp", "sync", "tost"),
             ("H3 同等性 blind − sync", "blind", "sync", "tost")]


def per_init_diff(q, eps, a, b):
    """初期値の種ごとの、試行ごとの差 D_is（2つの T60 の平均）。表は 試行 × 種"""
    def get(c):
        e = 0.0 if c == "sync" else eps
        return q[(q["cond"] == c) & (q["eps_ppm"] == e)].set_index(["trial", "init", "rt60"])["sdr_imp"]
    x = (get(a) - get(b)).dropna()
    return x.groupby(level=["trial", "init"]).mean().unstack("init")


def power_t(n, delta, sd, alpha):
    df = n - 1
    return 1 - stats.nct.cdf(stats.t.ppf(1 - alpha, df), df, delta / (sd / np.sqrt(n)))


def power_tost_approx(n, margin, sd, alpha):
    p_one = power_t(n, margin, sd, alpha)
    return max(0.0, 1 - 2 * (1 - p_one))


def power_tost_sim(n, margin, sd, alpha, sims=20000):
    x = RNG.normal(0.0, sd, (sims, n))
    m, se = x.mean(1), x.std(1, ddof=1) / np.sqrt(n)
    h = stats.t.ppf(1 - alpha, n - 1) * se
    return np.mean((m - h > -margin) & (m + h < margin))


def need_n(kind, sd, a, alpha):
    for n in range(4, 501):
        pw = power_t(n, a.sesoi, sd, alpha) if kind == "t" else power_tost_approx(n, a.delta, sd, alpha)
        if pw >= a.power:
            return n, pw
    return np.nan, np.nan


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("pilot")
    ap.add_argument("init", nargs="?")
    ap.add_argument("--eps", type=float, default=30.0)
    ap.add_argument("--delta", type=float, default=1.0)
    ap.add_argument("--sesoi", type=float, default=1.0)
    ap.add_argument("--alpha", type=float, default=0.05)
    ap.add_argument("--power", type=float, default=0.8)
    ap.add_argument("--tests", type=int, default=3)
    ap.add_argument("--real-eps", type=float, help="予備のセッションで測った時計のずれ [ppm]（実録音のセッション数 M を出す）")
    a = ap.parse_args()
    d = pd.read_csv(a.pilot)
    print(f"予備実験：{d['trial'].nunique()} 試行、T60 {[f'{r*1000:.0f}ms' for r in sorted(d['rt60'].unique())]}、"
          f"ずれ {[f'{e:.0f}' for e in sorted(d['eps_ppm'].unique()) if e]} ppm、K の候補 {[int(x) for x in sorted(d['k'].unique())]}")

    print("\n[1] K の選び方（予備実験の sync の SDR 改善量の平均 [dB]。本実験の試行は使わない）")
    sync = d[d["cond"] == "sync"].groupby(["k", "rt60"])["sdr_imp"].mean().unstack()
    sync.columns = [f"T60 {c*1000:.0f}ms" for c in sync.columns]
    print(sync.round(2).to_string())
    k = int(d[d["cond"] == "sync"].groupby("k")["sdr_imp"].mean().idxmax())
    print(f"  → K = {k}（初期値の実験 sim_init.csv も、この K で行う）")
    if not a.init:
        print("  sim_init.csv を作ったら、もう一度 sim_init.csv をつけて実行する")
        return
    q = pd.read_csv(a.init)
    if int(q["k"].iloc[0]) != k:
        print(f"  注意：sim_init.csv の K（{int(q['k'].iloc[0])}）が選んだ K と違う。選んだ K でやり直す")

    S0 = q["init"].nunique()
    g = q.groupby(["trial", "rt60", "cond"])["sdr_imp"]
    print(f"\n[2] 初期値の種だけを {S0} 通り変えたときのばらつき（{q['trial'].nunique()} 試行、{len(g)} 組）")
    print(f"  同じ試行・同じ条件の中の標準偏差：{np.sqrt(g.var(ddof=1).mean()):.2f} dB、最大 − 最小の平均 {(g.max() - g.min()).mean():.2f} dB")
    by = q.groupby(["cond", "rt60"]).apply(lambda x: np.sqrt(x.groupby("trial")["sdr_imp"].var(ddof=1).mean()))
    print("  条件・T60 ごと：" + "、".join(f"{c} {r*1000:.0f}ms {v:.2f}" for (c, r), v in by.items()))

    alpha_plan = a.alpha / a.tests
    print(f"\n[3] 差 D の分散の成分（{a.eps:.0f} ppm、2つの T60 の平均）。σ_D(S)² = σ_b² + σ_w²/S")
    comp = {}
    for name, c1, c2, kind in CONTRASTS:
        t = per_init_diff(q, a.eps, c1, c2)
        sw2 = float(t.var(axis=1, ddof=1).mean())
        sb2 = max(float(t.mean(axis=1).var(ddof=1)) - sw2 / S0, 0.0)
        comp[name] = (sb2, sw2, kind, float(t.values.mean()))
        print(f"  {name}：平均 {t.values.mean():+.2f} dB、σ_b = {np.sqrt(sb2):.2f}、σ_w = {np.sqrt(sw2):.2f} dB"
              f"（種1つなら σ_D = {np.sqrt(sb2 + sw2):.2f}）")

    dk = d[d["k"] == k]
    runs_trial = dk.groupby("trial").size().mean()                     # 種1つのときの1試行の ILRMA の回数
    sec_run = dk["sec"].median()                                       # 中央値（スリープなどの外れ値に引っぱられないように）
    print(f"\n[4] 種の数 S と必要な試行の数 n（α = {a.alpha}/{a.tests} = {alpha_plan:.4f}、検出力 {a.power}、"
          f"H1 の効果 {a.sesoi} dB、同等性の幅 ±{a.delta} dB）")
    print(f"  1試行の ILRMA は種1つで {runs_trial:.0f} 回、1回 {sec_run:.1f} 秒（予備実験の中央値）")
    best = None
    for S in (1, 2, 3, 5, 10):
        ns = []
        txt = []
        for name, (sb2, sw2, kind, m) in comp.items():
            sd = np.sqrt(sb2 + sw2 / S)
            n, pw = need_n(kind, sd, a, alpha_plan)
            ns.append(n)
            txt.append(f"{name[:2]} σ_D {sd:.2f}→n {n}")
        if any(np.isnan(ns)):
            print(f"  S = {S:2d}：" + "、".join(txt) + " → 必要な n が 500 を超える（Δ を見直すか、ばらつきを減らす工夫が要る）")
            continue
        n_all = int(np.ceil(max(ns + [12]) / 3) * 3)                    # 音源の組（3組）を同じ数ずつ。最低12
        hours = n_all * runs_trial * S * sec_run / 3600
        print(f"  S = {S:2d}：" + "、".join(txt) + f" → n = {n_all}、ILRMA {n_all*runs_trial*S:.0f} 回、約 {hours:.1f} 時間")
        if best is None or hours < best[2]:
            best = (S, n_all, hours)
    if best is None:
        print("  どの S でも n が 500 を超えた。計画を見直す")
        return
    S, n_all, hours = best
    print(f"  → 計算時間が最も短いのは S = {S}、n = {n_all}（約 {hours:.1f} 時間。ターミナルを3つに分けて --first で並べると約1/3）")

    print(f"\n[5] 確かめ：S = {S}、n = {n_all} の検出力（式 と シミュレーション）")
    for name, (sb2, sw2, kind, m) in comp.items():
        sd = np.sqrt(sb2 + sw2 / S)
        if kind == "t":
            f = power_t(n_all, a.sesoi, sd, alpha_plan)
            sim = np.mean([stats.ttest_1samp(RNG.normal(a.sesoi, sd, n_all), 0, alternative="greater").pvalue < alpha_plan
                           for _ in range(4000)])
        else:
            f = power_tost_approx(n_all, a.delta, sd, alpha_plan)
            sim = power_tost_sim(n_all, a.delta, sd, alpha_plan)
        print(f"  {name}：式 {f:.3f}、シミュレーション {sim:.3f}")
        if kind == "tost" and abs(m) > a.delta / 2:
            print(f"    注意：予備の差の平均 {m:+.2f} dB が幅の半分を超えている。本当に差があれば、同等と言える確率はこれより低い")

    if a.real_eps is not None:
        # 実録音（H4）のセッションの数 M：シミュレーションの T60 256ms の「試行の間の chirp − none」の平均と標準偏差を、
        # 測ったずれに合わせて使う（ずれ 0 で効果 0、ずれの点の間を直線で補間。標準偏差は近いずれの値）
        g = dk[np.isclose(dk["rt60"], 0.256)].pivot_table(index=["trial", "eps_ppm"], columns="cond", values="sdr_imp")
        dd = (g["chirp"] - g["none"]).dropna().groupby(level="eps_ppm")   # sync だけの組（eps 0）は差が NaN なので除く
        mu, sd = dd.mean(), dd.std(ddof=1)
        e = abs(a.real_eps)
        eff = float(np.interp(e, np.r_[0.0, mu.index.values], np.r_[0.0, mu.values]))
        s_near = float(sd.iloc[int(np.argmin(np.abs(sd.index.values - max(e, sd.index.min()))))])
        print(f"\n[6] 実録音のセッションの数 M（測ったずれ {e:.1f} ppm、T60 256ms のシミュレーションから：効果 {eff:+.2f} dB、標準偏差 {s_near:.2f} dB）")
        if eff < 1.0:
            print("  予想される効果が 1dB 未満。H4 は検定せず、差と区間の記述にとどめる（事前登録どおり）")
        else:
            for M in range(3, 101):
                pw = power_t(M, eff, s_near, a.alpha)
                if pw >= a.power:
                    break
            print(f"  片側 α = {a.alpha}、検出力 {a.power}：M = {max(M, 6)}（計算では {M}、最低6。検出力 {pw:.3f}）")
            print("  注意：シミュレーションの部屋は家の部屋より分離しやすいことが多いので、予備のセッションの区間の差も見て、多めにとる")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
S_SEL, N_SEL = int(num(r"→ 計算時間が最も短いのは S = (\d+)")), int(num(r"→ 計算時間が最も短いのは S = \d+、n = (\d+)"))
ok(1 <= S_SEL <= 10 and N_SEL >= 12, f"S と n が表示された（S = {S_SEL}、n = {N_SEL}。教科書の見本は S = 1、n = 150）")
pw = [(float(a), float(b)) for a, b in re.findall(r"：式 ([\d.]+)、シミュレーション ([\d.]+)", cap.stdout)]
ok(len(pw) == 3 and all(abs(a - b) <= 0.02 for a, b in pw), f"式とシミュレーションの検出力が 0.02 以内で一致 {pw}")

**事前登録**（段階2 の 5）：教科書では、ここで `prereg.md`（K-4-5 のひな形）に K・S・n・本実験の種（2000〜）を書き、予備実験の CSV と解析のコードと一緒に Git にコミットします（`git log -1 --format='%h %ci'` のコミットID と日時を計画書に書く）。ノートブックでは行いません。

## 4. 再生ファイルを作る（K-3-5、段階3 の 1、`make_play.py`）

教科書では、第7章の `record.py` で自分の声（公開されている文章を読む）と楽器を 30 秒ずつ録り（`--sec 30`、最大 −10〜−3 dBFS 程度）、`data/src_voice.wav` などにします。**ノートブックでは録音のかわりに**、librosa の例の音源から 30 秒の音源を 3 つ作ります（次のセル。ノートブックで足したものです）。

In [ ]:
# ノートブック版で足したセル：録音（record.py）のかわりに、librosa の例の音源から 30 秒の音源を作る
import numpy as np, librosa, soundfile as sf
for name, ex in (("src_voice", "libri1"), ("src_read2", "libri2"), ("src_trumpet", "trumpet")):
    x, _ = librosa.load(librosa.ex(ex), sr=16000)
    while len(x) < 31 * 16000:            # 短い音源はくり返してつなぐ
        x = np.r_[x, x]
    x = x[:30 * 16000]
    x = x / np.max(np.abs(x)) * 0.3       # 最大 約 −10 dBFS（録音の目安 −10〜−3 dBFS）
    sf.write(f"data/{name}.wav", x, 16000)
    print(f"data/{name}.wav：{len(x)/16000:.1f} 秒、最大 {20*np.log10(np.max(np.abs(x))):.1f} dBFS")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
fresh('play')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ['make_play.py', 'data/src_voice.wav', 'data/src_read2.wav', 'data/src_trumpet.wav']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python make_play.py data/src_voice.wav data/src_read2.wav data/src_trumpet.wav
#!/usr/bin/env python3
"""最終演習K：実録音の「再生ファイル」を作る（音源ごとに1つ。前後に合図のチャープを入れる）

使い方: cd ~/home-lab/final/k && uv run python make_play.py data/src_voice.wav data/src_guitar.wav data/src_read2.wav
        uv run python make_play.py --live      （3台目の機器がないとき：中身が無音の play/play_live.wav。Mac が afplay で鳴らす）
  音源ごとに play/play_<名前>.wav（48kHz・16ビット・モノラル）と、共通の並びの情報 play/layout.json を作る
  --nseg 3 --seg 8 : 1つのファイルに 8秒の区間を3つ（合わせて24秒）入れる。音源はそれ以上の長さが要る
  --skip 0.5       : 音源の頭の何秒をとばすか（録音の始めの雑音を避ける）

ファイルの中身：[無音0.2s][合図0.3s][無音0.7s][音源 24s][無音0.7s][合図0.3s][無音0.2s] ＝ 約26.4秒
音源の大きさ（実効値）は −26 dBFS にそろえ、合図はそれより約 11dB 大きい。鳴らす機器の音量は控えめにする（K-2 の注意）
"""
import argparse
import json
import pathlib
import sys

import numpy as np
import librosa
import soundfile as sf

import clockfix as cf

FS_PLAY = 48000


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sources", nargs="*")
    ap.add_argument("--live", action="store_true", help="音源を生で出すときの、合図だけの再生ファイルを作る")
    ap.add_argument("--nseg", type=int, default=3)
    ap.add_argument("--seg", type=float, default=8.0)
    ap.add_argument("--skip", type=float, default=0.5)
    ap.add_argument("--outdir", default="play")
    a = ap.parse_args()
    out = pathlib.Path(a.outdir)
    out.mkdir(exist_ok=True)
    lay = None
    if not a.sources and not a.live:
        sys.exit("音源の wav を指定するか、--live をつけてください")
    todo = [(p, None) for p in a.sources] + ([("live", np.zeros(int(a.nseg * a.seg * FS_PLAY)))] if a.live else [])
    for p, x in todo:
        if x is None:
            x, _ = librosa.load(p, sr=FS_PLAY, mono=True, offset=a.skip)
        need = a.nseg * a.seg
        if len(x) < need * FS_PLAY:
            sys.exit(f"{p} は {len(x)/FS_PLAY:.1f} 秒しかありません（{need:.0f} 秒以上必要）")
        wave, lay = cf.build_play(x, FS_PLAY, a.nseg, a.seg)
        name = out / f"play_{pathlib.Path(p).stem}.wav"
        if name.exists():
            sys.exit(f"{name} はすでにあります（上書きしないように止めました）")
        sf.write(name, wave, FS_PLAY, subtype="PCM_16")
        print(f"{name}：{len(wave)/FS_PLAY:.1f} 秒、ピーク {20*np.log10(np.max(np.abs(wave))):.1f} dBFS")
    if (out / "layout.json").exists() and json.load(open(out / "layout.json")) != lay:
        sys.exit(f"{out / 'layout.json'} が今の設定と違います（--nseg・--seg を変えた？）。古い再生ファイルと混ぜない")
    with open(out / "layout.json", "w") as f:
        json.dump(lay, f, indent=1)
    print(f"{out/'layout.json'} に並びの情報を保存しました：{lay}")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
import json
lay = json.load(open("play/layout.json"))
ok(lay["chirp_start"] == 9600 and lay["chirp_end"] == 1243200 and lay["seg_starts"] == [57600, 441600, 825600],
   "layout.json が chirp_start 9600・chirp_end 1243200・seg_starts [57600, 441600, 825600]")
pk = [float(v) for v in re.findall(r"ピーク ([-\d.]+) dBFS", cap.stdout)]
ok(len(pk) == 3 and all(v < -3 for v in pk), f"再生ファイルが 3 つでき、ピークが −3 dBFS より小さい（割れない）{pk}")

再生ファイルを 1 つ聞きます（最初と最後に合図のチャープが鳴ります。音量は小さく）。

In [ ]:
# wav を聞く（パソコンの音量を一番小さくしてから。1つずつ）
for f in ['play/play_src_voice.wav']:
    print(f)
    listen_file(f)

## 5. 合成の録音で、解析の手順を先に確かめる（K-3-4・K-3-6、段階3 の 2）

`synth_session.py --sessions 5 --bad` で、答えの分かった 5 セッション分の「Mac とスマホの録音」（48 kHz・16 ビット）を作り、`real_mix.py` と `fink_real_stats.py` を通します（II-5 の `synth_capture.py` と同じ考え方）。最後の s05 には「自動の音量調整」（1 つ目のファイルの途中で 3 dB 下がる）を入れてあり、除外の基準で除かれるはずです。

**ノートブックの既定は `real_mix.py --inits 1`（ILRMA の初期値の種 1 つ、約 3 分）**です。教科書の設定（`--inits 3`、1 セッション約 2 分、4 セッションで約 8 分）にするときは、セルの `sys.argv` をコメントの行に取りかえてください。種 1 つでは、K-7-1 の (8) の差・区間とは合いません。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
fresh('synth')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ['synth_session.py', '--outdir', 'synth', '--sessions', '5', '--bad']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python synth_session.py --outdir synth --sessions 5 --bad
#!/usr/bin/env python3
"""最終演習K：実録音と同じ形の「合成の録音」を作る（解析の手順を、答えの分かったデータで先に確かめるため）

使い方: cd ~/home-lab/final/k && uv run python synth_session.py --outdir synth
  play/ の再生ファイルと layout.json（make_play.py で作ったもの）を、シミュレーションの部屋で順番に鳴らし、
  Mac とスマホ（時計のずれ・録音開始のずれ・録音の大きさ・雑音つき）の録音を 48kHz・16ビットの wav で書く。
  synth/sessions.csv（real_mix.py にそのまま渡せる）と synth/truth.json（本当のずれ）も作る。
  --eps-ppm 23 : スマホの時計のずれ（セッションごとに ±0.3ppm ゆらす）
  --bad        : 最後のセッションに「自動の音量調整」（1つ目のファイルの途中で大きさが 3dB 下がる）を入れる（除外の確認）
"""
import argparse
import json
import pathlib
import sys

import numpy as np
import pandas as pd
import pyroomacoustics as pra
import soundfile as sf

import clockfix as cf

FS = 48000


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--play", default="play")
    ap.add_argument("--outdir", default="synth")
    ap.add_argument("--sessions", type=int, default=4)
    ap.add_argument("--eps-ppm", type=float, default=23.0)
    ap.add_argument("--rt60", type=float, default=0.35)
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--bad", action="store_true")
    a = ap.parse_args()
    out = pathlib.Path(a.outdir)
    if out.exists():
        sys.exit(f"{out} はすでにあります（上書きしないように止めました）")
    out.mkdir()
    rng = np.random.default_rng(a.seed)
    files = sorted(f for f in pathlib.Path(a.play).glob("play_*.wav") if f.stem != "play_live")   # 無音の合図だけのファイルは除く
    if len(files) < 2:
        sys.exit("play/ に再生ファイルが2つ以上必要です（make_play.py）")
    names = [f.stem[5:] for f in files]
    dim = np.array([3.6, 2.7, 2.4])                               # 6畳くらい
    e_abs, max_order = pra.inverse_sabine(a.rt60, dim)
    rows, truth = [], {}
    for s in range(a.sessions):
        i1, i2 = rng.choice(len(files), 2, replace=False)
        w1, _ = sf.read(files[i1])
        w2, _ = sf.read(files[i2])
        c = np.array([rng.uniform(1.2, 2.4), rng.uniform(1.0, 1.7), 0.75])
        d = rng.uniform(0.3, 0.8)
        mics = np.c_[c - [d / 2, 0, 0], c + [d / 2, 0, 0]]
        p1 = c + [rng.uniform(-1.0, -0.4), rng.uniform(0.6, 0.9), 0.3]
        p2 = c + [rng.uniform(0.4, 1.0), rng.uniform(0.6, 0.9), 0.3]
        t1, gap = 3.0, 4.0                                        # 1つ目を 3秒から、2つ目はその 4秒あと（置き換える時間）
        total = int((t1 + len(w1) / FS + gap + len(w2) / FS + 3.0) * FS)
        sig = np.zeros((2, total))
        a1 = int(t1 * FS)
        a2 = a1 + len(w1) + int(gap * FS)
        sig[0, a1: a1 + len(w1)] = w1
        sig[1, a2: a2 + len(w2)] = w2
        room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
        room.add_microphone_array(mics)
        room.add_source(p1, signal=sig[0])
        room.add_source(p2, signal=sig[1])
        room.simulate()
        mic = room.mic_array.signals[:, :total]
        floor = 10 ** (-60 / 20)                                  # 録音機の雑音 −60 dBFS
        eps = (a.eps_ppm + rng.uniform(-0.3, 0.3)) * 1e-6
        s0 = rng.uniform(0.5, 2.0) * FS                           # スマホが録音を始めた位置（Mac のサンプル番号）
        g2 = 10 ** (rng.uniform(-4, 4) / 20)
        g1 = 10 ** (-30 / 20) / np.sqrt(np.mean(mic[0, a1: a2 + len(w2)] ** 2))   # Mac の録音の実効値を約 −30 dBFS に
        rec1 = g1 * mic[0] + floor * rng.standard_normal(total)
        n2 = int((total - s0 - FS) * (1 + eps))
        rec2 = g1 * g2 * cf.frac_resample(mic[1], s0, 1 / (1 + eps), n2) + floor * rng.standard_normal(n2)
        if a.bad and s == a.sessions - 1:
            k0 = int((t1 + 8.0) * FS - s0)                        # 1つ目のファイルの途中（8秒〜18秒）で
            ramp = np.ones(n2)
            ramp[k0: k0 + 10 * FS] = np.linspace(1, 10 ** (-3 / 20), 10 * FS)
            ramp[k0 + 10 * FS:] = 10 ** (-3 / 20)                 # 3dB 下がる（自動の音量調整のまね）
            rec2 *= ramp
        sid = f"s{s+1:02d}"
        sf.write(out / f"{sid}_mac.wav", rec1, FS, subtype="PCM_16")
        sf.write(out / f"{sid}_phone.wav", rec2, FS, subtype="PCM_16")
        rows.append(dict(session=sid, date="synth", room="sim6jo", src1=names[i1], src2=names[i2],
                         mac_wav=f"{sid}_mac.wav", phone_wav=f"{sid}_phone.wav", d_mics_cm=round(d * 100), notes="合成"))
        truth[sid] = dict(eps_ppm=eps * 1e6, offset_s=-s0 * (1 + eps) / FS, bad=bool(a.bad and s == a.sessions - 1))
        print(f"{sid}：{names[i1]}→{names[i2]}、本当の eps {eps*1e6:+.2f} ppm、開始のずれ {-s0/FS:+.3f} 秒（スマホの方が遅く始めた）"
              f"、ピーク Mac {20*np.log10(np.max(np.abs(rec1))):.1f}・スマホ {20*np.log10(np.max(np.abs(rec2))):.1f} dBFS")
    pd.DataFrame(rows).to_csv(out / "sessions.csv", index=False)
    json.dump(truth, open(out / "truth.json", "w"), indent=1)
    print(f"{out}/ に {a.sessions} セッション分を作りました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
import json, os
truth = json.load(open("synth/truth.json"))
ok(sorted(truth) == ["s01", "s02", "s03", "s04", "s05"] and [k for k, v in truth.items() if v["bad"]] == ["s05"],
   "5 セッション分ができ、自動の音量調整を入れたのは最後の s05 だけ（synth/truth.json）")
ok(all(os.path.exists(f"synth/s0{i}_{w}.wav") for i in range(1, 6) for w in ("mac", "phone")) and os.path.exists("synth/sessions.csv"),
   "synth/ に Mac とスマホの録音（wav）と sessions.csv ができた")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
# 教科書の設定（種 3 つ）：sys.argv = ["real_mix.py", "synth/sessions.csv", "--data", "synth", "--out", "synth_results.csv", "--clock-out", "synth_clock.csv"]
fresh('synth_results.csv', 'synth_clock.csv', 'sep')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ['real_mix.py', 'synth/sessions.csv', '--data', 'synth', '--inits', '1', '--out', 'synth_results.csv', '--clock-out', 'synth_clock.csv']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python real_mix.py synth/sessions.csv --data synth --inits 1 --out synth_results.csv --clock-out synth_clock.csv
#!/usr/bin/env python3
"""最終演習K：Mac とスマホで別々に録った録音から、時計のずれを推定し、2チャネルの観測を作って ILRMA で分離・評価する

使い方: cd ~/home-lab/final/k && uv run python real_mix.py sessions.csv --out real_results.csv
  sessions.csv の列：session,date,room,src1,src2,mac_wav,phone_wav,d_mics_cm,notes
    src1・src2 は鳴らした再生ファイルの名前（play_ を除いたもの。例 src_voice・src_guitar）。1つ目を位置 P1、2つ目を P2 で鳴らす
    mac_wav・phone_wav は data/ の中の録音（スマホの録音は afconvert で wav にしておく）
  --layout play/layout.json : make_play.py が作った並びの情報
  --k 4                     : ILRMA の基底の数（事前登録で決めた値）
  --inits 3                 : ILRMA の初期値の種の数（事前登録で決めた値。解析では種の平均を使う）
  --phone-ch 0              : スマホの録音がステレオのとき、使うチャネル（混ぜると2本のマイクの和になるので混ぜない）

やること（セッションごと）
  1. 2つの録音を 16kHz に直し、合図の組を2組ずつさがす（find_chirps）
  2. 合図から eps と開始のずれを推定し（fit_clock）、事前登録した除外の基準を調べる
       ・合図の高さ（包絡線の中央値との比）が 20 未満 ・2つのファイルの eps の差が 1ppm を超える
       ・録音が割れている（|x| ≥ 0.999 のサンプルがある） ・同じファイルの始めと終わりの合図の大きさが 1dB 以上違う（自動の音量調整）
     除外したセッションも clock.csv には残す（除いた理由と一緒に報告する）
  3. 区間ごとに、none（開始だけ合わせる）・chirp（合図で推定して読み直す）・blind（合図を使わずに推定して読み直す）の
     3条件の観測を「重ね合わせ」で作り、ILRMA で分離して BSS Eval で評価する。正解は Mac で録った音源ごとの音
結果：real_results.csv（1行＝セッション×区間×条件）、clock.csv（1行＝セッション）、sep/ に分離音
"""
import argparse
import json
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import fast_bss_eval

sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
import bss
import clockfix as cf

FS = 16000
NFFT, HOP = 2048, 512
Q_MIN, EPS_DIFF_MAX, AGC_DB_MAX = 20.0, 1.0, 1.0           # 事前登録した除外の基準


def load(path, ch=None):
    x, sr = sf.read(path, always_2d=True)
    x = x[:, ch if ch is not None else 0]
    clip = int(np.sum(np.abs(x) >= 0.999))                       # 使うチャネルだけで数える
    return librosa.resample(x, orig_sr=sr, target_sr=FS, res_type="soxr_hq"), sr, clip


def chirp_db(x, c, chirp):
    """見つけた位置の合図の大きさ [dB]（自動の音量調整の検出に使う）"""
    n = len(chirp)
    return [20 * np.log10(np.sqrt(np.mean(x[int(p): int(p) + n] ** 2)) + 1e-12) for p in c]


def evaluate(ref, est):
    sdr, sir, sar, perm = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return float(np.mean(sdr)), float(np.mean(sir)), float(np.mean(sar)), perm


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sessions")
    ap.add_argument("--layout", default="play/layout.json")
    ap.add_argument("--data", default="data")
    ap.add_argument("--k", type=int, default=4)
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--inits", type=int, default=3, help="ILRMA の初期値の種の数（結果は種ごとに1行。解析で平均する）")
    ap.add_argument("--phone-ch", type=int, default=0)
    ap.add_argument("--out", default="real_results.csv")
    ap.add_argument("--clock-out", default="clock.csv")
    a = ap.parse_args()
    for p in (a.out, a.clock_out):
        if pathlib.Path(p).exists():
            sys.exit(f"{p} はすでにあります（上書きしないように止めました）")
    lay = json.load(open(a.layout))
    lays = [lay, lay]
    spacing = int(round((lay["chirp_end"] - lay["chirp_start"]) * FS / lay["fs"]))
    L = int(round(lay["seg_sec"] * FS))
    chirp = cf.make_chirp(FS, cf.CHIRP_SEC)
    ses = pd.read_csv(a.sessions, dtype={"session": str})
    pathlib.Path("sep").mkdir(exist_ok=True)
    rows, clocks = [], []
    for _, s in ses.iterrows():
        r1, sr1, clip1 = load(pathlib.Path(a.data) / s["mac_wav"])
        r2, sr2, clip2 = load(pathlib.Path(a.data) / s["phone_wav"], a.phone_ch)
        reasons = []
        try:
            c1, q1 = cf.find_chirps(r1, chirp, spacing)
            c2, q2 = cf.find_chirps(r2, chirp, spacing)
        except ValueError as e:
            clocks.append(dict(session=s["session"], ok=False, reason=f"合図が見つからない（{e}）"))
            print(f"セッション {s['session']}：合図が見つからないので除外")
            continue
        clk = cf.fit_clock(c1, c2, [(0, 1), (2, 3)])
        d1, d2 = chirp_db(r1, c1, chirp), chirp_db(r2, c2, chirp)
        agc = max(abs(d1[1] - d1[0]), abs(d1[3] - d1[2]), abs(d2[1] - d2[0]), abs(d2[3] - d2[2]))
        if min(q1.min(), q2.min()) < Q_MIN:
            reasons.append("合図が弱い")
        if abs(clk["eps_each"][0] - clk["eps_each"][1]) * 1e6 > EPS_DIFF_MAX:
            reasons.append("2つのファイルの eps が合わない")
        if clip1 or clip2:
            reasons.append(f"割れている（Mac {clip1}・スマホ {clip2} サンプル）")
        if agc > AGC_DB_MAX:
            reasons.append(f"合図の大きさが {agc:.1f}dB 変わった（自動の音量調整？）")
        ok = not reasons
        clocks.append(dict(session=s["session"], date=s.get("date", ""), sr_mac=sr1, sr_phone=sr2,
                           eps_ppm=clk["eps"] * 1e6, eps1_ppm=clk["eps_each"][0] * 1e6, eps2_ppm=clk["eps_each"][1] * 1e6,
                           offset_s=clk["alpha"] / FS, q_min=float(min(q1.min(), q2.min())), agc_db=agc,
                           clip_mac=clip1, clip_phone=clip2, ok=ok, reason="・".join(reasons)))
        print(f"セッション {s['session']}：eps {clk['eps']*1e6:+.2f} ppm（ファイルごと "
              f"{clk['eps_each'][0]*1e6:+.2f}・{clk['eps_each'][1]*1e6:+.2f}）、開始のずれ {clk['alpha']/FS:+.3f} 秒、"
              f"合図の高さ {min(q1.min(), q2.min()):.0f}、合図の大きさの変化 {agc:.2f} dB → {'使う' if ok else '除外：' + '・'.join(reasons)}")
        if not ok:
            continue
        for j in range(len(lay["seg_starts"])):
            mix, ref = cf.build_mixture(r1, r2, c1, lays, j, L, clk["eps"], clk["alpha"], "none")
            mix_c, _ = cf.build_mixture(r1, r2, c1, lays, j, L, clk["eps"], clk["alpha"], "resample")
            eb = cf.drift_search(mix[0], mix[1], FS)[0]               # none の観測から ε だけを推定
            mix_b, _ = cf.build_mixture(r1, r2, c1, lays, j, L, eb, c2[0] - (1 + eb) * c1[0], "resample")
            with np.errstate(divide="ignore"):
                sdr_in = evaluate(ref, np.stack([mix[0], mix[0]]))[0]
            tag = f"sep/{s['session']}_{j}"
            if j == 0:
                sf.write(f"{tag}_mix_mac.wav", mix[0] / np.max(np.abs(mix)) * 0.9, FS)
                for k in range(2):
                    sf.write(f"{tag}_ref{k+1}.wav", ref[k] / np.max(np.abs(ref[k])) * 0.9, FS)
            for cond, m, eh in (("none", mix, clk["eps"]), ("chirp", mix_c, clk["eps"]), ("blind", mix_b, eb)):
                X = np.stack([librosa.stft(c, n_fft=NFFT, hop_length=HOP) for c in m], axis=-1)
                res = []
                for s_init in range(a.inits):
                    t0 = time.time()
                    Y = bss.ilrma(X, a.k, a.iter, seed=s_init)[0]
                    y = np.stack([librosa.istft(Y[:, :, n], hop_length=HOP, length=L) for n in range(2)])
                    sdr, sir, sar, perm = evaluate(ref, y)
                    y = y[perm]                               # y[k] が音源 k に対応するように並べ替える
                    if j == 0 and s_init == 0:
                        for k in range(2):
                            sf.write(f"{tag}_{cond}_y{k+1}.wav", y[k] / (np.max(np.abs(y[k])) + 1e-9) * 0.9, FS)
                    rows.append(dict(session=s["session"], seg=j, init=s_init, pair=f"{s['src1']}-{s['src2']}", cond=cond,
                                     k=a.k, eps_hat_ppm=eh * 1e6, sdr_in=sdr_in, sdr=sdr, sir=sir, sar=sar,
                                     sdr_imp=sdr - sdr_in, sec=time.time() - t0))
                    res.append(sdr - sdr_in)
                print(f"  区間 {j} {cond:5s} 推定 {eh*1e6:+8.2f} ppm  SDR改善 {np.mean(res):6.2f} dB"
                      f"（種 {a.inits} 個の平均。最小 {np.min(res):.2f}・最大 {np.max(res):.2f}）", flush=True)
        pd.DataFrame(rows).to_csv(a.out, index=False)
    pd.DataFrame(clocks).to_csv(a.clock_out, index=False)
    pd.DataFrame(rows).to_csv(a.out, index=False)
    if not rows:
        print("注意：使えるセッションがありませんでした（除外の理由は clock.csv）")
    print(f"{a.out}（{len(rows)} 行）と {a.clock_out}（{len(clocks)} セッション）を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
import json
import pandas as pd
truth = json.load(open("synth/truth.json"))
c = pd.read_csv("synth_clock.csv", dtype={"session": str}).set_index("session")
err = {s: abs(c.loc[s, "eps_ppm"] - truth[s]["eps_ppm"]) for s in ("s01", "s02", "s03", "s04")}
ok(max(err.values()) < 0.05, f"s01〜s04 の推定値が truth.json の eps_ppm と 0.05 ppm 以内で一致（最大 {max(err.values()):.4f} ppm。見本では最大 0.01 ppm）")
fd = (c["eps1_ppm"] - c["eps2_ppm"]).abs()
ok(fd.max() <= 0.05, f"2つのファイルの値の差が 0.05 ppm 以下（最大 {fd.max():.4f} ppm）")
ok(bool(c.loc[["s01", "s02", "s03", "s04"], "ok"].all()) and not bool(c.loc["s05", "ok"])
   and "合図の大きさが 3.0dB 変わった" in str(c.loc["s05", "reason"]),
   f"s01〜s04 は「使う」、最後の s05 だけ「{c.loc['s05', 'reason']}」で除外された")
ok(c.loc[["s01", "s02", "s03", "s04"], "q_min"].min() >= 20, f"使うセッションの合図の高さが 20 以上（最小 {c.loc[['s01', 's02', 's03', 's04'], 'q_min'].min():.0f}。目安は 100 以上）")
r = pd.read_csv("synth_results.csv")
ok(len(r) == 4 * 3 * 3 * r["init"].nunique(), f"synth_results.csv は 4 セッション × 3 区間 × 3 条件 × 種 {r['init'].nunique()} つ ＝ {len(r)} 行")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['fink_real_stats.py', 'synth_results.csv', 'synth_clock.csv']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python fink_real_stats.py synth_results.csv synth_clock.csv
#!/usr/bin/env python3
"""最終演習K：実録音の結果（real_results.csv・clock.csv）を解析し、シミュレーションの予想と比べる

使い方: cd ~/home-lab/final/k && uv run python fink_real_stats.py real_results.csv clock.csv --sim sim_main.csv --t60 0.256

【主解析（事前登録）】H4：実録音でも chirp − none > 0。単位はセッション（区間はセッションの中の副次標本）。
  セッションを変量効果にした混合モデル（切片だけ）と、セッションごとの平均の片側 t 検定を並べる
【副次解析】
  ・時計のずれ eps：セッションごとの値、平均と 95% 区間、ファイルごとの値の差（推定の精度の目安）
  ・blind − chirp（合図なしでも、合図ありと同じくらい直せるか）
  ・シミュレーションの予想（同じくらいのずれと、--t60 の部屋）との比較
【統計の誤用の例】区間を独立な標本として数える（疑似反復）と、区間が狭く、p が小さく出すぎる
"""
import argparse
import sys

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("results")
    ap.add_argument("clock")
    ap.add_argument("--sim")
    ap.add_argument("--t60", type=float, default=0.256)
    ap.add_argument("--k", type=int, default=4)
    a = ap.parse_args()
    r = pd.read_csv(a.results, dtype={"session": str})
    c = pd.read_csv(a.clock, dtype={"session": str})

    print("[1] 時計のずれ（Mac を基準にしたスマホのサンプリング周波数のずれ）")
    for _, s in c.iterrows():
        if pd.isna(s.get("eps_ppm")):
            print(f"  {s['session']}：{s['reason']}")
            continue
        print(f"  {s['session']}：{s['eps_ppm']:+.2f} ppm（ファイルごと {s['eps1_ppm']:+.2f}・{s['eps2_ppm']:+.2f}）"
              f"{'' if s['ok'] else '  除外：' + str(s['reason'])}")
    ok = c[c["ok"] == True]
    e = ok["eps_ppm"]
    if len(e) >= 2:
        ci = stats.t.interval(0.95, len(e) - 1, e.mean(), e.std(ddof=1) / np.sqrt(len(e)))
        print(f"  使ったセッション {len(e)} 個：平均 {e.mean():+.2f} ppm（95% 区間 {ci[0]:+.2f}〜{ci[1]:+.2f}）、"
              f"セッション間の標準偏差 {e.std(ddof=1):.2f} ppm、ファイルごとの値の差の絶対値の平均 {(ok['eps1_ppm']-ok['eps2_ppm']).abs().mean():.3f} ppm")
        print(f"  8秒の区間の終わりで、ずれは {abs(e.mean())*1e-6*8*16000:.2f} サンプル（16kHz）＝ {abs(e.mean())*8:.0f} µs、"
              f"4kHz の位相で {360*4000*abs(e.mean())*1e-6*8:.0f} 度")

    if r.empty:
        sys.exit("使えるセッションがありません（clock.csv の除外の理由を確かめる）")
    w = r[r["k"] == a.k].pivot_table(index=["session", "seg"], columns="cond", values="sdr_imp", aggfunc="mean")   # 種の平均
    w["d"] = w["chirp"] - w["none"]
    w["db"] = w["blind"] - w["chirp"]
    print(f"\n[2] 主解析 H4：chirp − none > 0（{w.index.get_level_values(0).nunique()} セッション × 区間 = {len(w)} 個）")
    per = w.groupby(level="session")["d"].mean()
    n = len(per)
    ci_d = (per.mean(), per.mean())
    if n >= 2:
        t = stats.ttest_1samp(per, 0, alternative="greater")
        ci_d = stats.t.interval(0.95, n - 1, per.mean(), per.std(ddof=1) / np.sqrt(n))
        print(f"  セッションごとの平均で：差 {per.mean():+.2f} dB（95% 区間 {ci_d[0]:+.2f}〜{ci_d[1]:+.2f}）、片側 p = {t.pvalue:.3g}、n = {n} セッション")
    dd = w.reset_index()
    if n >= 3:
        m = smf.mixedlm("d ~ 1", dd, groups=dd["session"]).fit(reml=True)
        est, se = m.fe_params["Intercept"], m.bse_fe["Intercept"]
        var_s, var_e = float(m.cov_re.iloc[0, 0]), m.scale
        icc = var_s / (var_s + var_e)
        print(f"  混合モデル（セッションの変量効果）：差 {est:+.2f} dB（標準誤差 {se:.2f}）、級内相関 {icc:.2f}")
        mseg = len(w) / n
        print(f"  設計効果 1 + (m − 1)·ICC = {1 + (mseg - 1) * icc:.2f}（m = 1セッションあたりの区間 {mseg:.1f}）")
    naive = stats.ttest_1samp(w["d"], 0, alternative="greater")
    nci = stats.t.interval(0.95, len(w) - 1, w["d"].mean(), w["d"].std(ddof=1) / np.sqrt(len(w)))
    ratio = (nci[1] - nci[0]) / (ci_d[1] - ci_d[0]) if n >= 2 and ci_d[1] > ci_d[0] else np.nan
    if n < 2:
        print("  セッションが2つ以上ないと、セッションを単位にした区間が出せない（疑似反復の例は比べられない）")
    print(f"  【誤り：疑似反復】区間を独立に {len(w)} 個と数えると：95% 区間 {nci[0]:+.2f}〜{nci[1]:+.2f}、片側 p = {naive.pvalue:.3g}"
          f"（区間の幅はセッション単位の {ratio:.2f} 倍）")
    if np.isnan(ratio):
        pass
    elif ratio < 1:
        print(f"  狭くなる理由は2つ：①同じセッションの区間が似ている（級内相関）のに、標準誤差を √{len(w)} で割ってしまう。"
              f"②自由度が増えて t の分位点が小さくなる（自由度 {n-1} で {stats.t.ppf(0.975, n-1):.2f}、自由度 {len(w)-1} で "
              f"{stats.t.ppf(0.975, len(w)-1):.2f}）。セッションが少ないと②の寄与も大きい")
        print("  → 同じセッションの区間は、時計のずれ・部屋・置き方が同じなので似ている（級内相関が正）。独立と数えると区間が狭く、p が小さく出すぎる")
    else:
        print("  → このデータでは級内相関が小さく、区間はあまり狭くならなかった。それでも、独立かどうかはデータを見る前に設計で決める（単位はセッション）")

    print("\n[3] 副次：blind − chirp（合図を使わない推定は、合図ありとどれだけ違うか）")
    pb = w.groupby(level="session")["db"].mean()
    if len(pb) >= 2:
        ci = stats.t.interval(0.95, len(pb) - 1, pb.mean(), pb.std(ddof=1) / np.sqrt(len(pb)))
        print(f"  差 {pb.mean():+.2f} dB（95% 区間 {ci[0]:+.2f}〜{ci[1]:+.2f}）")
    eb = r[r["cond"] == "blind"].merge(c[["session", "eps_ppm"]], on="session")
    print(f"  合図なしの推定 − 合図ありの推定：平均 {(eb['eps_hat_ppm'] - eb['eps_ppm']).mean():+.2f} ppm、"
          f"標準偏差 {(eb['eps_hat_ppm'] - eb['eps_ppm']).std():.2f} ppm")

    pred = None
    if a.sim and len(e):
        s = pd.read_csv(a.sim)
        s = s[(s["k"] == a.k) & (np.isclose(s["rt60"], a.t60))]
        sw = s[s["cond"].isin(["none", "chirp"])].pivot_table(index=["trial", "eps_ppm"], columns="cond", values="sdr_imp",
                                                              aggfunc="mean")
        g = (sw["chirp"] - sw["none"]).groupby(level="eps_ppm").mean()
        # ずれ 0 では補正の効果も 0 なので、(0, 0) の点を足して、ずれ（ppm）の直線で補間する（10 ppm より小さいときの過大評価を防ぐ）
        xs = np.r_[0.0, g.index.values]
        ys = np.r_[0.0, g.values]
        pred = float(np.interp(abs(e.mean()), xs, ys))
        print(f"\n[4] シミュレーションの予想との比較（T60 {a.t60*1000:.0f} ms、ずれ {abs(e.mean()):.1f} ppm を (0,0) と各 ppm の点の間で補間）")
        print(f"  予想：chirp − none ≈ {pred:+.2f} dB、実録音：{per.mean():+.2f} dB")
        print("  【誤り】シミュレーションの数字を、そのまま実録音の結果として書くこと。部屋の残響・雑音・スピーカの指向性・"
              "マイクの特性・自動の音量調整などが違うので、実録音で確かめた値だけを実録音の結果として書く")

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
    if len(e):
        ax[0].errorbar(range(len(ok)), ok["eps_ppm"], fmt="o", label="セッションごと")
        ax[0].plot(range(len(ok)), ok["eps1_ppm"], "x", color="C1", label="ファイル1")
        ax[0].plot(range(len(ok)), ok["eps2_ppm"], "+", color="C2", label="ファイル2")
        ax[0].set_xticks(range(len(ok)), ok["session"])
    ax[0].set_ylabel("スマホの時計のずれ [ppm]")
    ax[0].set_title("(a) セッションごとの時計のずれ")
    ax[0].legend(fontsize=8)
    ax[0].grid(alpha=0.3)
    for i, (sid, g) in enumerate(w.groupby(level="session")):
        for _, row in g.iterrows():
            ax[1].plot([0, 1, 2], [row["none"], row["chirp"], row["blind"]], "-o", ms=3, alpha=0.6, color=f"C{i % 10}")
        ax[1].plot([], [], "-o", color=f"C{i % 10}", label=sid)
    ax[1].set_xticks([0, 1, 2], ["補正なし", "合図で補正", "合図なしで補正"])
    ax[1].set_ylabel("SDR 改善量 [dB]")
    ax[1].set_title("(b) 区間ごと（色＝セッション）")
    ax[1].legend(fontsize=7, ncol=2)
    ax[1].grid(alpha=0.3)
    ax[2].bar([0], [per.mean()], yerr=[[per.mean() - ci_d[0]], [ci_d[1] - per.mean()]],
              capsize=4, color="C0", label="実録音（セッション単位、95% 区間）")
    if pred is not None:
        ax[2].bar([1], [pred], color="C7", label="シミュレーションの予想")
    ax[2].set_xticks([0, 1], ["実録音", "シミュレーション"])
    ax[2].set_ylabel("合図で補正 − 補正なし [dB]")
    ax[2].set_title("(c) 補正の効果：実録音と予想")
    ax[2].legend(fontsize=8)
    ax[2].axhline(0, color="k", lw=0.8)
    fig.tight_layout()
    fig.savefig("fink_real.png", dpi=130)
    plt.show()   # ノートブック版：図を表示する
    print("\nfink_real.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
ratio = num(r"区間の幅はセッション単位の ([\d.]+) 倍")
ok(ratio < 1, f"[2] で、区間を独立に数えた区間がセッション単位の区間より狭く出る（幅 {ratio} 倍。見本は種 3 つで 0.54 倍）")
d4 = num(r"セッションごとの平均で：差 ([-+\d.]+) dB")
ok(d4 > 0, f"[2] の H4（合図で補正 − 補正なし）のセッションの平均が正（{d4:+.2f} dB。見本は種 3 つで +2.22 dB）")
ok("fink_real.png を保存しました" in cap.stdout, "fink_real.png ができた")

分離音を聞き比べます（段階5 の 5 の練習。最初は小さな音量で）：Mac で録った混合音 → 正解（音源1）→ 補正なし → 合図で補正 → 合図なしで補正（どれも s01 の区間 0、音源1 に当たる出力）。教科書の確認ポイントは「補正なしでは、高い音（子音、楽器の倍音）にもう一方の音源が残って聞こえ、補正ありではそれが小さくなる」です。合成の録音でも、区間や初期値によっては差が聞き取りにくいことがあります。聞こえ方は再生の環境と人によって違うので、ノートに自分の言葉で書いてください。

In [ ]:
# wav を聞く（パソコンの音量を一番小さくしてから。1つずつ）
for f in ['sep/s01_0_mix_mac.wav', 'sep/s01_0_ref1.wav', 'sep/s01_0_none_y1.wav', 'sep/s01_0_chirp_y1.wav', 'sep/s01_0_blind_y1.wav']:
    print(f)
    listen_file(f)

**スマホの録音の設定を確かめる（段階3 の 3）、予備のセッションを録る（段階3 の 4）は、実際の機器が要るので、ノートブックには入れていません。** 予備のセッションでは、`real_mix.py` の表示で合図の高さが 20 以上（目安 100 以上）、2 つのファイルの ε の差が 1 ppm 以下、合図の大きさの変化が 1 dB 以下で「使う」と判定されることを確かめます。

## 6. 実録音のセッションの数 M を出す（K-4-3、段階3 の 5、`fink_power.py --real-eps`）

教科書では、予備のセッションで測った ε を `--real-eps` に入れます。ここでは run_fink.sh の見本の値 20 ppm を使います。効果は、予備実験（T60 256 ms）の 10 ppm と 30 ppm の点の間の直線の補間です。試しの設定（2 試行）では、M は教科書と合いません。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['fink_power.py', 'sim_pilot.csv', 'sim_init.csv', '--real-eps', '20']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python fink_power.py sim_pilot.csv sim_init.csv --real-eps 20
#!/usr/bin/env python3
"""最終演習K：予備実験の結果から、K を選び、初期値の種の数 S と試行の数 n を決める（検出力の計算）

使い方: cd ~/home-lab/final/k
  uv run python fink_power.py sim_pilot.csv                       （1回目：K を選ぶだけ）
  uv run python fink_power.py sim_pilot.csv sim_init.csv --eps 30 --delta 1.0 --sesoi 1.0   （2回目：S と n を決める）
  sim_pilot.csv : 予備実験（K の候補を全部、初期値の種は1つ）
  sim_init.csv  : 初期値の種だけを変えた実験（1回目で選んだ K、30 ppm、sync・none・chirp・blind、種 4 つ）
  --delta 1.0   : 同等性の幅 Δ [dB]（事前登録の値。計画書に理由を書く）
  --sesoi 1.0   : 「補正の効果」として見逃したくない最小の大きさ [dB]
  --alpha 0.05 --power 0.8 --tests 3 : 有意水準・検出力・確証的な検定の数（ホルム法の最悪の場合 α/3 で計画する）

やること
  1. K の選び方：予備実験の sync（同期した理想の録音）の SDR 改善量の平均が一番大きい K（本実験の試行は使わない）
  2. 初期値の種だけを変えたときのばらつき
  3. 試行ごとの差 D（2つの T60 の平均）の分散を、2つの成分に分ける（変量効果のモデル）
       D_is = μ + b_i + e_is   （i：試行、s：初期値の種。b_i の分散 σ_b²、e_is の分散 σ_w²）
       試行ごとに S 個の種で平均した差の分散は σ_D²(S) = σ_b² + σ_w²/S
       σ_w² は「同じ試行の中での種の間の分散」の平均、σ_b² は「試行ごとの平均の分散 − σ_w²/S0」（S0＝実験の種の数）
  4. S ごとに、必要な試行の数 n を求める
       H1（片側の対応のある t 検定、効果 sesoi）：非心 t 分布で検出力を計算し、目標を超える最小の n。
       H2・H3（TOST、本当の差 0、幅 Δ）：近似式で n を出し、正規分布の乱数で TOST を実際に行うシミュレーションで確かめる
     ILRMA の回数（＝計算時間）が最も少ない (S, n) を選ぶ
"""
import argparse

import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(0)
CONTRASTS = [("H1 補正の効果 chirp − none > 0", "chirp", "none", "t"),
             ("H2 同等性 chirp − sync", "chirp", "sync", "tost"),
             ("H3 同等性 blind − sync", "blind", "sync", "tost")]


def per_init_diff(q, eps, a, b):
    """初期値の種ごとの、試行ごとの差 D_is（2つの T60 の平均）。表は 試行 × 種"""
    def get(c):
        e = 0.0 if c == "sync" else eps
        return q[(q["cond"] == c) & (q["eps_ppm"] == e)].set_index(["trial", "init", "rt60"])["sdr_imp"]
    x = (get(a) - get(b)).dropna()
    return x.groupby(level=["trial", "init"]).mean().unstack("init")


def power_t(n, delta, sd, alpha):
    df = n - 1
    return 1 - stats.nct.cdf(stats.t.ppf(1 - alpha, df), df, delta / (sd / np.sqrt(n)))


def power_tost_approx(n, margin, sd, alpha):
    p_one = power_t(n, margin, sd, alpha)
    return max(0.0, 1 - 2 * (1 - p_one))


def power_tost_sim(n, margin, sd, alpha, sims=20000):
    x = RNG.normal(0.0, sd, (sims, n))
    m, se = x.mean(1), x.std(1, ddof=1) / np.sqrt(n)
    h = stats.t.ppf(1 - alpha, n - 1) * se
    return np.mean((m - h > -margin) & (m + h < margin))


def need_n(kind, sd, a, alpha):
    for n in range(4, 501):
        pw = power_t(n, a.sesoi, sd, alpha) if kind == "t" else power_tost_approx(n, a.delta, sd, alpha)
        if pw >= a.power:
            return n, pw
    return np.nan, np.nan


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("pilot")
    ap.add_argument("init", nargs="?")
    ap.add_argument("--eps", type=float, default=30.0)
    ap.add_argument("--delta", type=float, default=1.0)
    ap.add_argument("--sesoi", type=float, default=1.0)
    ap.add_argument("--alpha", type=float, default=0.05)
    ap.add_argument("--power", type=float, default=0.8)
    ap.add_argument("--tests", type=int, default=3)
    ap.add_argument("--real-eps", type=float, help="予備のセッションで測った時計のずれ [ppm]（実録音のセッション数 M を出す）")
    a = ap.parse_args()
    d = pd.read_csv(a.pilot)
    print(f"予備実験：{d['trial'].nunique()} 試行、T60 {[f'{r*1000:.0f}ms' for r in sorted(d['rt60'].unique())]}、"
          f"ずれ {[f'{e:.0f}' for e in sorted(d['eps_ppm'].unique()) if e]} ppm、K の候補 {[int(x) for x in sorted(d['k'].unique())]}")

    print("\n[1] K の選び方（予備実験の sync の SDR 改善量の平均 [dB]。本実験の試行は使わない）")
    sync = d[d["cond"] == "sync"].groupby(["k", "rt60"])["sdr_imp"].mean().unstack()
    sync.columns = [f"T60 {c*1000:.0f}ms" for c in sync.columns]
    print(sync.round(2).to_string())
    k = int(d[d["cond"] == "sync"].groupby("k")["sdr_imp"].mean().idxmax())
    print(f"  → K = {k}（初期値の実験 sim_init.csv も、この K で行う）")
    if not a.init:
        print("  sim_init.csv を作ったら、もう一度 sim_init.csv をつけて実行する")
        return
    q = pd.read_csv(a.init)
    if int(q["k"].iloc[0]) != k:
        print(f"  注意：sim_init.csv の K（{int(q['k'].iloc[0])}）が選んだ K と違う。選んだ K でやり直す")

    S0 = q["init"].nunique()
    g = q.groupby(["trial", "rt60", "cond"])["sdr_imp"]
    print(f"\n[2] 初期値の種だけを {S0} 通り変えたときのばらつき（{q['trial'].nunique()} 試行、{len(g)} 組）")
    print(f"  同じ試行・同じ条件の中の標準偏差：{np.sqrt(g.var(ddof=1).mean()):.2f} dB、最大 − 最小の平均 {(g.max() - g.min()).mean():.2f} dB")
    by = q.groupby(["cond", "rt60"]).apply(lambda x: np.sqrt(x.groupby("trial")["sdr_imp"].var(ddof=1).mean()))
    print("  条件・T60 ごと：" + "、".join(f"{c} {r*1000:.0f}ms {v:.2f}" for (c, r), v in by.items()))

    alpha_plan = a.alpha / a.tests
    print(f"\n[3] 差 D の分散の成分（{a.eps:.0f} ppm、2つの T60 の平均）。σ_D(S)² = σ_b² + σ_w²/S")
    comp = {}
    for name, c1, c2, kind in CONTRASTS:
        t = per_init_diff(q, a.eps, c1, c2)
        sw2 = float(t.var(axis=1, ddof=1).mean())
        sb2 = max(float(t.mean(axis=1).var(ddof=1)) - sw2 / S0, 0.0)
        comp[name] = (sb2, sw2, kind, float(t.values.mean()))
        print(f"  {name}：平均 {t.values.mean():+.2f} dB、σ_b = {np.sqrt(sb2):.2f}、σ_w = {np.sqrt(sw2):.2f} dB"
              f"（種1つなら σ_D = {np.sqrt(sb2 + sw2):.2f}）")

    dk = d[d["k"] == k]
    runs_trial = dk.groupby("trial").size().mean()                     # 種1つのときの1試行の ILRMA の回数
    sec_run = dk["sec"].median()                                       # 中央値（スリープなどの外れ値に引っぱられないように）
    print(f"\n[4] 種の数 S と必要な試行の数 n（α = {a.alpha}/{a.tests} = {alpha_plan:.4f}、検出力 {a.power}、"
          f"H1 の効果 {a.sesoi} dB、同等性の幅 ±{a.delta} dB）")
    print(f"  1試行の ILRMA は種1つで {runs_trial:.0f} 回、1回 {sec_run:.1f} 秒（予備実験の中央値）")
    best = None
    for S in (1, 2, 3, 5, 10):
        ns = []
        txt = []
        for name, (sb2, sw2, kind, m) in comp.items():
            sd = np.sqrt(sb2 + sw2 / S)
            n, pw = need_n(kind, sd, a, alpha_plan)
            ns.append(n)
            txt.append(f"{name[:2]} σ_D {sd:.2f}→n {n}")
        if any(np.isnan(ns)):
            print(f"  S = {S:2d}：" + "、".join(txt) + " → 必要な n が 500 を超える（Δ を見直すか、ばらつきを減らす工夫が要る）")
            continue
        n_all = int(np.ceil(max(ns + [12]) / 3) * 3)                    # 音源の組（3組）を同じ数ずつ。最低12
        hours = n_all * runs_trial * S * sec_run / 3600
        print(f"  S = {S:2d}：" + "、".join(txt) + f" → n = {n_all}、ILRMA {n_all*runs_trial*S:.0f} 回、約 {hours:.1f} 時間")
        if best is None or hours < best[2]:
            best = (S, n_all, hours)
    if best is None:
        print("  どの S でも n が 500 を超えた。計画を見直す")
        return
    S, n_all, hours = best
    print(f"  → 計算時間が最も短いのは S = {S}、n = {n_all}（約 {hours:.1f} 時間。ターミナルを3つに分けて --first で並べると約1/3）")

    print(f"\n[5] 確かめ：S = {S}、n = {n_all} の検出力（式 と シミュレーション）")
    for name, (sb2, sw2, kind, m) in comp.items():
        sd = np.sqrt(sb2 + sw2 / S)
        if kind == "t":
            f = power_t(n_all, a.sesoi, sd, alpha_plan)
            sim = np.mean([stats.ttest_1samp(RNG.normal(a.sesoi, sd, n_all), 0, alternative="greater").pvalue < alpha_plan
                           for _ in range(4000)])
        else:
            f = power_tost_approx(n_all, a.delta, sd, alpha_plan)
            sim = power_tost_sim(n_all, a.delta, sd, alpha_plan)
        print(f"  {name}：式 {f:.3f}、シミュレーション {sim:.3f}")
        if kind == "tost" and abs(m) > a.delta / 2:
            print(f"    注意：予備の差の平均 {m:+.2f} dB が幅の半分を超えている。本当に差があれば、同等と言える確率はこれより低い")

    if a.real_eps is not None:
        # 実録音（H4）のセッションの数 M：シミュレーションの T60 256ms の「試行の間の chirp − none」の平均と標準偏差を、
        # 測ったずれに合わせて使う（ずれ 0 で効果 0、ずれの点の間を直線で補間。標準偏差は近いずれの値）
        g = dk[np.isclose(dk["rt60"], 0.256)].pivot_table(index=["trial", "eps_ppm"], columns="cond", values="sdr_imp")
        dd = (g["chirp"] - g["none"]).dropna().groupby(level="eps_ppm")   # sync だけの組（eps 0）は差が NaN なので除く
        mu, sd = dd.mean(), dd.std(ddof=1)
        e = abs(a.real_eps)
        eff = float(np.interp(e, np.r_[0.0, mu.index.values], np.r_[0.0, mu.values]))
        s_near = float(sd.iloc[int(np.argmin(np.abs(sd.index.values - max(e, sd.index.min()))))])
        print(f"\n[6] 実録音のセッションの数 M（測ったずれ {e:.1f} ppm、T60 256ms のシミュレーションから：効果 {eff:+.2f} dB、標準偏差 {s_near:.2f} dB）")
        if eff < 1.0:
            print("  予想される効果が 1dB 未満。H4 は検定せず、差と区間の記述にとどめる（事前登録どおり）")
        else:
            for M in range(3, 101):
                pw = power_t(M, eff, s_near, a.alpha)
                if pw >= a.power:
                    break
            print(f"  片側 α = {a.alpha}、検出力 {a.power}：M = {max(M, 6)}（計算では {M}、最低6。検出力 {pw:.3f}）")
            print("  注意：シミュレーションの部屋は家の部屋より分離しやすいことが多いので、予備のセッションの区間の差も見て、多めにとる")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
eff = num(r"T60 256ms のシミュレーションから：効果 ([-+\d.na]+) dB")
ok(-3 < eff < 10, f"[6] に効果の予想が数で出て、−3〜+10 dB の範囲にある（{eff:+.2f} dB。試しの設定の見本は +2.29 dB。nan ならこの確かめで止まる）")
if eff < 1.0:
    ok("H4 は検定せず、差と区間の記述にとどめる" in cap.stdout, "予想される効果が 1dB 未満なので「H4 は検定せず記述にとどめる」と表示された")
else:
    M = int(num(r"：M = (\d+)（計算では"))
    ok(M >= 6, f"[6] に M（最低 6）が表示された（M = {M}）")

## 7. 本実験（シミュレーション）と主解析（K-4、段階4 の 1・段階5 の 1、`sim_async.py`・`fink_stats.py`）

**ノートブックの既定は試しの設定**（`--trials 3`：3 試行 × 2 T60 × 13 観測 = 78 行、約 6 分）です。教科書の本番は、計画書の K・S・n（見本 K = 4、S = 1、n = 150）で 3900 行、1 つずつなら約 5.4 時間、ターミナルを 3 つに分けて約 2 時間です（`--first 0 --trials 50` などに分け、各ターミナルで `export VECLIB_MAXIMUM_THREADS=1` とし、最後に CSV をつなぐ。run_fink.sh の段階4）。ノートブックの K は、教科書の見本と同じ 4 に固定します（主解析の `--k 4` と合わせるため）。

3 試行では、主解析の区間は広く、p 値も判定も K-7-1 の (2) とは合いません（「支持されない」になることもあります）。ここでは、表と図の読み方、H1〜H3 の差の向き、ずれの推定の精度を確かめます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
# 本番（教科書の段階4 の 1。n = 150、約5.4時間）：sys.argv = ["sim_async.py", "--trials", "150", "--seed", "2000", "--k", "4", "--out", "sim_main.csv"]
fresh('sim_main.csv')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ['sim_async.py', '--trials', '3', '--seed', '2000', '--k', '4', '--out', 'sim_main.csv']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python sim_async.py --trials 3 --seed 2000 --k 4 --out sim_main.csv
#!/usr/bin/env python3
"""最終演習K：2台の録音機による「非同期の2チャネル録音」をシミュレーションで作り、ずれの補正の効果を ILRMA で調べる

使い方: cd ~/home-lab/final/k
        uv run python sim_async.py --trials 6 --seed 1000 --k 2 4 --out sim_pilot.csv                 （予備実験）
        uv run python sim_async.py --trials 150 --seed 2000 --k 4 --out sim_main.csv                  （本実験。値は計画書のもの）
  --sources : 音源の wav、または話者ごとのフォルダ（中の wav・flac をつなぐ。JVS・LibriTTS の1人分など）。3つ以上。
              組み合わせを試行ごとに順に使う。省略すると librosa の例 libri1・libri2・trumpet（短い音源はくり返してつなぐ）
  --eps     : 機器2 のサンプリング周波数のずれ [ppm]（いくつでも）
  --rt60    : 残響時間 [s]（いくつでも。同じ部屋の形・置き方で吸音だけ変える）
  --k       : ILRMA の基底の数（いくつでも。予備実験で選び、本実験では1つに固定する）
  --seed    : 乱数の種のはじめ（試行 i は seed + i）。予備実験と本実験で重ならないようにする
  --gen fft / --drift 0.5 : 副次の確かめ。機器2 の録音を「直すのとは別の方法」で作る（作りと直しが同じだと、同等になるのが
              当たり前になる。inverse crime）、または時計のずれが時間とともに変わる録音にする
  --first   : 最初の試行の番号（例：--first 12 --trials 12 で後半だけ。分けて実行し、あとで CSV をつなぐ）

1回の試行（部屋の形・機器と音源の置き方・音源の組と区間）の中で、次の条件をすべて同じ録音から作る（対応のある比較）
  sync   : 同期した理想の録音機（機器2 の時計が機器1 と同じ。ε = 0）  … シミュレーションでしか作れない
  none   : 開始のずれだけ合図で合わせ、サンプリング周波数のずれは直さない
  chirp  : 合図（チャープ）で eps と開始のずれを推定し、機器2 を読み直す
  blind  : eps だけを、合図を使わずに none の観測（開始のずれは合図で合わせたもの）から推定し、
           その eps で並べ直して読み直す（合図は開始の位置だけに使う）
  oracle : eps だけ真の値で読み直す（chirp との差＝eps の推定の誤差の影響）
  機器2 の録音は、どの条件も同じ経路（大きさ ±6dB・帯域制限の読み直し・同じ雑音）で作り、開始は合図の始めの位置で合わせる。
  そのため sync と oracle の差は「ずれのある時計で録って、読み直したこと」だけによる。
結果は1行＝試行×T60×ずれ×条件×K×初期値の種。sdr などは2つの音源の平均。所要時間の目安は K-2 に書いた
"""
import argparse
import itertools
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import librosa
import pyroomacoustics as pra
import fast_bss_eval

sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
import bss                                                   # 第7章の自前の ILRMA（~/home-lab/ch7/bss.py）
import clockfix as cf

try:                                                         # IV-5 の共通ライブラリ（--gen fft のときだけ使う）
    sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
    from common import array as arr
except ImportError:
    arr = None

FS = 16000
NFFT, HOP = 2048, 512                                        # 第7章と同じ（16kHz で 128ms の窓）
SNR_DB = 35.0                                                # 録音機の雑音（静かな部屋の目安）


def load_sources(paths, fs):
    if not paths:
        paths = [librosa.ex(n) for n in ("libri1", "libri2", "trumpet")]
    out = []
    for p in paths:
        p = pathlib.Path(p)
        if p.is_dir():                                         # フォルダなら、中の wav・flac を名前の順につなぐ（1人の話者＝1つの音源）
            files = sorted([*p.rglob("*.wav"), *p.rglob("*.flac")])[:60]
            x = np.concatenate([librosa.effects.trim(librosa.load(f, sr=fs, mono=True)[0], top_db=40)[0] for f in files])
        else:
            x, _ = librosa.load(p, sr=fs, mono=True)
        x = x / (np.std(x) + 1e-9)
        out.append(x)
    return out


def excerpt(x, n, rng):
    """長さ n を切り出す（短い音源はくり返してつなぐ。つなぎ目は 20ms で重ねる）"""
    while len(x) < n + 1:
        r = int(0.02 * FS)
        fade = np.linspace(0, 1, r)
        x = np.r_[x[:-r], x[-r:] * (1 - fade) + x[:r] * fade, x[r:]]
    a = rng.integers(0, len(x) - n)
    return x[a: a + n]


def place(rng):
    """家の部屋（6〜8畳くらい）と、机の上の2台の録音機、2つの音源の位置"""
    dim = np.array([rng.uniform(3.2, 4.5), rng.uniform(2.6, 3.6), 2.4])
    while True:
        c = np.array([rng.uniform(1.2, dim[0] - 1.2), rng.uniform(1.2, dim[1] - 1.2), 0.8])
        d = rng.uniform(0.3, 1.0)                              # 2台の間隔 30〜100cm
        th = rng.uniform(0, np.pi)
        u = np.array([np.cos(th), np.sin(th), 0])
        mics = np.c_[c - d / 2 * u, c + d / 2 * u]
        ang = rng.uniform(0, 2 * np.pi, 2)
        diff = abs((ang[0] - ang[1] + np.pi) % (2 * np.pi) - np.pi)
        if diff < np.radians(40):
            continue
        srcs = []
        for a in ang:
            r = rng.uniform(0.8, 1.6)
            srcs.append(c + np.array([r * np.cos(a), r * np.sin(a), rng.uniform(0.3, 0.5)]))
        if not all(np.all(s[:2] > 0.4) and np.all(s[:2] < dim[:2] - 0.4) for s in srcs):
            continue
        # 2本のマイクで見分けられる置き方か：2つの音源の到達時間差（マイク2 − マイク1）が 3 サンプル以上違うこと。
        # マイクを結ぶ線をはさんで鏡に映した位置や、2台の真ん中の垂直な面の近くに2つとも置くと、到達時間差が同じになり分けられない
        tdoa = [(np.linalg.norm(s - mics[:, 1]) - np.linalg.norm(s - mics[:, 0])) / 343.0 * FS for s in srcs]
        if abs(tdoa[0] - tdoa[1]) >= 3.0:
            return dim, mics, srcs, d


def simulate(dim, mics, srcs, rt60, session):
    """部屋の中で、2つの再生ファイルを順番に鳴らしたときの、2本のマイクの連続した信号（同期・理想）"""
    if rt60 == 0:                                              # 無響（直接音だけ）
        room = pra.ShoeBox(dim, fs=FS, max_order=0)
    else:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    room.add_microphone_array(mics)
    for k in range(2):
        room.add_source(srcs[k], signal=session[k])
    room.simulate()
    return room.mic_array.signals                               # (2, 時間)


def evaluate(ref, est):
    sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return float(np.mean(sdr)), float(np.mean(sir)), float(np.mean(sar))


def stft(x):
    return np.stack([librosa.stft(c, n_fft=NFFT, hop_length=HOP) for c in x], axis=-1)     # (F, T, M)


def istft(Y, n):
    return np.stack([librosa.istft(Y[:, :, k], hop_length=HOP, length=n) for k in range(Y.shape[2])])


def separate(mix, k, n_iter, seed):
    return istft(bss.ilrma(stft(mix), k, n_iter, seed=seed)[0], mix.shape[1])


def oracle_w(mix, ref):
    """正解を使った「時間に依らない分離行列」の上限：周波数ごとに W_f = R_sx R_xx⁻¹（最小二乗）

    ILRMA と同じ「周波数ごとに1つの W_f」というモデルで、正解を知っていれば到達できる成績。
    これが下がるなら、ずれのせいで「モデルそのもの」が合わなくなっている。これが高いのに ILRMA だけ下がるなら「推定」のせい
    """
    X, S = stft(mix), stft(ref)
    Rxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Rsx = np.einsum("ftn,ftm->fnm", S, X.conj())
    W = Rsx @ np.linalg.inv(Rxx + 1e-9 * np.trace(Rxx, axis1=1, axis2=2)[:, None, None] * np.eye(2))
    return istft(np.einsum("fnm,ftm->ftn", W, X), mix.shape[1])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sources", nargs="*")
    ap.add_argument("--trials", type=int, default=2)
    ap.add_argument("--first", type=int, default=0, help="最初の試行の番号（何回かに分けて実行するとき）")
    ap.add_argument("--seed", type=int, default=1000)
    ap.add_argument("--eps", type=float, nargs="+", default=[10, 30, 100])
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.0, 0.256])
    ap.add_argument("--init-seeds", type=int, default=1, help="ILRMA の初期値の乱数の種をいくつ試すか")
    ap.add_argument("--k", type=int, nargs="+", default=[2])
    ap.add_argument("--dur", type=float, default=8.0)
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--conds", nargs="+", default=["sync", "none", "chirp", "blind", "oracle"])
    ap.add_argument("--gen", choices=["sinc", "fft"], default="sinc",
                    help="機器2 の録音の作り方（副次の確かめ）。fft は IV-5 の common/array.py の sfo_resample（直すほうとは別の方法）")
    ap.add_argument("--drift", type=float, default=0.0,
                    help="機器2 の時計のずれが時間とともに変わる速さ [ppm/分]（副次の確かめ。ε(t) = ε + drift·t）")
    ap.add_argument("--out", default="sim_pilot.csv")
    a = ap.parse_args()
    if pathlib.Path(a.out).exists():
        sys.exit(f"{a.out} はすでにあります（上書きしないように止めました）")
    srcs = load_sources(a.sources, FS)
    pairs = list(itertools.combinations(range(len(srcs)), 2))
    L = int(a.dur * FS)
    chirp = cf.make_chirp(FS, cf.CHIRP_SEC)
    rows, t_all = [], time.time()
    for i in range(a.first, a.first + a.trials):
        seed = a.seed + i
        rng = np.random.default_rng(seed)
        pair = pairs[i % len(pairs)]
        dim, mics, spos, d_mic = place(rng)
        # 再生ファイル2つ（音源ごとに区間を乱数で選ぶ）と、それを順番に鳴らすセッション
        plays, lays = [], []
        for k in range(2):
            w, lay = cf.build_play(excerpt(srcs[pair[k]], L, rng), FS, 1, a.dur)
            plays.append(w)
            lays.append(lay)
        spacing = lays[0]["chirp_end"] - lays[0]["chirp_start"]   # 1つのファイルの中の合図の間隔（サンプル）
        pad, gap = int(1.5 * FS), int(1.0 * FS)
        total = pad + len(plays[0]) + gap + len(plays[1]) + pad
        session = np.zeros((2, total))
        session[0, pad: pad + len(plays[0])] = plays[0]
        session[1, pad + len(plays[0]) + gap: pad + len(plays[0]) + gap + len(plays[1])] = plays[1]
        s0 = rng.uniform(0.2, 1.0) * FS                          # 機器2 が録音を始めた位置（機器1 のサンプル番号）
        gain2 = 10 ** (rng.uniform(-6, 6) / 20)                  # 機器2 の録音の大きさ（機器ごとに違う）
        for rt60 in a.rt60:
            mic = simulate(dim, mics, spos, rt60, session)[:, :total]
            sig_rms = np.sqrt(np.mean(mic[:, pad: -pad] ** 2))
            # 雑音は条件・ずれによらず同じものを使う（試行と T60 ごとに別の乱数の種から作る）
            nrng = np.random.default_rng([seed, int(rt60 * 1000), 7])
            n2max = int((total - s0 - FS) * (1 + max(list(a.eps) + [0.0]) * 1e-6)) + 1
            noise1 = nrng.standard_normal(total) * sig_rms * 10 ** (-SNR_DB / 20)
            noise2 = nrng.standard_normal(n2max) * gain2 * sig_rms * 10 ** (-SNR_DB / 20)
            rec1 = mic[0] + noise1
            c1, q1 = cf.find_chirps(rec1, chirp, spacing)
            for eps_ppm in [0.0] + list(a.eps):
                eps = eps_ppm * 1e-6
                # 機器2 の録音：sync（ε = 0 の理想の録音機）も、ずれのある録音機も、同じ経路（大きさ・読み直し・雑音）で作る
                n2 = int((total - s0 - FS) * (1 + eps))
                if a.gen == "fft":                                     # 作るほうを別の方法（FFT の読み直し）にする
                    s0i = int(round(s0))
                    y2, _ = arr.sfo_resample(mic[1][s0i:], eps)
                    rec2 = gain2 * y2[:n2] + noise2[:n2]
                elif a.drift and eps_ppm != 0:                        # ε(t) = ε + β t（t は機器1 の秒）。sync（ε=0）は同期のまま
                    beta = a.drift * 1e-6 / 60
                    nn = np.arange(n2)
                    rec2 = gain2 * cf.frac_read(mic[1], s0 + nn / (1 + eps) - beta * nn ** 2 / (2 * FS)) + noise2[:n2]
                else:
                    rec2 = gain2 * cf.frac_resample(mic[1], s0, 1 / (1 + eps), n2) + noise2[:n2]
                c2, q2 = cf.find_chirps(rec2, chirp, spacing)
                clk = cf.fit_clock(c1, c2, [(0, 1), (2, 3)])
                # 開始の合わせ方はどの条件も同じ（合図の始めの位置。P1 から2台までの到達時間の差が一定の遅れとして入る）
                start = lambda e: float(c2[0] - (1 + e) * c1[0])
                conds = {}
                if eps_ppm == 0.0:
                    if "sync" in a.conds:
                        conds["sync"] = (0.0, start(0.0), "resample", np.nan)
                else:
                    if "none" in a.conds or "blind" in a.conds:
                        conds["none"] = (clk["eps"], clk["alpha"], "none", clk["eps"])
                    if "chirp" in a.conds:
                        conds["chirp"] = (clk["eps"], clk["alpha"], "resample", clk["eps"])
                    if "oracle" in a.conds:
                        conds["oracle"] = (eps, start(eps), "resample", eps)    # ε だけ真の値（開始は合図）
                for name, (e, al, mode, e_hat) in conds.items():
                    mix, ref = cf.build_mixture(rec1, rec2, c1, lays, 0, L, e, al, mode)
                    todo = [(name, mix, e_hat)]
                    if name == "none" and "blind" in a.conds:
                        eb = cf.drift_search(mix[0], mix[1], FS)[0]           # none の観測から ε だけを推定
                        mb, _ = cf.build_mixture(rec1, rec2, c1, lays, 0, L, eb, start(eb), "resample")
                        todo.append(("blind", mb, eb))
                        if "none" not in a.conds:
                            todo = todo[1:]
                    with np.errstate(divide="ignore"):
                        sdr_in = evaluate(ref, np.stack([mix[0], mix[0]]))[0]
                    for cname, m, eh in todo:
                        orw = evaluate(ref, oracle_w(m, ref))[0] - sdr_in
                        for k in a.k:
                            for s_init in range(a.init_seeds):
                                t0 = time.time()
                                y = separate(m, k, a.iter, seed + 100000 * s_init)
                                sdr, sir, sar = evaluate(ref, y)
                                rows.append(dict(trial=i, seed=seed, init=s_init, gen=a.gen, drift=a.drift if eps_ppm else 0.0,
                                                 pair=f"{pair[0]}-{pair[1]}",
                                                 d_mic=round(d_mic, 3), rt60=rt60, eps_ppm=eps_ppm, cond=cname, k=k,
                                                 eps_hat_ppm=eh * 1e6, sdr_in=sdr_in, sdr=sdr, sir=sir, sar=sar,
                                                 sdr_imp=sdr - sdr_in, orw_imp=orw,
                                                 chirp_q=float(min(q1.min(), q2.min())),
                                                 sec=time.time() - t0))
                                print(f"試行 {i:2d} T60 {rt60*1000:3.0f}ms ずれ {eps_ppm:5.0f}ppm {cname:6s} K={k} 初期値{s_init}"
                                      f"  推定 {eh*1e6:8.2f}ppm  SDR改善 {sdr - sdr_in:6.2f} dB（正解の W なら {orw:6.2f}）"
                                      f"  {time.time()-t0:4.1f}秒", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)       # 途中でも保存する
    print(f"{a.out} に保存しました（{len(rows)} 行、{(time.time()-t_all)/60:.1f} 分）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
import pandas as pd
d = pd.read_csv("sim_main.csv")
nt, s = d["trial"].nunique(), d["init"].nunique()
ok(len(d) == nt * 2 * 13 * s, f"sim_main.csv は {nt} 試行 × 2 T60 × 13 観測 × 種 {s} つ ＝ {len(d)} 行（教科書の本番は 150 × 2 × 13 × 1 ＝ 3900 行）")
ch = d[d["cond"] == "chirp"]
e = (ch["eps_hat_ppm"] - ch["eps_ppm"]).abs()
ok(e.max() < 0.1, f"「合図で推定」が本当の値と 0.1 ppm 以内（T60 256ms でも。最大 {e.max():.3f} ppm）")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['fink_stats.py', 'sim_main.csv', '--k', '4', '--eps', '30', '--delta', '1.0']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python fink_stats.py sim_main.csv --k 4 --eps 30 --delta 1.0
#!/usr/bin/env python3
"""最終演習K：シミュレーションの本実験（sim_main.csv）を、事前登録した計画どおりに解析する

使い方: cd ~/home-lab/final/k && uv run python fink_stats.py sim_main.csv --k 4 --eps 30 --delta 1.0
  --k      : 事前登録した ILRMA の基底の数（予備実験で選んだもの）
  --eps    : 主要な比較のずれ [ppm]
  --delta  : 同等性の幅 Δ [dB]
  初期値の種を複数（--init-seeds）で実行した CSV は、種の平均をその試行・条件の値にしてから解析する
  --q-min  : 除外の基準（合図の高さがこれ未満の試行を除く。SDR を見る前に決めたもの）

【主解析（事前登録）】単位は試行。ずれ --eps のときの、試行ごとの差で
  H1：chirp − none > 0（片側の対応のある t 検定。試行ごとの差は2つの T60 の平均）
  H2：chirp − sync が ±Δ の中（TOST：2つの片側検定。試行ごとの差は2つの T60 の平均）
  H3：blind − sync が ±Δ の中（同上）
  ［2′］結果を見たあとの追加の解析：H2・H3 を T60 ごとに判定する、より厳しい基準（登録した判定は変えない）
  3つの p 値をホルム法で補正し、補正後の p < 0.05 だけで判定する（TOST の p は2つの片側の p の大きいほう）。
  区間は記述のために出す：90% 区間（補正なしの TOST に対応）と、96.7% 区間（ホルム法の最も厳しい場合 α/3 に対応）
【副次解析（事前登録・記述）】
  ・この分野の論文でよく見る形の表（条件ごとの SDR の平均 ± 標準偏差、試行＝乱数の種）
  ・損失（sync − 各条件）の、条件×ずれごとの平均と区間（試行を単位にした t 区間）
  ・混合モデル：損失の分散を「試行」「試行の中の T60」「残り」に分け、級内相関を示す（区間はこれでは出さない。
    条件によって残りの分散が大きく違う（oracle は 0 付近、none の 100 ppm は大きい）ので、共通の分散を仮定した区間は合わない）
  ・正解の W の上限（orw_imp）で「モデルのせいか、推定のせいか」を分ける
  ・ずれの推定の誤差（chirp・blind）
  ・ベイズでの判断（平均に一様、標準偏差 σ に 1/σ の事前分布 → 差の平均の事後分布は t 分布。
    95% 最高密度区間（HDI）が ±Δ に入るか（Kruschke の判定）と、±Δ に入る事後確率）
【統計の誤用の例（この演習のデータで）】
  ・平均だけで比べる（失敗した試行の割合が見えない）
  ・「有意な差がない」を「同じ」と読む（いちばん小さいずれの「補正なし − 同期」を、少ない試行の t 検定だけで判断した場合）
"""
import argparse

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
CONDS = ["none", "chirp", "blind", "oracle"]
LABEL = {"sync": "同期（理想）", "none": "補正なし", "chirp": "合図で補正", "blind": "合図なしで補正", "oracle": "真の値で補正"}


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj, run = np.empty_like(p), 0.0
    for i, j in enumerate(order):
        run = max(run, (len(p) - i) * p[j])
        adj[j] = min(1.0, run)
    return adj


def wide(d, eps):
    """試行 × T60 ごとに、sync と各条件の SDR 改善量を横に並べる"""
    s = d[d["cond"] == "sync"].set_index(["trial", "rt60"])["sdr_imp"].rename("sync")
    o = d[(d["eps_ppm"] == eps) & (d["cond"] != "sync")].pivot_table(index=["trial", "rt60"], columns="cond", values="sdr_imp")
    return o.join(s, how="inner")


def tost(x, delta, alpha=0.05):
    n, m, se = len(x), x.mean(), x.std(ddof=1) / np.sqrt(len(x))
    p_lo = 1 - stats.t.cdf((m + delta) / se, n - 1)          # H0: μ ≤ −Δ
    p_hi = stats.t.cdf((m - delta) / se, n - 1)              # H0: μ ≥ +Δ
    h = stats.t.ppf(1 - alpha, n - 1) * se
    return max(p_lo, p_hi), (m - h, m + h)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv")
    ap.add_argument("--k", type=int, default=4)
    ap.add_argument("--eps", type=float, default=30.0)
    ap.add_argument("--delta", type=float, default=1.0)
    ap.add_argument("--q-min", type=float, default=20.0)
    a = ap.parse_args()
    d = pd.read_csv(a.csv)
    d = d[d["k"] == a.k]
    S = d["init"].nunique()
    # 初期値の種 S 個の平均を、その試行・条件の値にする（この分野の論文の「種の平均」と同じ考え方）
    d = (d.groupby(["trial", "pair", "rt60", "eps_ppm", "cond"], as_index=False)
          [["sdr", "sir", "sar", "sdr_in", "sdr_imp", "orw_imp", "eps_hat_ppm", "chirp_q"]].mean())
    bad = sorted(d.loc[d["chirp_q"] < a.q_min, "trial"].unique())
    print(f"[0] {d['trial'].nunique()} 試行、T60 {[f'{r*1000:.0f}ms' for r in sorted(d['rt60'].unique())]}、"
          f"ずれ {[f'{e:.0f}' for e in sorted(d['eps_ppm'].unique())[1:]]} ppm、K = {a.k}、初期値の種 {S} 個の平均。"
          f"除外（合図の高さ < {a.q_min}）：{bad if bad else 'なし'}")
    d = d[~d["trial"].isin(bad)]

    print("\n[1] この分野の論文でよく見る形の表：SDR [dB] の平均 ± 標準偏差（試行＝乱数の種。標準偏差は「ばらつき」で、平均の不確かさではない）")
    tab = d.groupby(["rt60", "eps_ppm", "cond"])["sdr"].agg(["mean", "std"]).reset_index()
    for rt, g in tab.groupby("rt60"):
        print(f"  T60 = {rt*1000:.0f} ms")
        for _, r in g.iterrows():
            print(f"    {'—' if r.eps_ppm == 0 else f'{r.eps_ppm:.0f} ppm':>8s}  {LABEL[r.cond]:10s} {r['mean']:6.2f} ± {r['std']:.2f}")

    print(f"\n[2] 主解析（事前登録の基準：ずれ {a.eps:.0f} ppm、試行ごとの差は2つの T60 の平均、単位は試行）")
    w = wide(d, a.eps)
    per = w.groupby(level="trial").mean()
    D1, D2, D3 = per["chirp"] - per["none"], per["chirp"] - per["sync"], per["blind"] - per["sync"]
    n = len(per)
    p1 = stats.ttest_1samp(D1, 0, alternative="greater").pvalue
    ci1 = stats.t.interval(0.95, n - 1, D1.mean(), D1.std(ddof=1) / np.sqrt(n))
    p2, ci2 = tost(D2, a.delta)
    p3, ci3 = tost(D3, a.delta)
    a_w = 0.05 / 3                                                     # ホルム法の最も厳しい場合
    _, ci2w = tost(D2, a.delta, alpha=a_w)
    _, ci3w = tost(D3, a.delta, alpha=a_w)
    adj = holm([p1, p2, p3])
    res = [("H1 chirp − none > 0", D1, p1, f"95% 区間 {ci1[0]:+.2f}〜{ci1[1]:+.2f}"),
           (f"H2 chirp − sync が ±{a.delta}dB の中", D2, p2,
            f"90% 区間 {ci2[0]:+.2f}〜{ci2[1]:+.2f}、96.7% 区間 {ci2w[0]:+.2f}〜{ci2w[1]:+.2f}"),
           (f"H3 blind − sync が ±{a.delta}dB の中", D3, p3,
            f"90% 区間 {ci3[0]:+.2f}〜{ci3[1]:+.2f}、96.7% 区間 {ci3w[0]:+.2f}〜{ci3w[1]:+.2f}")]
    for (name, x, p, ci_txt), pa in zip(res, adj):
        print(f"  {name}：差の平均 {x.mean():+.2f} dB（{ci_txt}）、n = {n}、"
              f"p = {p:.2g}（ホルム補正後 {pa:.2g}）→ {'支持' if pa < 0.05 else '支持されない'}")
    print("  判定はホルム補正後の p だけで行う（区間は記述のため。96.7% 区間が ±Δ に入れば、補正後の p も必ず 0.05 未満）")
    print(f"  効果の大きさ：H1 の差の平均 / 差の標準偏差 = {D1.mean()/D1.std(ddof=1):.2f}（対応のある標準化平均差。dB のままの値のほうが伝わる）")

    print("\n[2′] 結果を見たあとの追加の解析（より厳しい基準）：H2・H3 を T60 ごとに TOST し、両方で通るか"
          "（平均では T60 の間の打ち消し合いを見逃すおそれがあるため。p は2つの T60 の p の大きいほう）")
    rts = sorted(w.index.get_level_values("rt60").unique())
    pe, ends = {}, {}
    for name, c in (("H2", "chirp"), ("H3", "blind")):
        ps, txt, e_ = [], [], []
        for rt in rts:
            g = w.xs(rt, level="rt60")
            x = g[c] - g["sync"]
            pr, ci = tost(x, a.delta)
            _, ciw = tost(x, a.delta, alpha=a_w)
            ps.append(pr)
            e_ += [abs(ciw[0]), abs(ciw[1])]
            txt.append(f"T60 {rt*1000:.0f}ms {x.mean():+.2f} dB（96.7% 区間 {ciw[0]:+.2f}〜{ciw[1]:+.2f}、p = {pr:.2g}）")
        pe[name], ends[name] = max(ps), max(e_)
        print(f"  {name}：" + "、".join(txt))
    adj2 = holm([p1, pe["H2"], pe["H3"]])
    print(f"  ホルム補正後：H2 {adj2[1]:.2g}、H3 {adj2[2]:.2g} → {'どちらも通る' if max(adj2[1:]) < 0.05 else '通らないものがある'}（登録した基準の判定は変えない）")
    print("  感度の報告（同等と言える最小の幅＝96.7% 区間の端の絶対値の大きいほう）：")
    for name, x, ciw in (("H2 chirp − sync", D2, ci2w), ("H3 blind − sync", D3, ci3w)):
        print(f"    {name}：平均の基準なら {max(abs(ciw[0]), abs(ciw[1])):.2f} dB、T60 ごとの基準なら {ends[name[:2]]:.2f} dB")
    for c in ("chirp", "blind"):
        fr = [(np.abs(w.xs(rt, level='rt60')[c] - w.xs(rt, level='rt60')['sync']) > a.delta).mean() for rt in rts]
        print(f"    {c} − sync の絶対値が {a.delta} dB を超えた試行の割合："
              + "、".join(f"T60 {rt*1000:.0f}ms {f*100:.0f}%" for rt, f in zip(rts, fr))
              + "（同等なのは「部屋全体での平均」。1つ1つの部屋では、初期値などで 1dB 以上違うことがある）")

    print("\n[3] ベイズでの判断（平均に一様・σ に 1/σ の事前分布 → 差の平均の事後分布は t 分布）")
    for name, x in (("chirp − sync", D2), ("blind − sync", D3)):
        m, se = x.mean(), x.std(ddof=1) / np.sqrt(len(x))
        post = stats.t(len(x) - 1, m, se)
        pin = post.cdf(a.delta) - post.cdf(-a.delta)
        lo, hi = post.ppf([0.025, 0.975])                          # 左右対称なので 95% HDI と同じ
        inside = "入る（同等と判断）" if (lo > -a.delta and hi < a.delta) else "入らない（判断を保留）"
        print(f"  {name}：95% HDI {lo:+.2f}〜{hi:+.2f} dB → ±{a.delta} dB に{inside}。±{a.delta} dB の中にある事後確率 {pin:.3f}")

    print("\n[4] 損失 = sync − 条件 [dB]（条件×ずれごと。2つの T60 の平均。試行を単位にした 95% t 区間）")
    rows = []
    for e in sorted(d["eps_ppm"].unique()):
        if e == 0:
            continue
        we = wide(d, e)
        for c in CONDS:
            if c in we:
                for (tr, rt), r in we.iterrows():
                    rows.append(dict(trial=tr, rt60=rt, eps=e, cond=c, loss=r["sync"] - r[c]))
    L = pd.DataFrame(rows).merge(d[["trial", "pair"]].drop_duplicates(), on="trial")
    summ = []
    for (c, e), g in L.groupby(["cond", "eps"]):
        x = g.groupby("trial")["loss"].mean()
        lo, hi = stats.t.interval(0.95, len(x) - 1, x.mean(), x.std(ddof=1) / np.sqrt(len(x)))
        per_rt = g.groupby("rt60")["loss"].mean().tolist()
        summ.append(dict(cond=c, eps=e, est=x.mean(), lo=lo, hi=hi, per_rt=per_rt))
    S = pd.DataFrame(summ)
    S["o"] = S["cond"].map({c: i for i, c in enumerate(CONDS)})
    for _, r in S.sort_values(["o", "eps"]).iterrows():
        print(f"  {LABEL[r.cond]:10s} {r.eps:5.0f} ppm：損失 {r.est:+6.2f} dB（95% 区間 {r.lo:+.2f}〜{r.hi:+.2f}）"
              f"  T60 ごと " + "・".join(f"{v:+.2f}" for v in r.per_rt))
    L["cell"] = L["cond"] + "_" + L["eps"].astype(int).astype(str) + "_" + (L["rt60"] * 1000).astype(int).astype(str)
    mm = smf.mixedlm("loss ~ 0 + C(cell)", L, groups=L["trial"], re_formula="1", vc_formula={"rt": "0 + C(rt60)"}).fit(reml=True)
    v_tr, v_rt, v_res = float(mm.cov_re.iloc[0, 0]), float(mm.vcomp[0]), mm.scale
    tot = v_tr + v_rt + v_res
    print(f"  混合モデル（分散の成分）：試行 {np.sqrt(v_tr):.2f} dB、試行の中の T60 {np.sqrt(v_rt):.2f} dB、残り {np.sqrt(v_res):.2f} dB"
          f"（標準偏差）。試行と T60 で説明される割合 {(v_tr + v_rt) / tot:.2f}"
          "（同じ試行・同じ T60 の値は似ている → 試行を単位にし、条件の比較は試行の中で行う理由）")

    print("\n[5] 正解の W（周波数ごとに1つの分離行列を、正解との最小二乗で決めたもの。SDR を直接最大にしたものではない目安の上限）")
    print(f"  ILRMA ≤ 正解の W だった行の割合：{(d['orw_imp'] >= d['sdr_imp']).mean()*100:.0f}%")
    orw = d.groupby(["rt60", "eps_ppm", "cond"])[["sdr_imp", "orw_imp"]].mean()
    for (rt, e, c), r in orw.iterrows():
        print(f"  T60 {rt*1000:3.0f}ms {'—' if e == 0 else f'{e:.0f} ppm':>8s} {LABEL[c]:10s} ILRMA {r.sdr_imp:6.2f} dB  正解の W {r.orw_imp:6.2f} dB"
              f"  差 {r.orw_imp - r.sdr_imp:5.2f}")
    print("  読み方：補正なしで正解の W も下がる → 時間に依らない分離行列で届く上限そのものが下がる（モデルの限界）。")
    print("  ILRMA と正解の W の差がずれによらずほぼ一定なら、ILRMA の損失の大部分は上限が下がったことで説明できる。")
    print("  差が広がるなら、ずれのせいで ILRMA の推定も迷っている（推定のせいも加わる）。差を条件ごとに見比べる")

    print("\n[6] ずれの推定の誤差 [ppm]（推定 − 本当の値）")
    for c in ("chirp", "blind"):
        x = d[d["cond"] == c]
        err = x["eps_hat_ppm"] - x["eps_ppm"]
        for rt, g in err.groupby(x["rt60"]):
            print(f"  {LABEL[c]:10s} T60 {rt*1000:3.0f}ms：平均 {g.mean():+.3f}、標準偏差 {g.std():.3f}、絶対値の最大 {g.abs().max():.3f}")

    print("\n[7] 統計の誤用の例")
    fail = d.assign(fail=d["sdr_imp"] < 3.0).groupby(["eps_ppm", "cond"])["fail"].mean()
    mean = d.groupby(["eps_ppm", "cond"])["sdr_imp"].agg(["mean", "median"])
    print("  (a) 平均だけで比べる：SDR 改善量の平均・中央値と、「3dB 未満（分離できたと言いにくい）」の試行の割合")
    for (e, c), r in mean.iterrows():
        print(f"      {'—' if e == 0 else f'{e:.0f} ppm':>8s} {LABEL[c]:10s} 平均 {r['mean']:6.2f}  中央値 {r['median']:6.2f}  3dB 未満 {fail[(e, c)]*100:4.0f}%")
    e_min = min(e for e in d["eps_ppm"].unique() if e > 0)            # いちばん小さいずれ
    w_min = wide(d, e_min).groupby(level="trial").mean()
    Dm = (w_min["none"] - w_min["sync"])
    x6 = Dm.iloc[:6]
    p_t = stats.ttest_1samp(x6, 0).pvalue
    p_tost6, ci6 = tost(x6, a.delta)
    sig = "有意" if p_t < 0.05 else "有意でない"
    print(f"  (b) 「有意な差がない」＝「同じ」ではない：{e_min:.0f} ppm で「補正なし − 同期」を、最初の6試行だけで t 検定すると"
          f" 差の平均 {x6.mean():+.2f} dB、p = {p_t:.2f}（{sig}）。")
    print(f"      ここで「{e_min:.0f} ppm なら直さなくても同期と同じ」と読むのは誤り。TOST の p = {p_tost6:.2f}"
          f"（90% 区間 {ci6[0]:+.2f}〜{ci6[1]:+.2f} dB）で、同等とも言えない。{n} 試行では差の平均 {Dm.mean():+.2f} dB"
          f"（95% 区間 {stats.t.interval(0.95, len(Dm) - 1, Dm.mean(), Dm.std(ddof=1) / np.sqrt(len(Dm)))[0]:+.2f}〜"
          f"{stats.t.interval(0.95, len(Dm) - 1, Dm.mean(), Dm.std(ddof=1) / np.sqrt(len(Dm)))[1]:+.2f}）")

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    for c, col in zip(CONDS, ["C3", "C0", "C2", "C7"]):
        s = S[S["cond"] == c]
        ax[0].errorbar(s["eps"], s["est"], yerr=[s["est"] - s["lo"], s["hi"] - s["est"]], fmt="-o", capsize=3, color=col, label=LABEL[c])
    ax[0].axhline(0, color="k", lw=0.8)
    ax[0].set_xscale("log")
    ax[0].set_xlabel("機器2 のサンプリング周波数のずれ [ppm]")
    ax[0].set_ylabel("同期（理想）からの損失 [dB]（2つの T60 の平均）")
    ax[0].set_title("(a) ずれの大きさと損失（試行を単位にした 95% 区間）")
    ax[0].legend(fontsize=8)
    ax[0].grid(alpha=0.3)
    for i, (name, x, ci, col) in enumerate([("H1 合図で補正 − 補正なし", D1, ci1, "C0"),
                                            ("H2 合図で補正 − 同期", D2, ci2w, "C2"), ("H3 合図なし − 同期", D3, ci3w, "C1")]):
        ax[1].plot(x.values, np.full(len(x), i) + np.linspace(-0.15, 0.15, len(x)), "o", ms=3, alpha=0.5, color=col)
        ax[1].plot([ci[0], ci[1]], [i, i], "-", lw=5, color=col)
        ax[1].plot(x.mean(), i, "D", color="k")
    ax[1].axvspan(-a.delta, a.delta, color="0.85", label=f"同等とみなす幅 ±{a.delta} dB")
    ax[1].axvline(0, color="k", lw=0.8)
    ax[1].set_yticks([0, 1, 2], ["H1", "H2", "H3"])
    ax[1].set_xlabel(f"試行ごとの差 [dB]（{a.eps:.0f} ppm、2つの T60 の平均＝登録した基準。太線：H1 は 95%、H2・H3 は 96.7% 区間）")
    ax[1].set_title("(b) 主解析：点は試行、◆は平均")
    ax[1].legend(fontsize=8, loc="lower right")
    for rt, mk in zip(sorted(w.index.get_level_values("rt60").unique()), ["o", "s"]):
        g = w.xs(rt, level="rt60")
        for _, r in g.iterrows():
            ax[2].plot([0, 1, 2], [r["sync"], r["none"], r["chirp"]], "-", marker=mk, ms=3, alpha=0.5,
                       color="C0" if rt == 0 else "C3")
        ax[2].plot([], [], "-", marker=mk, color="C0" if rt == 0 else "C3", label=f"T60 {rt*1000:.0f} ms")
    ax[2].set_xticks([0, 1, 2], ["同期", "補正なし", "合図で補正"])
    ax[2].set_ylabel("SDR 改善量 [dB]")
    ax[2].set_title(f"(c) 同じ試行を線で結ぶ（{a.eps:.0f} ppm）")
    ax[2].legend(fontsize=8)
    ax[2].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("fink_sim.png", dpi=130)
    plt.show()   # ノートブック版：図を表示する
    print("\nfink_sim.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
h = re.findall(r"^  (H[123]) .*?：差の平均 ([-+\d.]+) dB.*?→ (支持|支持されない)$", cap.stdout, re.M)
ok([x[0] for x in h] == ["H1", "H2", "H3"], f"[2] に H1〜H3 の差の平均・区間・p 値（ホルム補正後）と判定が出る {h}")
ok(float(h[0][1]) > 0, f"H1（合図で補正 − 補正なし）の差の平均が正（{h[0][1]} dB。教科書の 150 試行では +7.47 dB。試行が少ないと判定は「支持されない」になることがある）")
loss = [num(rf"補正なし\s+{e} ppm：損失\s+([-+\d.]+) dB") for e in (10, 30, 100)]
ok(loss[0] < loss[1] < loss[2], f"[4] 補正なしの損失は、ずれが大きいほど大きい {loss}（教科書 +2.89・+7.47・+10.95 dB）")
ok(num(r"ILRMA ≤ 正解の W だった行の割合：(\d+)%") >= 90, "[5] ほとんどの行で ILRMA ≤ 正解の W（教科書は 3900 行すべて）")
ce = [num(rf"合図で補正\s+T60\s+{t}ms：平均 [-+\d.]+、標準偏差 [\d.]+、絶対値の最大 ([\d.]+)") for t in (0, 256)]
ok(max(ce) <= 0.03, f"[6] 合図による推定の誤差が 0.02 ppm 程度以下（絶対値の最大 {ce}）")
ok("fink_sim.png を保存しました" in cap.stdout, "fink_sim.png ができた")

## 8. 副次の確かめ：作りと直しが同じ問題（K-3-4 の注意、段階5 の 3、`sim_async.py --gen fft / --drift 0.5`・`fink_check.py`）

機器2 の録音を「直すのとは別の方法」で作ったとき（`--gen fft`：IV-5 の `common/array.py` の `sfo_resample`）と、時計のずれが時間とともに変わるとき（`--drift 0.5`：0.5 ppm/分）に、合図で補正したものが同期と同じくらいになるかを確かめます（inverse crime ではないか）。

**ノートブックの既定は 3 試行ずつ**（それぞれ約 1.5 分）です。教科書は 12 試行ずつ（それぞれ約 10 分）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
# 教科書（12 試行）：--trials 12
fresh('sim_check_fft.csv')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ['sim_async.py', '--trials', '3', '--seed', '3000', '--k', '4', '--eps', '30', '--conds', 'sync', 'none', 'chirp', '--gen', 'fft', '--out', 'sim_check_fft.csv']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python sim_async.py --trials 3 --seed 3000 --k 4 --eps 30 --conds sync none chirp --gen fft --out sim_check_fft.csv
#!/usr/bin/env python3
"""最終演習K：2台の録音機による「非同期の2チャネル録音」をシミュレーションで作り、ずれの補正の効果を ILRMA で調べる

使い方: cd ~/home-lab/final/k
        uv run python sim_async.py --trials 6 --seed 1000 --k 2 4 --out sim_pilot.csv                 （予備実験）
        uv run python sim_async.py --trials 150 --seed 2000 --k 4 --out sim_main.csv                  （本実験。値は計画書のもの）
  --sources : 音源の wav、または話者ごとのフォルダ（中の wav・flac をつなぐ。JVS・LibriTTS の1人分など）。3つ以上。
              組み合わせを試行ごとに順に使う。省略すると librosa の例 libri1・libri2・trumpet（短い音源はくり返してつなぐ）
  --eps     : 機器2 のサンプリング周波数のずれ [ppm]（いくつでも）
  --rt60    : 残響時間 [s]（いくつでも。同じ部屋の形・置き方で吸音だけ変える）
  --k       : ILRMA の基底の数（いくつでも。予備実験で選び、本実験では1つに固定する）
  --seed    : 乱数の種のはじめ（試行 i は seed + i）。予備実験と本実験で重ならないようにする
  --gen fft / --drift 0.5 : 副次の確かめ。機器2 の録音を「直すのとは別の方法」で作る（作りと直しが同じだと、同等になるのが
              当たり前になる。inverse crime）、または時計のずれが時間とともに変わる録音にする
  --first   : 最初の試行の番号（例：--first 12 --trials 12 で後半だけ。分けて実行し、あとで CSV をつなぐ）

1回の試行（部屋の形・機器と音源の置き方・音源の組と区間）の中で、次の条件をすべて同じ録音から作る（対応のある比較）
  sync   : 同期した理想の録音機（機器2 の時計が機器1 と同じ。ε = 0）  … シミュレーションでしか作れない
  none   : 開始のずれだけ合図で合わせ、サンプリング周波数のずれは直さない
  chirp  : 合図（チャープ）で eps と開始のずれを推定し、機器2 を読み直す
  blind  : eps だけを、合図を使わずに none の観測（開始のずれは合図で合わせたもの）から推定し、
           その eps で並べ直して読み直す（合図は開始の位置だけに使う）
  oracle : eps だけ真の値で読み直す（chirp との差＝eps の推定の誤差の影響）
  機器2 の録音は、どの条件も同じ経路（大きさ ±6dB・帯域制限の読み直し・同じ雑音）で作り、開始は合図の始めの位置で合わせる。
  そのため sync と oracle の差は「ずれのある時計で録って、読み直したこと」だけによる。
結果は1行＝試行×T60×ずれ×条件×K×初期値の種。sdr などは2つの音源の平均。所要時間の目安は K-2 に書いた
"""
import argparse
import itertools
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import librosa
import pyroomacoustics as pra
import fast_bss_eval

sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
import bss                                                   # 第7章の自前の ILRMA（~/home-lab/ch7/bss.py）
import clockfix as cf

try:                                                         # IV-5 の共通ライブラリ（--gen fft のときだけ使う）
    sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
    from common import array as arr
except ImportError:
    arr = None

FS = 16000
NFFT, HOP = 2048, 512                                        # 第7章と同じ（16kHz で 128ms の窓）
SNR_DB = 35.0                                                # 録音機の雑音（静かな部屋の目安）


def load_sources(paths, fs):
    if not paths:
        paths = [librosa.ex(n) for n in ("libri1", "libri2", "trumpet")]
    out = []
    for p in paths:
        p = pathlib.Path(p)
        if p.is_dir():                                         # フォルダなら、中の wav・flac を名前の順につなぐ（1人の話者＝1つの音源）
            files = sorted([*p.rglob("*.wav"), *p.rglob("*.flac")])[:60]
            x = np.concatenate([librosa.effects.trim(librosa.load(f, sr=fs, mono=True)[0], top_db=40)[0] for f in files])
        else:
            x, _ = librosa.load(p, sr=fs, mono=True)
        x = x / (np.std(x) + 1e-9)
        out.append(x)
    return out


def excerpt(x, n, rng):
    """長さ n を切り出す（短い音源はくり返してつなぐ。つなぎ目は 20ms で重ねる）"""
    while len(x) < n + 1:
        r = int(0.02 * FS)
        fade = np.linspace(0, 1, r)
        x = np.r_[x[:-r], x[-r:] * (1 - fade) + x[:r] * fade, x[r:]]
    a = rng.integers(0, len(x) - n)
    return x[a: a + n]


def place(rng):
    """家の部屋（6〜8畳くらい）と、机の上の2台の録音機、2つの音源の位置"""
    dim = np.array([rng.uniform(3.2, 4.5), rng.uniform(2.6, 3.6), 2.4])
    while True:
        c = np.array([rng.uniform(1.2, dim[0] - 1.2), rng.uniform(1.2, dim[1] - 1.2), 0.8])
        d = rng.uniform(0.3, 1.0)                              # 2台の間隔 30〜100cm
        th = rng.uniform(0, np.pi)
        u = np.array([np.cos(th), np.sin(th), 0])
        mics = np.c_[c - d / 2 * u, c + d / 2 * u]
        ang = rng.uniform(0, 2 * np.pi, 2)
        diff = abs((ang[0] - ang[1] + np.pi) % (2 * np.pi) - np.pi)
        if diff < np.radians(40):
            continue
        srcs = []
        for a in ang:
            r = rng.uniform(0.8, 1.6)
            srcs.append(c + np.array([r * np.cos(a), r * np.sin(a), rng.uniform(0.3, 0.5)]))
        if not all(np.all(s[:2] > 0.4) and np.all(s[:2] < dim[:2] - 0.4) for s in srcs):
            continue
        # 2本のマイクで見分けられる置き方か：2つの音源の到達時間差（マイク2 − マイク1）が 3 サンプル以上違うこと。
        # マイクを結ぶ線をはさんで鏡に映した位置や、2台の真ん中の垂直な面の近くに2つとも置くと、到達時間差が同じになり分けられない
        tdoa = [(np.linalg.norm(s - mics[:, 1]) - np.linalg.norm(s - mics[:, 0])) / 343.0 * FS for s in srcs]
        if abs(tdoa[0] - tdoa[1]) >= 3.0:
            return dim, mics, srcs, d


def simulate(dim, mics, srcs, rt60, session):
    """部屋の中で、2つの再生ファイルを順番に鳴らしたときの、2本のマイクの連続した信号（同期・理想）"""
    if rt60 == 0:                                              # 無響（直接音だけ）
        room = pra.ShoeBox(dim, fs=FS, max_order=0)
    else:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    room.add_microphone_array(mics)
    for k in range(2):
        room.add_source(srcs[k], signal=session[k])
    room.simulate()
    return room.mic_array.signals                               # (2, 時間)


def evaluate(ref, est):
    sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return float(np.mean(sdr)), float(np.mean(sir)), float(np.mean(sar))


def stft(x):
    return np.stack([librosa.stft(c, n_fft=NFFT, hop_length=HOP) for c in x], axis=-1)     # (F, T, M)


def istft(Y, n):
    return np.stack([librosa.istft(Y[:, :, k], hop_length=HOP, length=n) for k in range(Y.shape[2])])


def separate(mix, k, n_iter, seed):
    return istft(bss.ilrma(stft(mix), k, n_iter, seed=seed)[0], mix.shape[1])


def oracle_w(mix, ref):
    """正解を使った「時間に依らない分離行列」の上限：周波数ごとに W_f = R_sx R_xx⁻¹（最小二乗）

    ILRMA と同じ「周波数ごとに1つの W_f」というモデルで、正解を知っていれば到達できる成績。
    これが下がるなら、ずれのせいで「モデルそのもの」が合わなくなっている。これが高いのに ILRMA だけ下がるなら「推定」のせい
    """
    X, S = stft(mix), stft(ref)
    Rxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Rsx = np.einsum("ftn,ftm->fnm", S, X.conj())
    W = Rsx @ np.linalg.inv(Rxx + 1e-9 * np.trace(Rxx, axis1=1, axis2=2)[:, None, None] * np.eye(2))
    return istft(np.einsum("fnm,ftm->ftn", W, X), mix.shape[1])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sources", nargs="*")
    ap.add_argument("--trials", type=int, default=2)
    ap.add_argument("--first", type=int, default=0, help="最初の試行の番号（何回かに分けて実行するとき）")
    ap.add_argument("--seed", type=int, default=1000)
    ap.add_argument("--eps", type=float, nargs="+", default=[10, 30, 100])
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.0, 0.256])
    ap.add_argument("--init-seeds", type=int, default=1, help="ILRMA の初期値の乱数の種をいくつ試すか")
    ap.add_argument("--k", type=int, nargs="+", default=[2])
    ap.add_argument("--dur", type=float, default=8.0)
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--conds", nargs="+", default=["sync", "none", "chirp", "blind", "oracle"])
    ap.add_argument("--gen", choices=["sinc", "fft"], default="sinc",
                    help="機器2 の録音の作り方（副次の確かめ）。fft は IV-5 の common/array.py の sfo_resample（直すほうとは別の方法）")
    ap.add_argument("--drift", type=float, default=0.0,
                    help="機器2 の時計のずれが時間とともに変わる速さ [ppm/分]（副次の確かめ。ε(t) = ε + drift·t）")
    ap.add_argument("--out", default="sim_pilot.csv")
    a = ap.parse_args()
    if pathlib.Path(a.out).exists():
        sys.exit(f"{a.out} はすでにあります（上書きしないように止めました）")
    srcs = load_sources(a.sources, FS)
    pairs = list(itertools.combinations(range(len(srcs)), 2))
    L = int(a.dur * FS)
    chirp = cf.make_chirp(FS, cf.CHIRP_SEC)
    rows, t_all = [], time.time()
    for i in range(a.first, a.first + a.trials):
        seed = a.seed + i
        rng = np.random.default_rng(seed)
        pair = pairs[i % len(pairs)]
        dim, mics, spos, d_mic = place(rng)
        # 再生ファイル2つ（音源ごとに区間を乱数で選ぶ）と、それを順番に鳴らすセッション
        plays, lays = [], []
        for k in range(2):
            w, lay = cf.build_play(excerpt(srcs[pair[k]], L, rng), FS, 1, a.dur)
            plays.append(w)
            lays.append(lay)
        spacing = lays[0]["chirp_end"] - lays[0]["chirp_start"]   # 1つのファイルの中の合図の間隔（サンプル）
        pad, gap = int(1.5 * FS), int(1.0 * FS)
        total = pad + len(plays[0]) + gap + len(plays[1]) + pad
        session = np.zeros((2, total))
        session[0, pad: pad + len(plays[0])] = plays[0]
        session[1, pad + len(plays[0]) + gap: pad + len(plays[0]) + gap + len(plays[1])] = plays[1]
        s0 = rng.uniform(0.2, 1.0) * FS                          # 機器2 が録音を始めた位置（機器1 のサンプル番号）
        gain2 = 10 ** (rng.uniform(-6, 6) / 20)                  # 機器2 の録音の大きさ（機器ごとに違う）
        for rt60 in a.rt60:
            mic = simulate(dim, mics, spos, rt60, session)[:, :total]
            sig_rms = np.sqrt(np.mean(mic[:, pad: -pad] ** 2))
            # 雑音は条件・ずれによらず同じものを使う（試行と T60 ごとに別の乱数の種から作る）
            nrng = np.random.default_rng([seed, int(rt60 * 1000), 7])
            n2max = int((total - s0 - FS) * (1 + max(list(a.eps) + [0.0]) * 1e-6)) + 1
            noise1 = nrng.standard_normal(total) * sig_rms * 10 ** (-SNR_DB / 20)
            noise2 = nrng.standard_normal(n2max) * gain2 * sig_rms * 10 ** (-SNR_DB / 20)
            rec1 = mic[0] + noise1
            c1, q1 = cf.find_chirps(rec1, chirp, spacing)
            for eps_ppm in [0.0] + list(a.eps):
                eps = eps_ppm * 1e-6
                # 機器2 の録音：sync（ε = 0 の理想の録音機）も、ずれのある録音機も、同じ経路（大きさ・読み直し・雑音）で作る
                n2 = int((total - s0 - FS) * (1 + eps))
                if a.gen == "fft":                                     # 作るほうを別の方法（FFT の読み直し）にする
                    s0i = int(round(s0))
                    y2, _ = arr.sfo_resample(mic[1][s0i:], eps)
                    rec2 = gain2 * y2[:n2] + noise2[:n2]
                elif a.drift and eps_ppm != 0:                        # ε(t) = ε + β t（t は機器1 の秒）。sync（ε=0）は同期のまま
                    beta = a.drift * 1e-6 / 60
                    nn = np.arange(n2)
                    rec2 = gain2 * cf.frac_read(mic[1], s0 + nn / (1 + eps) - beta * nn ** 2 / (2 * FS)) + noise2[:n2]
                else:
                    rec2 = gain2 * cf.frac_resample(mic[1], s0, 1 / (1 + eps), n2) + noise2[:n2]
                c2, q2 = cf.find_chirps(rec2, chirp, spacing)
                clk = cf.fit_clock(c1, c2, [(0, 1), (2, 3)])
                # 開始の合わせ方はどの条件も同じ（合図の始めの位置。P1 から2台までの到達時間の差が一定の遅れとして入る）
                start = lambda e: float(c2[0] - (1 + e) * c1[0])
                conds = {}
                if eps_ppm == 0.0:
                    if "sync" in a.conds:
                        conds["sync"] = (0.0, start(0.0), "resample", np.nan)
                else:
                    if "none" in a.conds or "blind" in a.conds:
                        conds["none"] = (clk["eps"], clk["alpha"], "none", clk["eps"])
                    if "chirp" in a.conds:
                        conds["chirp"] = (clk["eps"], clk["alpha"], "resample", clk["eps"])
                    if "oracle" in a.conds:
                        conds["oracle"] = (eps, start(eps), "resample", eps)    # ε だけ真の値（開始は合図）
                for name, (e, al, mode, e_hat) in conds.items():
                    mix, ref = cf.build_mixture(rec1, rec2, c1, lays, 0, L, e, al, mode)
                    todo = [(name, mix, e_hat)]
                    if name == "none" and "blind" in a.conds:
                        eb = cf.drift_search(mix[0], mix[1], FS)[0]           # none の観測から ε だけを推定
                        mb, _ = cf.build_mixture(rec1, rec2, c1, lays, 0, L, eb, start(eb), "resample")
                        todo.append(("blind", mb, eb))
                        if "none" not in a.conds:
                            todo = todo[1:]
                    with np.errstate(divide="ignore"):
                        sdr_in = evaluate(ref, np.stack([mix[0], mix[0]]))[0]
                    for cname, m, eh in todo:
                        orw = evaluate(ref, oracle_w(m, ref))[0] - sdr_in
                        for k in a.k:
                            for s_init in range(a.init_seeds):
                                t0 = time.time()
                                y = separate(m, k, a.iter, seed + 100000 * s_init)
                                sdr, sir, sar = evaluate(ref, y)
                                rows.append(dict(trial=i, seed=seed, init=s_init, gen=a.gen, drift=a.drift if eps_ppm else 0.0,
                                                 pair=f"{pair[0]}-{pair[1]}",
                                                 d_mic=round(d_mic, 3), rt60=rt60, eps_ppm=eps_ppm, cond=cname, k=k,
                                                 eps_hat_ppm=eh * 1e6, sdr_in=sdr_in, sdr=sdr, sir=sir, sar=sar,
                                                 sdr_imp=sdr - sdr_in, orw_imp=orw,
                                                 chirp_q=float(min(q1.min(), q2.min())),
                                                 sec=time.time() - t0))
                                print(f"試行 {i:2d} T60 {rt60*1000:3.0f}ms ずれ {eps_ppm:5.0f}ppm {cname:6s} K={k} 初期値{s_init}"
                                      f"  推定 {eh*1e6:8.2f}ppm  SDR改善 {sdr - sdr_in:6.2f} dB（正解の W なら {orw:6.2f}）"
                                      f"  {time.time()-t0:4.1f}秒", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)       # 途中でも保存する
    print(f"{a.out} に保存しました（{len(rows)} 行、{(time.time()-t_all)/60:.1f} 分）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
import pandas as pd
d = pd.read_csv("sim_check_fft.csv")
ok(len(d) == d["trial"].nunique() * 2 * 3 and set(d["gen"]) == {"fft"}, f"sim_check_fft.csv：{d['trial'].nunique()} 試行 × 2 T60 × 3 条件 ＝ {len(d)} 行、作り方 fft（IV-5 の sfo_resample）")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
# 教科書（12 試行）：--trials 12
fresh('sim_check_drift.csv')   # ノートブック版：やり直せるように前に作ったものを消す（教科書のスクリプトは上書きしないように止まる）
import sys
sys.argv = ['sim_async.py', '--trials', '3', '--seed', '3000', '--k', '4', '--eps', '30', '--conds', 'sync', 'none', 'chirp', '--drift', '0.5', '--out', 'sim_check_drift.csv']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python sim_async.py --trials 3 --seed 3000 --k 4 --eps 30 --conds sync none chirp --drift 0.5 --out sim_check_drift.csv
#!/usr/bin/env python3
"""最終演習K：2台の録音機による「非同期の2チャネル録音」をシミュレーションで作り、ずれの補正の効果を ILRMA で調べる

使い方: cd ~/home-lab/final/k
        uv run python sim_async.py --trials 6 --seed 1000 --k 2 4 --out sim_pilot.csv                 （予備実験）
        uv run python sim_async.py --trials 150 --seed 2000 --k 4 --out sim_main.csv                  （本実験。値は計画書のもの）
  --sources : 音源の wav、または話者ごとのフォルダ（中の wav・flac をつなぐ。JVS・LibriTTS の1人分など）。3つ以上。
              組み合わせを試行ごとに順に使う。省略すると librosa の例 libri1・libri2・trumpet（短い音源はくり返してつなぐ）
  --eps     : 機器2 のサンプリング周波数のずれ [ppm]（いくつでも）
  --rt60    : 残響時間 [s]（いくつでも。同じ部屋の形・置き方で吸音だけ変える）
  --k       : ILRMA の基底の数（いくつでも。予備実験で選び、本実験では1つに固定する）
  --seed    : 乱数の種のはじめ（試行 i は seed + i）。予備実験と本実験で重ならないようにする
  --gen fft / --drift 0.5 : 副次の確かめ。機器2 の録音を「直すのとは別の方法」で作る（作りと直しが同じだと、同等になるのが
              当たり前になる。inverse crime）、または時計のずれが時間とともに変わる録音にする
  --first   : 最初の試行の番号（例：--first 12 --trials 12 で後半だけ。分けて実行し、あとで CSV をつなぐ）

1回の試行（部屋の形・機器と音源の置き方・音源の組と区間）の中で、次の条件をすべて同じ録音から作る（対応のある比較）
  sync   : 同期した理想の録音機（機器2 の時計が機器1 と同じ。ε = 0）  … シミュレーションでしか作れない
  none   : 開始のずれだけ合図で合わせ、サンプリング周波数のずれは直さない
  chirp  : 合図（チャープ）で eps と開始のずれを推定し、機器2 を読み直す
  blind  : eps だけを、合図を使わずに none の観測（開始のずれは合図で合わせたもの）から推定し、
           その eps で並べ直して読み直す（合図は開始の位置だけに使う）
  oracle : eps だけ真の値で読み直す（chirp との差＝eps の推定の誤差の影響）
  機器2 の録音は、どの条件も同じ経路（大きさ ±6dB・帯域制限の読み直し・同じ雑音）で作り、開始は合図の始めの位置で合わせる。
  そのため sync と oracle の差は「ずれのある時計で録って、読み直したこと」だけによる。
結果は1行＝試行×T60×ずれ×条件×K×初期値の種。sdr などは2つの音源の平均。所要時間の目安は K-2 に書いた
"""
import argparse
import itertools
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import librosa
import pyroomacoustics as pra
import fast_bss_eval

sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
import bss                                                   # 第7章の自前の ILRMA（~/home-lab/ch7/bss.py）
import clockfix as cf

try:                                                         # IV-5 の共通ライブラリ（--gen fft のときだけ使う）
    sys.path.insert(0, ".")   # ノートブック版：第7章の bss.py と common/ は、このノートブックの隣に書き出してある
    from common import array as arr
except ImportError:
    arr = None

FS = 16000
NFFT, HOP = 2048, 512                                        # 第7章と同じ（16kHz で 128ms の窓）
SNR_DB = 35.0                                                # 録音機の雑音（静かな部屋の目安）


def load_sources(paths, fs):
    if not paths:
        paths = [librosa.ex(n) for n in ("libri1", "libri2", "trumpet")]
    out = []
    for p in paths:
        p = pathlib.Path(p)
        if p.is_dir():                                         # フォルダなら、中の wav・flac を名前の順につなぐ（1人の話者＝1つの音源）
            files = sorted([*p.rglob("*.wav"), *p.rglob("*.flac")])[:60]
            x = np.concatenate([librosa.effects.trim(librosa.load(f, sr=fs, mono=True)[0], top_db=40)[0] for f in files])
        else:
            x, _ = librosa.load(p, sr=fs, mono=True)
        x = x / (np.std(x) + 1e-9)
        out.append(x)
    return out


def excerpt(x, n, rng):
    """長さ n を切り出す（短い音源はくり返してつなぐ。つなぎ目は 20ms で重ねる）"""
    while len(x) < n + 1:
        r = int(0.02 * FS)
        fade = np.linspace(0, 1, r)
        x = np.r_[x[:-r], x[-r:] * (1 - fade) + x[:r] * fade, x[r:]]
    a = rng.integers(0, len(x) - n)
    return x[a: a + n]


def place(rng):
    """家の部屋（6〜8畳くらい）と、机の上の2台の録音機、2つの音源の位置"""
    dim = np.array([rng.uniform(3.2, 4.5), rng.uniform(2.6, 3.6), 2.4])
    while True:
        c = np.array([rng.uniform(1.2, dim[0] - 1.2), rng.uniform(1.2, dim[1] - 1.2), 0.8])
        d = rng.uniform(0.3, 1.0)                              # 2台の間隔 30〜100cm
        th = rng.uniform(0, np.pi)
        u = np.array([np.cos(th), np.sin(th), 0])
        mics = np.c_[c - d / 2 * u, c + d / 2 * u]
        ang = rng.uniform(0, 2 * np.pi, 2)
        diff = abs((ang[0] - ang[1] + np.pi) % (2 * np.pi) - np.pi)
        if diff < np.radians(40):
            continue
        srcs = []
        for a in ang:
            r = rng.uniform(0.8, 1.6)
            srcs.append(c + np.array([r * np.cos(a), r * np.sin(a), rng.uniform(0.3, 0.5)]))
        if not all(np.all(s[:2] > 0.4) and np.all(s[:2] < dim[:2] - 0.4) for s in srcs):
            continue
        # 2本のマイクで見分けられる置き方か：2つの音源の到達時間差（マイク2 − マイク1）が 3 サンプル以上違うこと。
        # マイクを結ぶ線をはさんで鏡に映した位置や、2台の真ん中の垂直な面の近くに2つとも置くと、到達時間差が同じになり分けられない
        tdoa = [(np.linalg.norm(s - mics[:, 1]) - np.linalg.norm(s - mics[:, 0])) / 343.0 * FS for s in srcs]
        if abs(tdoa[0] - tdoa[1]) >= 3.0:
            return dim, mics, srcs, d


def simulate(dim, mics, srcs, rt60, session):
    """部屋の中で、2つの再生ファイルを順番に鳴らしたときの、2本のマイクの連続した信号（同期・理想）"""
    if rt60 == 0:                                              # 無響（直接音だけ）
        room = pra.ShoeBox(dim, fs=FS, max_order=0)
    else:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    room.add_microphone_array(mics)
    for k in range(2):
        room.add_source(srcs[k], signal=session[k])
    room.simulate()
    return room.mic_array.signals                               # (2, 時間)


def evaluate(ref, est):
    sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return float(np.mean(sdr)), float(np.mean(sir)), float(np.mean(sar))


def stft(x):
    return np.stack([librosa.stft(c, n_fft=NFFT, hop_length=HOP) for c in x], axis=-1)     # (F, T, M)


def istft(Y, n):
    return np.stack([librosa.istft(Y[:, :, k], hop_length=HOP, length=n) for k in range(Y.shape[2])])


def separate(mix, k, n_iter, seed):
    return istft(bss.ilrma(stft(mix), k, n_iter, seed=seed)[0], mix.shape[1])


def oracle_w(mix, ref):
    """正解を使った「時間に依らない分離行列」の上限：周波数ごとに W_f = R_sx R_xx⁻¹（最小二乗）

    ILRMA と同じ「周波数ごとに1つの W_f」というモデルで、正解を知っていれば到達できる成績。
    これが下がるなら、ずれのせいで「モデルそのもの」が合わなくなっている。これが高いのに ILRMA だけ下がるなら「推定」のせい
    """
    X, S = stft(mix), stft(ref)
    Rxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Rsx = np.einsum("ftn,ftm->fnm", S, X.conj())
    W = Rsx @ np.linalg.inv(Rxx + 1e-9 * np.trace(Rxx, axis1=1, axis2=2)[:, None, None] * np.eye(2))
    return istft(np.einsum("fnm,ftm->ftn", W, X), mix.shape[1])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sources", nargs="*")
    ap.add_argument("--trials", type=int, default=2)
    ap.add_argument("--first", type=int, default=0, help="最初の試行の番号（何回かに分けて実行するとき）")
    ap.add_argument("--seed", type=int, default=1000)
    ap.add_argument("--eps", type=float, nargs="+", default=[10, 30, 100])
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.0, 0.256])
    ap.add_argument("--init-seeds", type=int, default=1, help="ILRMA の初期値の乱数の種をいくつ試すか")
    ap.add_argument("--k", type=int, nargs="+", default=[2])
    ap.add_argument("--dur", type=float, default=8.0)
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--conds", nargs="+", default=["sync", "none", "chirp", "blind", "oracle"])
    ap.add_argument("--gen", choices=["sinc", "fft"], default="sinc",
                    help="機器2 の録音の作り方（副次の確かめ）。fft は IV-5 の common/array.py の sfo_resample（直すほうとは別の方法）")
    ap.add_argument("--drift", type=float, default=0.0,
                    help="機器2 の時計のずれが時間とともに変わる速さ [ppm/分]（副次の確かめ。ε(t) = ε + drift·t）")
    ap.add_argument("--out", default="sim_pilot.csv")
    a = ap.parse_args()
    if pathlib.Path(a.out).exists():
        sys.exit(f"{a.out} はすでにあります（上書きしないように止めました）")
    srcs = load_sources(a.sources, FS)
    pairs = list(itertools.combinations(range(len(srcs)), 2))
    L = int(a.dur * FS)
    chirp = cf.make_chirp(FS, cf.CHIRP_SEC)
    rows, t_all = [], time.time()
    for i in range(a.first, a.first + a.trials):
        seed = a.seed + i
        rng = np.random.default_rng(seed)
        pair = pairs[i % len(pairs)]
        dim, mics, spos, d_mic = place(rng)
        # 再生ファイル2つ（音源ごとに区間を乱数で選ぶ）と、それを順番に鳴らすセッション
        plays, lays = [], []
        for k in range(2):
            w, lay = cf.build_play(excerpt(srcs[pair[k]], L, rng), FS, 1, a.dur)
            plays.append(w)
            lays.append(lay)
        spacing = lays[0]["chirp_end"] - lays[0]["chirp_start"]   # 1つのファイルの中の合図の間隔（サンプル）
        pad, gap = int(1.5 * FS), int(1.0 * FS)
        total = pad + len(plays[0]) + gap + len(plays[1]) + pad
        session = np.zeros((2, total))
        session[0, pad: pad + len(plays[0])] = plays[0]
        session[1, pad + len(plays[0]) + gap: pad + len(plays[0]) + gap + len(plays[1])] = plays[1]
        s0 = rng.uniform(0.2, 1.0) * FS                          # 機器2 が録音を始めた位置（機器1 のサンプル番号）
        gain2 = 10 ** (rng.uniform(-6, 6) / 20)                  # 機器2 の録音の大きさ（機器ごとに違う）
        for rt60 in a.rt60:
            mic = simulate(dim, mics, spos, rt60, session)[:, :total]
            sig_rms = np.sqrt(np.mean(mic[:, pad: -pad] ** 2))
            # 雑音は条件・ずれによらず同じものを使う（試行と T60 ごとに別の乱数の種から作る）
            nrng = np.random.default_rng([seed, int(rt60 * 1000), 7])
            n2max = int((total - s0 - FS) * (1 + max(list(a.eps) + [0.0]) * 1e-6)) + 1
            noise1 = nrng.standard_normal(total) * sig_rms * 10 ** (-SNR_DB / 20)
            noise2 = nrng.standard_normal(n2max) * gain2 * sig_rms * 10 ** (-SNR_DB / 20)
            rec1 = mic[0] + noise1
            c1, q1 = cf.find_chirps(rec1, chirp, spacing)
            for eps_ppm in [0.0] + list(a.eps):
                eps = eps_ppm * 1e-6
                # 機器2 の録音：sync（ε = 0 の理想の録音機）も、ずれのある録音機も、同じ経路（大きさ・読み直し・雑音）で作る
                n2 = int((total - s0 - FS) * (1 + eps))
                if a.gen == "fft":                                     # 作るほうを別の方法（FFT の読み直し）にする
                    s0i = int(round(s0))
                    y2, _ = arr.sfo_resample(mic[1][s0i:], eps)
                    rec2 = gain2 * y2[:n2] + noise2[:n2]
                elif a.drift and eps_ppm != 0:                        # ε(t) = ε + β t（t は機器1 の秒）。sync（ε=0）は同期のまま
                    beta = a.drift * 1e-6 / 60
                    nn = np.arange(n2)
                    rec2 = gain2 * cf.frac_read(mic[1], s0 + nn / (1 + eps) - beta * nn ** 2 / (2 * FS)) + noise2[:n2]
                else:
                    rec2 = gain2 * cf.frac_resample(mic[1], s0, 1 / (1 + eps), n2) + noise2[:n2]
                c2, q2 = cf.find_chirps(rec2, chirp, spacing)
                clk = cf.fit_clock(c1, c2, [(0, 1), (2, 3)])
                # 開始の合わせ方はどの条件も同じ（合図の始めの位置。P1 から2台までの到達時間の差が一定の遅れとして入る）
                start = lambda e: float(c2[0] - (1 + e) * c1[0])
                conds = {}
                if eps_ppm == 0.0:
                    if "sync" in a.conds:
                        conds["sync"] = (0.0, start(0.0), "resample", np.nan)
                else:
                    if "none" in a.conds or "blind" in a.conds:
                        conds["none"] = (clk["eps"], clk["alpha"], "none", clk["eps"])
                    if "chirp" in a.conds:
                        conds["chirp"] = (clk["eps"], clk["alpha"], "resample", clk["eps"])
                    if "oracle" in a.conds:
                        conds["oracle"] = (eps, start(eps), "resample", eps)    # ε だけ真の値（開始は合図）
                for name, (e, al, mode, e_hat) in conds.items():
                    mix, ref = cf.build_mixture(rec1, rec2, c1, lays, 0, L, e, al, mode)
                    todo = [(name, mix, e_hat)]
                    if name == "none" and "blind" in a.conds:
                        eb = cf.drift_search(mix[0], mix[1], FS)[0]           # none の観測から ε だけを推定
                        mb, _ = cf.build_mixture(rec1, rec2, c1, lays, 0, L, eb, start(eb), "resample")
                        todo.append(("blind", mb, eb))
                        if "none" not in a.conds:
                            todo = todo[1:]
                    with np.errstate(divide="ignore"):
                        sdr_in = evaluate(ref, np.stack([mix[0], mix[0]]))[0]
                    for cname, m, eh in todo:
                        orw = evaluate(ref, oracle_w(m, ref))[0] - sdr_in
                        for k in a.k:
                            for s_init in range(a.init_seeds):
                                t0 = time.time()
                                y = separate(m, k, a.iter, seed + 100000 * s_init)
                                sdr, sir, sar = evaluate(ref, y)
                                rows.append(dict(trial=i, seed=seed, init=s_init, gen=a.gen, drift=a.drift if eps_ppm else 0.0,
                                                 pair=f"{pair[0]}-{pair[1]}",
                                                 d_mic=round(d_mic, 3), rt60=rt60, eps_ppm=eps_ppm, cond=cname, k=k,
                                                 eps_hat_ppm=eh * 1e6, sdr_in=sdr_in, sdr=sdr, sir=sir, sar=sar,
                                                 sdr_imp=sdr - sdr_in, orw_imp=orw,
                                                 chirp_q=float(min(q1.min(), q2.min())),
                                                 sec=time.time() - t0))
                                print(f"試行 {i:2d} T60 {rt60*1000:3.0f}ms ずれ {eps_ppm:5.0f}ppm {cname:6s} K={k} 初期値{s_init}"
                                      f"  推定 {eh*1e6:8.2f}ppm  SDR改善 {sdr - sdr_in:6.2f} dB（正解の W なら {orw:6.2f}）"
                                      f"  {time.time()-t0:4.1f}秒", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)       # 途中でも保存する
    print(f"{a.out} に保存しました（{len(rows)} 行、{(time.time()-t_all)/60:.1f} 分）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
import pandas as pd
d = pd.read_csv("sim_check_drift.csv")
ok(len(d) == d["trial"].nunique() * 2 * 3 and d.loc[d["eps_ppm"] > 0, "drift"].eq(0.5).all() and d.loc[d["cond"] == "sync", "drift"].eq(0).all(),
   f"sim_check_drift.csv：{len(d)} 行、ずれのある条件だけ 0.5 ppm/分で変わる（sync は同期のまま）")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['fink_check.py', 'sim_check_fft.csv', 'sim_check_drift.csv', '--k', '4', '--eps', '30', '--delta', '1.0']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python fink_check.py sim_check_fft.csv sim_check_drift.csv --k 4 --eps 30 --delta 1.0
#!/usr/bin/env python3
"""最終演習K：副次の確かめ（作り方を変えたシミュレーション）を、主解析と同じ形で要約する

使い方: cd ~/home-lab/final/k && uv run python fink_check.py sim_check_fft.csv sim_check_drift.csv --k 4 --eps 30 --delta 1.0
  sim_async.py を --gen fft（作るほうを IV-5 の sfo_resample にする）や --drift 0.5（時計のずれが 0.5 ppm/分で変わる）で
  少ない試行だけ動かした CSV を読み、T60 ごとに「合図で補正 − 同期」「合図で補正 − 補正なし」の差と区間、TOST の p、
  ε の推定の誤差を出す。主解析（H1〜H3）の代わりではなく、「作りと直しが同じなので同等になって当たり前」
  （inverse crime）ではないかを確かめるためのもの
"""
import argparse

import numpy as np
import pandas as pd
from scipy import stats


def tost(x, delta, alpha):
    n, m, se = len(x), x.mean(), x.std(ddof=1) / np.sqrt(len(x))
    p = max(1 - stats.t.cdf((m + delta) / se, n - 1), stats.t.cdf((m - delta) / se, n - 1))
    h = stats.t.ppf(1 - alpha, n - 1) * se
    return p, m - h, m + h


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv", nargs="+")
    ap.add_argument("--k", type=int, default=4)
    ap.add_argument("--eps", type=float, default=30.0)
    ap.add_argument("--delta", type=float, default=1.0)
    a = ap.parse_args()
    for path in a.csv:
        d = pd.read_csv(path)
        d = d[d["k"] == a.k].groupby(["trial", "rt60", "eps_ppm", "cond"], as_index=False)[["sdr_imp", "eps_hat_ppm"]].mean()
        how = pd.read_csv(path)
        label = f"作り方 {how['gen'].iloc[0]}、ずれの変化 {how['drift'].max()} ppm/分（sync は同期のまま）"
        print(f"[{path}] {label}、{d['trial'].nunique()} 試行")
        s = d[d["cond"] == "sync"].set_index(["trial", "rt60"])["sdr_imp"]
        w = d[d["eps_ppm"] == a.eps].pivot_table(index=["trial", "rt60"], columns="cond", values="sdr_imp")
        for rt, g in w.groupby(level="rt60"):
            g = g.droplevel("rt60")
            ss = s.xs(rt, level="rt60").loc[g.index]
            d2 = g["chirp"] - ss
            d1 = g["chirp"] - g["none"]
            p, lo, hi = tost(d2, a.delta, 0.05 / 3)
            print(f"  T60 {rt*1000:3.0f}ms：合図で補正 − 同期 {d2.mean():+.2f} dB（96.7% 区間 {lo:+.2f}〜{hi:+.2f}、TOST p = {p:.2g}）、"
                  f"合図で補正 − 補正なし {d1.mean():+.2f} dB")
        c = d[d["cond"] == "chirp"]
        err = c["eps_hat_ppm"] - c["eps_ppm"]
        print(f"  合図による ε の推定 − 区間の始めの ε：平均 {err.mean():+.3f} ppm、標準偏差 {err.std():.3f} ppm"
              "（ずれが変わる録音では、推定は2つのファイルの間の平均の ε になる）")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
rows = re.findall(r"T60\s+(\d+)ms：合図で補正 − 同期 ([-+\d.]+) dB（96\.7% 区間 ([-+\d.]+)〜([-+\d.]+)、TOST p = ([\d.e-]+)）、合図で補正 − 補正なし ([-+\d.]+) dB", cap.stdout)
ok(len(rows) == 4, f"2 つの CSV × 2 T60 ＝ 4 行に「合図で補正 − 同期」の差と 96.7% 区間が出る（見本の値は K-7-1 の (9)）")
ok(all(abs(float(r[1])) < 1.0 for r in rows), f"どの行も「合図で補正 − 同期」の平均の大きさが 1 dB 未満 {[r[1] for r in rows]}（教科書 -0.44〜-0.09 dB）")
ok(all(float(r[5]) > 1.0 for r in rows), f"どの行も「合図で補正 − 補正なし」が正で 1 dB より大きい {[r[5] for r in rows]}（教科書 +2.37〜+8.15 dB）")
err = [num(r"作り方 fft.*?区間の始めの ε：平均 ([-+\d.]+) ppm", text=cap.stdout.replace("\n", " ")),
       num(r"ずれの変化 0\.5 ppm/分.*?区間の始めの ε：平均 ([-+\d.]+) ppm", text=cap.stdout.replace("\n", " "))]
ok(abs(err[0] - 0.011) < 0.01 and abs(err[1] - 0.110) < 0.02,
   f"ε の推定 − 区間の始めの ε：作り方 FFT で約 +0.011 ppm、ずれが変わる録音で約 +0.110 ppm（2 つのファイルの間の平均の ε になる）{err}")

## 9. 合成の録音の結果を、シミュレーションの予想と比べる（段階5 の 2 の練習、`fink_real_stats.py --sim`）

教科書の段階5 の 2 では、本実験の実録音（`real_mix.py sessions.csv --k 4 --inits 3 --out real_results.csv --clock-out clock.csv`）を解析します。**ノートブックには実録音がないので**、5 節の合成の録音の結果を、7 節のシミュレーション（T60 256 ms）の予想と比べる練習をします。[4] の【誤り】の注意（シミュレーションの数字を、そのまま実録音の結果として書かない）も読んでください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
# 教科書（実録音）：sys.argv = ["fink_real_stats.py", "real_results.csv", "clock.csv", "--sim", "sim_main.csv", "--t60", "0.256", "--k", "4"]
import sys
sys.argv = ['fink_real_stats.py', 'synth_results.csv', 'synth_clock.csv', '--sim', 'sim_main.csv', '--t60', '0.256', '--k', '4']   # ノートブック版：ターミナルでの引数の代わり
# 教科書：uv run python fink_real_stats.py synth_results.csv synth_clock.csv --sim sim_main.csv --t60 0.256 --k 4
#!/usr/bin/env python3
"""最終演習K：実録音の結果（real_results.csv・clock.csv）を解析し、シミュレーションの予想と比べる

使い方: cd ~/home-lab/final/k && uv run python fink_real_stats.py real_results.csv clock.csv --sim sim_main.csv --t60 0.256

【主解析（事前登録）】H4：実録音でも chirp − none > 0。単位はセッション（区間はセッションの中の副次標本）。
  セッションを変量効果にした混合モデル（切片だけ）と、セッションごとの平均の片側 t 検定を並べる
【副次解析】
  ・時計のずれ eps：セッションごとの値、平均と 95% 区間、ファイルごとの値の差（推定の精度の目安）
  ・blind − chirp（合図なしでも、合図ありと同じくらい直せるか）
  ・シミュレーションの予想（同じくらいのずれと、--t60 の部屋）との比較
【統計の誤用の例】区間を独立な標本として数える（疑似反復）と、区間が狭く、p が小さく出すぎる
"""
import argparse
import sys

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("results")
    ap.add_argument("clock")
    ap.add_argument("--sim")
    ap.add_argument("--t60", type=float, default=0.256)
    ap.add_argument("--k", type=int, default=4)
    a = ap.parse_args()
    r = pd.read_csv(a.results, dtype={"session": str})
    c = pd.read_csv(a.clock, dtype={"session": str})

    print("[1] 時計のずれ（Mac を基準にしたスマホのサンプリング周波数のずれ）")
    for _, s in c.iterrows():
        if pd.isna(s.get("eps_ppm")):
            print(f"  {s['session']}：{s['reason']}")
            continue
        print(f"  {s['session']}：{s['eps_ppm']:+.2f} ppm（ファイルごと {s['eps1_ppm']:+.2f}・{s['eps2_ppm']:+.2f}）"
              f"{'' if s['ok'] else '  除外：' + str(s['reason'])}")
    ok = c[c["ok"] == True]
    e = ok["eps_ppm"]
    if len(e) >= 2:
        ci = stats.t.interval(0.95, len(e) - 1, e.mean(), e.std(ddof=1) / np.sqrt(len(e)))
        print(f"  使ったセッション {len(e)} 個：平均 {e.mean():+.2f} ppm（95% 区間 {ci[0]:+.2f}〜{ci[1]:+.2f}）、"
              f"セッション間の標準偏差 {e.std(ddof=1):.2f} ppm、ファイルごとの値の差の絶対値の平均 {(ok['eps1_ppm']-ok['eps2_ppm']).abs().mean():.3f} ppm")
        print(f"  8秒の区間の終わりで、ずれは {abs(e.mean())*1e-6*8*16000:.2f} サンプル（16kHz）＝ {abs(e.mean())*8:.0f} µs、"
              f"4kHz の位相で {360*4000*abs(e.mean())*1e-6*8:.0f} 度")

    if r.empty:
        sys.exit("使えるセッションがありません（clock.csv の除外の理由を確かめる）")
    w = r[r["k"] == a.k].pivot_table(index=["session", "seg"], columns="cond", values="sdr_imp", aggfunc="mean")   # 種の平均
    w["d"] = w["chirp"] - w["none"]
    w["db"] = w["blind"] - w["chirp"]
    print(f"\n[2] 主解析 H4：chirp − none > 0（{w.index.get_level_values(0).nunique()} セッション × 区間 = {len(w)} 個）")
    per = w.groupby(level="session")["d"].mean()
    n = len(per)
    ci_d = (per.mean(), per.mean())
    if n >= 2:
        t = stats.ttest_1samp(per, 0, alternative="greater")
        ci_d = stats.t.interval(0.95, n - 1, per.mean(), per.std(ddof=1) / np.sqrt(n))
        print(f"  セッションごとの平均で：差 {per.mean():+.2f} dB（95% 区間 {ci_d[0]:+.2f}〜{ci_d[1]:+.2f}）、片側 p = {t.pvalue:.3g}、n = {n} セッション")
    dd = w.reset_index()
    if n >= 3:
        m = smf.mixedlm("d ~ 1", dd, groups=dd["session"]).fit(reml=True)
        est, se = m.fe_params["Intercept"], m.bse_fe["Intercept"]
        var_s, var_e = float(m.cov_re.iloc[0, 0]), m.scale
        icc = var_s / (var_s + var_e)
        print(f"  混合モデル（セッションの変量効果）：差 {est:+.2f} dB（標準誤差 {se:.2f}）、級内相関 {icc:.2f}")
        mseg = len(w) / n
        print(f"  設計効果 1 + (m − 1)·ICC = {1 + (mseg - 1) * icc:.2f}（m = 1セッションあたりの区間 {mseg:.1f}）")
    naive = stats.ttest_1samp(w["d"], 0, alternative="greater")
    nci = stats.t.interval(0.95, len(w) - 1, w["d"].mean(), w["d"].std(ddof=1) / np.sqrt(len(w)))
    ratio = (nci[1] - nci[0]) / (ci_d[1] - ci_d[0]) if n >= 2 and ci_d[1] > ci_d[0] else np.nan
    if n < 2:
        print("  セッションが2つ以上ないと、セッションを単位にした区間が出せない（疑似反復の例は比べられない）")
    print(f"  【誤り：疑似反復】区間を独立に {len(w)} 個と数えると：95% 区間 {nci[0]:+.2f}〜{nci[1]:+.2f}、片側 p = {naive.pvalue:.3g}"
          f"（区間の幅はセッション単位の {ratio:.2f} 倍）")
    if np.isnan(ratio):
        pass
    elif ratio < 1:
        print(f"  狭くなる理由は2つ：①同じセッションの区間が似ている（級内相関）のに、標準誤差を √{len(w)} で割ってしまう。"
              f"②自由度が増えて t の分位点が小さくなる（自由度 {n-1} で {stats.t.ppf(0.975, n-1):.2f}、自由度 {len(w)-1} で "
              f"{stats.t.ppf(0.975, len(w)-1):.2f}）。セッションが少ないと②の寄与も大きい")
        print("  → 同じセッションの区間は、時計のずれ・部屋・置き方が同じなので似ている（級内相関が正）。独立と数えると区間が狭く、p が小さく出すぎる")
    else:
        print("  → このデータでは級内相関が小さく、区間はあまり狭くならなかった。それでも、独立かどうかはデータを見る前に設計で決める（単位はセッション）")

    print("\n[3] 副次：blind − chirp（合図を使わない推定は、合図ありとどれだけ違うか）")
    pb = w.groupby(level="session")["db"].mean()
    if len(pb) >= 2:
        ci = stats.t.interval(0.95, len(pb) - 1, pb.mean(), pb.std(ddof=1) / np.sqrt(len(pb)))
        print(f"  差 {pb.mean():+.2f} dB（95% 区間 {ci[0]:+.2f}〜{ci[1]:+.2f}）")
    eb = r[r["cond"] == "blind"].merge(c[["session", "eps_ppm"]], on="session")
    print(f"  合図なしの推定 − 合図ありの推定：平均 {(eb['eps_hat_ppm'] - eb['eps_ppm']).mean():+.2f} ppm、"
          f"標準偏差 {(eb['eps_hat_ppm'] - eb['eps_ppm']).std():.2f} ppm")

    pred = None
    if a.sim and len(e):
        s = pd.read_csv(a.sim)
        s = s[(s["k"] == a.k) & (np.isclose(s["rt60"], a.t60))]
        sw = s[s["cond"].isin(["none", "chirp"])].pivot_table(index=["trial", "eps_ppm"], columns="cond", values="sdr_imp",
                                                              aggfunc="mean")
        g = (sw["chirp"] - sw["none"]).groupby(level="eps_ppm").mean()
        # ずれ 0 では補正の効果も 0 なので、(0, 0) の点を足して、ずれ（ppm）の直線で補間する（10 ppm より小さいときの過大評価を防ぐ）
        xs = np.r_[0.0, g.index.values]
        ys = np.r_[0.0, g.values]
        pred = float(np.interp(abs(e.mean()), xs, ys))
        print(f"\n[4] シミュレーションの予想との比較（T60 {a.t60*1000:.0f} ms、ずれ {abs(e.mean()):.1f} ppm を (0,0) と各 ppm の点の間で補間）")
        print(f"  予想：chirp − none ≈ {pred:+.2f} dB、実録音：{per.mean():+.2f} dB")
        print("  【誤り】シミュレーションの数字を、そのまま実録音の結果として書くこと。部屋の残響・雑音・スピーカの指向性・"
              "マイクの特性・自動の音量調整などが違うので、実録音で確かめた値だけを実録音の結果として書く")

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
    if len(e):
        ax[0].errorbar(range(len(ok)), ok["eps_ppm"], fmt="o", label="セッションごと")
        ax[0].plot(range(len(ok)), ok["eps1_ppm"], "x", color="C1", label="ファイル1")
        ax[0].plot(range(len(ok)), ok["eps2_ppm"], "+", color="C2", label="ファイル2")
        ax[0].set_xticks(range(len(ok)), ok["session"])
    ax[0].set_ylabel("スマホの時計のずれ [ppm]")
    ax[0].set_title("(a) セッションごとの時計のずれ")
    ax[0].legend(fontsize=8)
    ax[0].grid(alpha=0.3)
    for i, (sid, g) in enumerate(w.groupby(level="session")):
        for _, row in g.iterrows():
            ax[1].plot([0, 1, 2], [row["none"], row["chirp"], row["blind"]], "-o", ms=3, alpha=0.6, color=f"C{i % 10}")
        ax[1].plot([], [], "-o", color=f"C{i % 10}", label=sid)
    ax[1].set_xticks([0, 1, 2], ["補正なし", "合図で補正", "合図なしで補正"])
    ax[1].set_ylabel("SDR 改善量 [dB]")
    ax[1].set_title("(b) 区間ごと（色＝セッション）")
    ax[1].legend(fontsize=7, ncol=2)
    ax[1].grid(alpha=0.3)
    ax[2].bar([0], [per.mean()], yerr=[[per.mean() - ci_d[0]], [ci_d[1] - per.mean()]],
              capsize=4, color="C0", label="実録音（セッション単位、95% 区間）")
    if pred is not None:
        ax[2].bar([1], [pred], color="C7", label="シミュレーションの予想")
    ax[2].set_xticks([0, 1], ["実録音", "シミュレーション"])
    ax[2].set_ylabel("合図で補正 − 補正なし [dB]")
    ax[2].set_title("(c) 補正の効果：実録音と予想")
    ax[2].legend(fontsize=8)
    ax[2].axhline(0, color="k", lw=0.8)
    fig.tight_layout()
    fig.savefig("fink_real.png", dpi=130)
    plt.show()   # ノートブック版：図を表示する
    print("\nfink_real.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の K-5 の手順の確認ポイント）
pred, real = num(r"予想：chirp − none ≈ ([-+\d.]+) dB"), num(r"実録音：([-+\d.]+) dB")
ok(pred > 0 and real > 0, f"[4] にシミュレーションの予想（{pred:+.2f} dB）と、合成の録音の値（{real:+.2f} dB）が並んで出る（見本は種 3 つで +1.96・+2.22 dB）")
ok("【誤り】シミュレーションの数字を、そのまま実録音の結果として書くこと" in cap.stdout, "[4] に「シミュレーションの数字を、そのまま実録音の結果として書かない」の注意が出る")

## おわりに

- 時計のずれの大きさ、合図による ε の推定と読み直し、ILRMA の SDR 改善量の比べ方（sync・none・chirp・blind・oracle、正解の W）、予備実験からの K・S・n の決め方、合成の録音での除外の判定と疑似反復の誤り、主解析（H1〜H3、ホルム補正、TOST、ベイズ）、作りと直しが同じ問題を、試しの設定で確かめました。
- 本当の結論（K-7-1 の値と比べられるもの）は、3・7・8 節を本番の設定にし、**事前登録（`prereg.md` のコミット）をしてから**本実験を行うと得られます。事前登録の前に本実験の結果を見ないでください。
- このノートブックに入れていないもの：論文を読む段階1、事前登録とコミット、実際の2台の録音（record.py・スマホ・afinfo・afconvert・sessions.csv）、datahash.py、聴取実験 listen_test.py、報告と発表（段階6）。